# CAPSTONE — The 50-Agent Autonomous Financial Institution
## From Task-Solving Swarms to Endogenous Financial Organization

This capstone changes the object of study. Earlier swarm experiments asked populations of artificial specialists to solve a defined financial problem: identify a market regime, construct a portfolio, reason about corporate restructuring, search for transactions, or derive an approximation to a difficult pricing problem. Here, instead of giving the swarm a single task, we create an artificial financial institution and ask whether its internal organization can emerge, adapt, and sometimes disappear as the financial world changes.

The experiment begins with fifty heterogeneous specialists and an initial capital base of USD 10 billion. The agents are deliberately different. Some think like macro investors, credit analysts, equity investors, derivatives specialists, venture capitalists, corporate-finance advisers, treasury managers, payment-system economists, digital-asset researchers, risk managers, market-microstructure specialists, behavioral economists, and quantitative researchers. A small number are intellectual outsiders—complex-systems, information-theory, physics, engineering, and game-theory specialists—whose role is to introduce alternative representations of the same evidence.

No permanent trading desks are imposed on this population. The institution has a constitution: preserve solvency, maintain liquidity, seek attractive risk-adjusted returns, respect risk limits, and document its decisions. It does not, however, begin with a fixed organizational chart. Each specialist observes a common simulated financial world through a different informational lens. Agents form beliefs, communicate those beliefs, identify complementary expertise, and may propose temporary coalitions. Coalitions are not merely numerical clusters. They must articulate a thesis in natural language, negotiate internal disagreement, write an investment memorandum, request scarce capital, and defend the proposal before an LLM-based Investment Committee.

Language is therefore part of the mechanism. Evidence does not allocate capital by itself. Evidence is interpreted, transformed into an argument, challenged by other agents, and evaluated by an institutional decision process. We distinguish evidentiary strength from persuasive strength. A beautifully argued thesis may be wrong. An awkwardly presented thesis may be correct. When outcomes arrive, the institution updates not only financial capital but also epistemic reputation: who was calibrated, who detected a regime early, who repeatedly persuaded the committee without producing results, and whose dissent proved valuable.

The simulated world runs on two clocks. The fast clock is deterministic Python. It creates synthetic multi-asset returns, macroeconomic variables, credit conditions, liquidity, volatility, and hidden regime transitions. It computes signals, portfolio returns, drawdowns, concentration, risk usage, and performance attribution. The slow clock is institutional. Claude Sonnet 5 is invoked when interpretation matters: belief formation, coalition negotiation, memorandum drafting, committee cross-examination, decision writing, postmortem reasoning, and selected organizational changes. This architecture makes the experiment computationally feasible while keeping the LLM at the center of genuinely qualitative financial judgment.

A coalition is a temporary institution. It has a thesis, membership, dissenting view, requested capital, risk budget, expected horizon, falsification conditions, and dissolution rules. Coalitions may grow, shrink, split, merge, mutate, or disappear. An inflation coalition can become a rates-defense coalition; a financial-stress coalition can later mutate into distressed opportunities; an AI-infrastructure coalition can dissolve when its evidence weakens. Members then return to the population and may join other groups. The fifty specialists are relatively persistent, but the network connecting them is endogenous.

The Investment Committee is itself heterogeneous. The CIO considers opportunity cost and portfolio fit. The CRO emphasizes tail risk, concentration, model uncertainty, and drawdown. Treasury protects liquidity. A quantitative reviewer checks whether the narrative is supported by numerical evidence. A skeptic searches for alternative explanations and narrative overreach. Institutional memory recalls earlier decisions and outcomes. The committee may approve, reject, resize, or condition a proposal. A coalition that requests USD 1 billion may receive USD 300 million subject to explicit triggers.

The scientific question is therefore larger than “does the portfolio make money?” We want to observe the biography of an artificial financial institution. Does a society of heterogeneous agents discover recognizable organizational forms without being told to create them? Does diversity improve adaptation? Do persuasive coalitions capture too much capital? Does institutional memory reduce repeated mistakes? Does the network centralize in crises and decentralize in recoveries? Are successful structures persistent, or does yesterday’s successful organization become tomorrow’s rigidity?

The notebook is intentionally synthetic and educational. It is not an investment recommendation, a production risk system, or a claim that LLMs should autonomously control real capital. Synthetic regimes allow us to know the hidden truth and therefore evaluate whether the institution actually learned something. The exercise is a controlled laboratory for studying autonomous financial behavior, endogenous coalition dynamics, governance, persuasion, capital allocation, and institutional learning.

The central hypothesis is that a heterogeneous population of persistent financial agents, operating under scarce capital and explicit governance, can develop temporary endogenous coalitions whose structure changes with the financial regime; structured language-based debate plus quantitative evidence can produce an adaptive institution whose organization itself becomes a state variable.

The architecture is:

**Financial world → specialist perceptions → LLM beliefs → communication network → coalition proposals → internal debate → investment memoranda → Investment Committee hearings → capital allocation → realized outcomes → reputation and memory → migration / mutation / dissolution → new institution.**

The capstone begins with fifty strangers and USD 10 billion. It ends by asking a different question from every earlier swarm: **what kind of financial institution did they become, and why?**

## CODE UNIT 1 — Constitution, Claude Sonnet 5 compatibility, reproducibility and controls

The institution must exist before any agent is allowed to act. This first unit therefore establishes the charter, the capital base, the liquidity floor, concentration limits, simulation horizon, reproducibility seed, and explicit LLM-call budget. Autonomy is not modeled as the absence of constraints. It is modeled as freedom to reorganize and reason inside a persistent institutional constitution.

The engineering issue is equally important. The notebook uses the current Anthropic Python SDK pattern and the model identifier `claude-sonnet-5`. The API key is obtained from Colab Secrets under `ANTHROPIC_API_KEY`, never embedded in the notebook. A deliberately tiny live compatibility call is available before the swarm begins expensive inference. The wrapper records provenance and falls back gracefully when no key is available.

The LLM budget is explicit because the experiment has two clocks. Python handles frequent numerical operations; Sonnet handles selected moments of institutional judgment. The notebook therefore remains practical even though language models participate in belief formation, coalition negotiation, investment memoranda, committee deliberation, and organizational adaptation.

Finally, governance parameters are stored as data. Every later committee can receive the same charter. Coalitions may mutate, agents may gain or lose influence, and the network may reorganize, but the institution cannot silently abolish its liquidity requirement or create capital that does not exist. This cell therefore defines the constitutional boundary inside which endogenous behavior will occur.

In [6]:
# CODE UNIT 1 — Constitution, Claude Sonnet 5 compatibility, reproducibility and controls
!pip -q install -U anthropic networkx pandas numpy matplotlib scikit-learn

import os, json, random, re
from dataclasses import dataclass, asdict
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt

SEED=766
random.seed(SEED); np.random.seed(SEED)
INITIAL_CAPITAL=10_000_000_000
MIN_LIQUIDITY=0.20
MAX_COALITION_CAPITAL=0.20
N_AGENTS=50
N_DAYS=1000
DECISION_EVERY=50
MODEL="claude-sonnet-5"
MAX_LLM_CALLS=120
LLM_CALLS=0
PROVENANCE=[]

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY=userdata.get("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY=os.getenv("ANTHROPIC_API_KEY")

LIVE_LLM=bool(ANTHROPIC_API_KEY)
client=None
if LIVE_LLM:
    import anthropic
    client=anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

CHARTER={"mandate":"Preserve solvency, maintain liquidity, seek attractive risk-adjusted returns, and learn.",
"initial_capital":INITIAL_CAPITAL,"minimum_liquidity_fraction":MIN_LIQUIDITY,
"maximum_single_coalition_fraction":MAX_COALITION_CAPITAL,
"governance":["CIO","CRO","Treasury","Quant Review","Skeptic","Institutional Memory"]}

def extract_text(message):
    return "\n".join(b.text for b in message.content if getattr(b,"type",None)=="text")

def llm_call(system,prompt,purpose,max_tokens=1200):
    global LLM_CALLS
    if (not LIVE_LLM) or LLM_CALLS>=MAX_LLM_CALLS: return None
    msg=client.messages.create(model=MODEL,max_tokens=max_tokens,system=system,
                               messages=[{"role":"user","content":prompt}])
    LLM_CALLS+=1
    txt=extract_text(msg)
    PROVENANCE.append({"purpose":purpose,"model":MODEL,"text":txt[:5000]})
    return txt

def claude_self_test():
    if not LIVE_LLM: return "LIVE_LLM=False — add ANTHROPIC_API_KEY in Colab Secrets."
    return llm_call("Return only the requested token.","Return exactly: SONNET5_OK",
                    "compatibility_self_test",64)

print("MODEL:",MODEL,"| LIVE_LLM:",LIVE_LLM)
print("Self-test:",claude_self_test())

MODEL: claude-sonnet-5 | LIVE_LLM: True
Self-test: SONNET5_OK


## CODE UNIT 2 — Synthetic financial world and hidden regimes

The swarm requires a world rich enough to create disagreement and change. This unit constructs a synthetic multi-asset economy governed by persistent hidden regimes: Calm Growth, Inflation Shock, Financial Stress, Recovery, and Technology Boom. Each regime alters expected returns, volatility, credit spreads, liquidity, interest rates, inflation, and an AI-investment-demand proxy.

The agents are not shown the regime label. They observe manifestations of the hidden state. That separation is crucial because the experimenter retains a ground truth against which beliefs can later be evaluated. If a coalition claims that systemic stress is developing, we can ask whether the hidden process actually entered Financial Stress and whether the coalition detected it early or merely narrated it after the fact.

The synthetic environment is not intended to reproduce every empirical property of real financial markets. Its role is methodological: it generates coherent, changing conditions in which different specialties can legitimately emphasize different signals. A credit analyst can worry about widening spreads while a venture specialist focuses on long-duration technology demand and Treasury focuses on disappearing liquidity.

This is the fast clock of the institution. One thousand market days can be generated almost instantly. The slow LLM clock will wake only at institutional decision dates or meaningful events. This separation is what allows the notebook to study complex autonomous organization without turning the exercise into fifty agents chatting on every simulated trading day.

In [7]:
# CODE UNIT 2 — Synthetic financial world and hidden regimes
regimes=["Calm Growth","Inflation Shock","Financial Stress","Recovery","Technology Boom"]
assets=["EQUITY","DURATION","CREDIT","VOL_HEDGE","CRYPTO","CASH"]
transition=np.array([[.92,.03,.02,.01,.02],[.04,.88,.05,.02,.01],[.01,.03,.90,.05,.01],
                     [.06,.01,.02,.88,.03],[.05,.02,.02,.03,.88]])
mu={"Calm Growth":[.00045,.00012,.00025,-.00015,.00055,.00010],
"Inflation Shock":[-.00025,-.00045,-.00015,.00040,-.00030,.00013],
"Financial Stress":[-.00120,.00035,-.00090,.00160,-.00150,.00012],
"Recovery":[.00085,.00020,.00065,-.00035,.00090,.00010],
"Technology Boom":[.00105,-.00005,.00025,.00010,.00140,.00010]}
vol={"Calm Growth":[.009,.005,.004,.012,.025,.0002],"Inflation Shock":[.014,.009,.007,.018,.035,.0002],
"Financial Stress":[.025,.012,.015,.030,.055,.0002],"Recovery":[.014,.007,.008,.016,.035,.0002],
"Technology Boom":[.017,.007,.008,.020,.045,.0002]}
state=0; rows=[]
for t in range(N_DAYS):
    if t: state=np.random.choice(len(regimes),p=transition[state])
    rg=regimes[state]; r=np.random.normal(mu[rg],vol[rg])
    inflation={"Calm Growth":2.2,"Inflation Shock":6.2,"Financial Stress":3.4,"Recovery":2.6,"Technology Boom":2.4}[rg]+np.random.normal(0,.25)
    rate10y={"Calm Growth":3.4,"Inflation Shock":5.4,"Financial Stress":3.0,"Recovery":3.2,"Technology Boom":3.7}[rg]+np.random.normal(0,.18)
    spread={"Calm Growth":110,"Inflation Shock":180,"Financial Stress":430,"Recovery":150,"Technology Boom":125}[rg]+np.random.normal(0,18)
    liquidity={"Calm Growth":.82,"Inflation Shock":.55,"Financial Stress":.20,"Recovery":.68,"Technology Boom":.74}[rg]+np.random.normal(0,.04)
    ai={"Calm Growth":.45,"Inflation Shock":.25,"Financial Stress":.18,"Recovery":.55,"Technology Boom":.95}[rg]+np.random.normal(0,.05)
    rows.append([t,rg,inflation,rate10y,spread,liquidity,ai,*r])
world=pd.DataFrame(rows,columns=["day","hidden_regime","inflation","rate10y","credit_spread","liquidity","ai_demand",*assets])
world["market_vol_20"]=world["EQUITY"].rolling(20).std().fillna(world["EQUITY"].std())*np.sqrt(252)
world["equity_mom_20"]=world["EQUITY"].rolling(20).sum().fillna(0)
world["credit_change_20"]=world["credit_spread"].diff(20).fillna(0)
world["liquidity_change_20"]=world["liquidity"].diff(20).fillna(0)
display(world.head()); print(world.hidden_regime.value_counts())

,day,hidden_regime,inflation,rate10y,credit_spread,liquidity,ai_demand,EQUITY,DURATION,CREDIT,VOL_HEDGE,CRYPTO,CASH,market_vol_20,equity_mom_20,credit_change_20,liquidity_change_20
0,0,Calm Growth,2.069639,3.419272,113.888293,0.794659,0.459509,-0.003672,0.012992,-0.000880,0.000229,0.001787,0.000100,0.274742,0.0,0.0,0.0
1,1,Calm Growth,1.877620,3.487802,94.455281,0.786098,0.456556,-0.001779,-0.002419,-0.003046,-0.004408,0.029818,0.000053,0.274742,0.0,0.0,0.0
2,2,Calm Growth,2.534984,3.430943,57.555197,0.788845,0.500628,0.011363,-0.005305,0.001587,-0.002001,-0.004827,0.000255,0.274742,0.0,0.0,0.0
3,3,Calm Growth,2.156323,3.338045,90.990317,0.854592,0.387710,-0.016503,-0.002908,0.006011,-0.009423,-0.021457,-0.000270,0.274742,0.0,0.0,0.0
4,4,Calm Growth,2.078534,3.662600,118.019008,0.805151,0.525193,0.013992,0.000586,0.006526,0.006571,-0.018489,-0.000078,0.274742,0.0,0.0,0.0


hidden_regime
Calm Growth         253
Financial Stress    227
Recovery            186
Technology Boom     168
Inflation Shock     166
Name: count, dtype: int64


## CODE UNIT 3 — Instantiate fifty heterogeneous specialists

A swarm becomes institutionally interesting only when its members are meaningfully heterogeneous. This unit creates fifty persistent specialist identities covering macro, rates, sovereign and corporate credit, public equities, derivatives, volatility, market microstructure, systematic trading, private markets, venture capital, M&A, restructuring, treasury, risk, payments, fintech, digital assets, infrastructure, power, semiconductors, AI economics, real estate, commodities, behavioral finance, econometrics, complex systems, information theory, control engineering, and game theory.

Each agent also receives a horizon and a risk temperament. These characteristics are not cosmetic. A defensive liquidity specialist should interpret the same world differently from an opportunistic venture investor. Persistent identity allows the experiment to track whether an initially peripheral specialist later becomes central.

The critical distinction is between agents and organizations. Agents persist; coalitions do not. The fifty specialists constitute the population from which temporary institutional structures emerge. A credit specialist may belong to Financial Stress during one period and later migrate to Distressed Opportunities. A game-theory specialist may be peripheral until competition among coalitions makes strategic framing important.

The cell also initializes reputation, epistemic score, persuasive score, and influence separately. This avoids a simplistic assumption that financial success, intellectual reliability, rhetorical power, and organizational authority are identical. Their divergence will later become one of the most interesting features of the artificial institution.

In [8]:
# CODE UNIT 3 — Instantiate fifty heterogeneous specialists
specialties=["Global Macro","Rates","Sovereign Credit","Investment Grade Credit","High Yield","Bank Credit",
"Public Equity Fundamental","Equity Quant","Equity Growth","Equity Value","Equity Quality","Small Cap",
"Derivatives","Volatility","Options Convexity","Market Microstructure","Systematic Trend","Relative Value",
"Distressed Debt","Private Credit","Venture Capital","Growth Equity","Private Equity","M&A","Corporate Restructuring",
"Capital Structure","Treasury","Liquidity Risk","Market Risk","Credit Risk","Operational Risk","Portfolio Construction",
"Payments","FinTech","Digital Assets","Crypto Market Structure","Stablecoins","Infrastructure Finance","Power Markets",
"Semiconductors","AI Economics","Real Estate","Commodities","FX","Behavioral Finance","Econometrics",
"Complex Systems","Information Theory","Control Engineering","Game Theory"]
@dataclass
class Agent:
    id:int; specialty:str; horizon:str; risk_temperament:str
    reputation:float=.50; epistemic_score:float=.50; persuasion_score:float=.50; influence:float=.50
agents=[Agent(i,sp,random.choice(["days-weeks","1-3 months","3-12 months","multi-year"]),
              random.choice(["defensive","balanced","opportunistic"])) for i,sp in enumerate(specialties)]
agent_df=pd.DataFrame([asdict(a) for a in agents])
display(agent_df.head(10))

,id,specialty,horizon,risk_temperament,reputation,epistemic_score,persuasion_score,influence
0,0,Global Macro,3-12 months,defensive,0.5,0.5,0.5,0.5
1,1,Rates,3-12 months,balanced,0.5,0.5,0.5,0.5
2,2,Sovereign Credit,3-12 months,opportunistic,0.5,0.5,0.5,0.5
3,3,Investment Grade Credit,1-3 months,balanced,0.5,0.5,0.5,0.5
4,4,High Yield,days-weeks,balanced,0.5,0.5,0.5,0.5
5,5,Bank Credit,multi-year,opportunistic,0.5,0.5,0.5,0.5
6,6,Public Equity Fundamental,3-12 months,balanced,0.5,0.5,0.5,0.5
7,7,Equity Quant,multi-year,opportunistic,0.5,0.5,0.5,0.5
8,8,Equity Growth,days-weeks,defensive,0.5,0.5,0.5,0.5
9,9,Equity Value,multi-year,defensive,0.5,0.5,0.5,0.5


## CODE UNIT 4 — Partial perception of a common financial world

If every specialist receives exactly the same information, heterogeneity becomes little more than role-playing. This unit therefore creates partial observability. Each specialty receives a deliberately different subset of the world state. Credit specialists emphasize spreads and liquidity; rates and macro specialists emphasize inflation and yields; Treasury and risk emphasize funding conditions; technology and venture specialists see AI-demand and long-duration variables; digital-asset specialists see crypto and risk appetite.

Agents nevertheless share enough context to communicate. The purpose is not to isolate them but to make collaboration economically useful. A coalition can possess a richer information set than any individual member. This gives endogenous organization a genuine informational rationale.

The code compresses raw history into rolling signals such as market volatility, equity momentum, spread changes, and liquidity changes. This also improves LLM discipline. Sonnet receives a compact evidence packet rather than thousands of observations and is instructed not to invent facts outside that packet.

Partial observability turns coalition formation into a form of information aggregation. A liquidity agent may notice deteriorating market depth, a credit agent widening spreads, and a volatility agent rising convexity demand. Their alliance can become informative precisely because each contributes a different fragment of the underlying state.

In [9]:
# CODE UNIT 4 — Partial perception of a common financial world
def lens_for(sp):
    s=sp.lower(); base=["market_vol_20","equity_mom_20"]
    if any(k in s for k in ["credit","distressed","bank","capital structure"]): return base+["credit_spread","credit_change_20","liquidity"]
    if any(k in s for k in ["rate","macro","sovereign","fx","commod"]): return base+["inflation","rate10y","credit_spread"]
    if any(k in s for k in ["liquidity","treasury","microstructure","risk"]): return base+["liquidity","liquidity_change_20","credit_spread"]
    if any(k in s for k in ["venture","growth","ai","semi","power","infrastructure"]): return base+["ai_demand","rate10y","liquidity"]
    if any(k in s for k in ["crypto","digital","stablecoin","payments","fintech"]): return base+["CRYPTO","liquidity","market_vol_20"]
    return base+["credit_spread","rate10y","liquidity","ai_demand"]
def perception_packet(agent,day):
    row=world.loc[day]
    return {k:round(float(row[k]),4) for k in lens_for(agent.specialty)}
decision_days=list(range(100,N_DAYS-60,DECISION_EVERY))
EVENT_DAY=decision_days[0]
print(agents[0].specialty,perception_packet(agents[0],EVENT_DAY))

Global Macro {'market_vol_20': 0.2332, 'equity_mom_20': 0.0877, 'inflation': 2.1891, 'rate10y': 3.5944, 'credit_spread': 111.5448}


## CODE UNIT 5 — LLM belief formation

This is the first major slow-clock operation. Each specialist receives its identity, horizon, risk temperament, and current evidence packet. Claude Sonnet 5 is asked to transform those observations into a structured financial belief: thesis, confidence, directional interpretation, horizon, key evidence, desired collaborators, falsification condition, and a short natural-language argument.

The LLM is not used as a calculator. Python already knows the numbers. Sonnet’s task is interpretation. A rise in volatility can represent systemic deterioration, temporary repricing, a hedging opportunity, or noise depending on the agent’s intellectual lens. That ambiguity is precisely why language reasoning belongs here.

Beliefs are stored in a structured format so later stages can combine quantitative and linguistic information. The agent must say not only what it believes but also what would make it change its mind. Falsifiability is therefore built into the institutional protocol.

A deterministic fallback is provided so the notebook remains executable without live inference. In the intended capstone run, however, the live LLM mode is the intellectually important case. It turns fifty numerical observers into fifty interpreters, each producing an argument that can attract collaborators or provoke disagreement.

In [10]:
# CODE UNIT 5 — LLM belief formation
def parse_json_loose(text):
    if not text:return None
    try:return json.loads(text)
    except:
        m=re.search(r"\{.*\}",text,re.S)
        if m:
            try:return json.loads(m.group())
            except:return None
def fallback_belief(a,p):
    stress=p.get("credit_change_20",0)>25 or p.get("liquidity",1)<.4 or p.get("market_vol_20",0)>.28
    tech=p.get("ai_demand",0)>.7
    if stress: thesis,direction,conf="Financial conditions are deteriorating; prioritize resilience.","defensive",.72
    elif tech: thesis,direction,conf="AI-linked investment demand is strengthening.","risk_on",.68
    else: thesis,direction,conf="Signals are mixed; maintain selective risk and liquidity.","neutral",.56
    return {"thesis":thesis,"confidence":conf,"direction":direction,"horizon":a.horizon,
            "evidence":list(p)[:4],"desired_collaborators":["complementary specialist"],
            "falsification":"Sustained reversal in dominant signals.","argument":f"{a.specialty} interpretation."}
def form_belief(a,day):
    p=perception_packet(a,day)
    prompt=f'''Specialist #{a.id}: {a.specialty}. Horizon={a.horizon}; temperament={a.risk_temperament}.
Evidence packet ONLY: {json.dumps(p)}
Return ONLY valid JSON: thesis, confidence (0-1), direction (risk_on|defensive|neutral|relative_value),
horizon, evidence (list), desired_collaborators (list), falsification, argument (max 90 words).
Do not invent unavailable facts.'''
    return parse_json_loose(llm_call("You are a specialist inside a governed artificial financial institution.",
                                     prompt,f"belief_{a.id}_{day}",450)) or fallback_belief(a,p)
beliefs={a.id:form_belief(a,EVENT_DAY) for a in agents}
display(pd.DataFrame([{"agent":i,"specialty":agents[i].specialty,**b} for i,b in beliefs.items()]).head())

,agent,specialty,thesis,confidence,direction,horizon,evidence,desired_collaborators,falsification,argument
0,0,Global Macro,Signals are mixed; maintain selective risk and...,0.56,neutral,3-12 months,"[market_vol_20, equity_mom_20, inflation, rate...",[complementary specialist],Sustained reversal in dominant signals.,Global Macro interpretation.
1,1,Rates,Signals are mixed; maintain selective risk and...,0.56,neutral,3-12 months,"[market_vol_20, equity_mom_20, inflation, rate...",[complementary specialist],Sustained reversal in dominant signals.,Rates interpretation.
2,2,Sovereign Credit,Signals are mixed; maintain selective risk and...,0.56,neutral,3-12 months,"[market_vol_20, equity_mom_20, credit_spread, ...",[complementary specialist],Sustained reversal in dominant signals.,Sovereign Credit interpretation.
3,3,Investment Grade Credit,Signals are mixed; maintain selective risk and...,0.56,neutral,1-3 months,"[market_vol_20, equity_mom_20, credit_spread, ...",[complementary specialist],Sustained reversal in dominant signals.,Investment Grade Credit interpretation.
4,4,High Yield,Signals are mixed; maintain selective risk and...,0.56,neutral,days-weeks,"[market_vol_20, equity_mom_20, credit_spread, ...",[complementary specialist],Sustained reversal in dominant signals.,High Yield interpretation.


## CODE UNIT 6 — Endogenous communication network

Beliefs create the raw material for social organization. This unit constructs a dynamic affinity network among the fifty agents. Edges become stronger when agents share a meaningful directional concern, possess complementary rather than redundant expertise, have compatible horizons, and express sufficiently strong confidence.

Complementarity matters. Five nearly identical equity agents repeating the same view should not automatically create the most valuable coalition. A credit specialist, liquidity specialist, volatility specialist, and banking specialist may be more institutionally useful because they observe related phenomena through different mechanisms.

The graph is only a candidate-generation layer. Mathematical affinity does not itself create a coalition. It identifies conversations worth having. The LLM will subsequently decide whether a candidate community can articulate a coherent temporary institutional purpose.

Because beliefs change, the graph changes. During stress, credit, liquidity, Treasury, volatility, and banking nodes may become tightly connected. During a technology boom, venture, semiconductors, power, infrastructure, and AI economics may become central. Network statistics therefore become part of the institution’s observable state.

This creates a crucial conceptual shift: organizational structure is no longer an input. It becomes an output of changing beliefs and relationships.

In [17]:
# ------------------------------------------------------------
# OVERLAPPING CANDIDATE COALITION GENERATION
# ------------------------------------------------------------

AFFINITY_THRESHOLD = 0.52
MIN_COALITION_SIZE = 3
MAX_COALITION_SIZE = 8
TARGET_CANDIDATES = 12

# Build richer communication graph
G = nx.Graph()

for a in agents:
    G.add_node(
        a.id,
        specialty=a.specialty,
        reputation=a.reputation
    )

for i in range(N_AGENTS):
    for j in range(i + 1, N_AGENTS):

        bi, bj = beliefs[i], beliefs[j]

        direction = (
            1.0 if bi["direction"] == bj["direction"]
            else 0.40
        )

        diversity = (
            1.0
            if family(agents[i].specialty)
            != family(agents[j].specialty)
            else 0.50
        )

        confidence = (
            float(bi["confidence"]) +
            float(bj["confidence"])
        ) / 2

        horizon = (
            1.0
            if bi["horizon"] == bj["horizon"]
            else 0.65
        )

        affinity = (
            0.25 * direction +
            0.35 * diversity +
            0.20 * confidence +
            0.20 * horizon
        )

        if affinity >= AFFINITY_THRESHOLD:
            G.add_edge(i, j, weight=affinity)


# ------------------------------------------------------------
# METHOD 1 — Modularity communities
# ------------------------------------------------------------

candidate_sets = []

base_communities = list(
    nx.algorithms.community.greedy_modularity_communities(
        G,
        weight="weight"
    )
)

for c in base_communities:

    c = list(c)

    if len(c) >= MIN_COALITION_SIZE:

        if len(c) <= MAX_COALITION_SIZE:
            candidate_sets.append(c)

        else:
            # break very large communities into smaller candidates
            ranked = sorted(
                c,
                key=lambda n: G.degree(n, weight="weight"),
                reverse=True
            )

            candidate_sets.append(
                ranked[:MAX_COALITION_SIZE]
            )


# ------------------------------------------------------------
# METHOD 2 — Ego-network coalitions
# ------------------------------------------------------------

central_agents = sorted(
    G.nodes(),
    key=lambda n: G.degree(n, weight="weight"),
    reverse=True
)[:15]

for leader in central_agents:

    neighbors = sorted(
        G.neighbors(leader),
        key=lambda n: G[leader][n]["weight"],
        reverse=True
    )

    candidate = [leader] + neighbors[:MAX_COALITION_SIZE - 1]

    if len(candidate) >= MIN_COALITION_SIZE:
        candidate_sets.append(candidate)


# ------------------------------------------------------------
# REMOVE EXACT DUPLICATES, BUT ALLOW OVERLAP
# ------------------------------------------------------------

unique_candidates = []

seen = set()

for c in candidate_sets:

    key = tuple(sorted(c))

    if key not in seen:
        seen.add(key)
        unique_candidates.append(sorted(c))


# Rank candidates by internal affinity
def coalition_quality(members):

    weights = []

    for i in members:
        for j in members:

            if i < j and G.has_edge(i, j):
                weights.append(G[i][j]["weight"])

    return np.mean(weights) if weights else 0


unique_candidates = sorted(
    unique_candidates,
    key=coalition_quality,
    reverse=True
)

communities = unique_candidates[:TARGET_CANDIDATES]

print(
    f"{len(communities)} candidate coalitions generated"
)

for k, c in enumerate(communities):

    print(
        f"\nCandidate {k+1}: "
        f"{len(c)} agents | "
        f"quality={coalition_quality(c):.3f}"
    )

    print(
        [agents[i].specialty for i in c]
    )

12 candidate coalitions generated

Candidate 1: 8 agents | quality=0.899
['Global Macro', 'Rates', 'Sovereign Credit', 'Public Equity Fundamental', 'Volatility', 'Distressed Debt', 'Corporate Restructuring', 'Infrastructure Finance']

Candidate 2: 8 agents | quality=0.899
['Global Macro', 'Rates', 'Sovereign Credit', 'Public Equity Fundamental', 'Volatility', 'Distressed Debt', 'Corporate Restructuring', 'Semiconductors']

Candidate 3: 8 agents | quality=0.899
['Investment Grade Credit', 'Relative Value', 'Liquidity Risk', 'Credit Risk', 'Operational Risk', 'AI Economics', 'Commodities', 'Complex Systems']

Candidate 4: 8 agents | quality=0.899
['Investment Grade Credit', 'Relative Value', 'Liquidity Risk', 'Credit Risk', 'AI Economics', 'Commodities', 'Econometrics', 'Complex Systems']

Candidate 5: 8 agents | quality=0.899
['Investment Grade Credit', 'Relative Value', 'Credit Risk', 'Operational Risk', 'AI Economics', 'Commodities', 'Econometrics', 'Complex Systems']

Candidate 6: 8 

## CODE UNIT 7 — LLM alliance proposals and coalition formation

Candidate communities now attempt to become actual coalitions. A representative receives the identities and beliefs of potential members together with the institutional charter. Claude is asked whether these specialists have a coherent reason to organize now. Mere agreement is insufficient; the coalition should combine complementary expertise and articulate why collective analysis is superior to separate voices.

If the alliance is justified, Sonnet creates a coalition contract containing a name, shared thesis, dissenting element, internal skeptic, expected lifespan, and dissolution condition. The contract gives the coalition temporary institutional identity but no capital.

This distinction is important. Organization precedes funding. A group can deserve investigation without deserving money. The architecture therefore separates social emergence from financial authority.

The notebook keeps inference practical by evaluating candidate communities rather than running every possible pairwise negotiation. Yet the essential behavior remains endogenous: the numerical network proposes possible alliances, while the language model interprets whether those alliances make economic and intellectual sense.

The resulting coalitions are temporary hypotheses about how the institution should organize itself under the current state of the world.

In [19]:
# ============================================================
# CODE UNIT 7 — LLM ALLIANCE PROPOSALS AND COALITION FORMATION
# ============================================================
#
# PURPOSE
# -------
# Transform the overlapping candidate alliances generated in
# Cell 6 into genuine temporary financial coalitions.
#
# IMPORTANT DESIGN PRINCIPLE:
# A coalition does NOT require complete agreement.
# It requires:
#   1. a financially meaningful shared question/opportunity/risk,
#   2. complementary expertise,
#   3. enough common ground to formulate a testable thesis.
#
# Internal disagreement is explicitly desirable.
#
# Agents may appear in MULTIPLE coalitions.
# Coalitions are temporary and endogenous.
# ============================================================


coalitions = []
rejected_coalitions = []


# ------------------------------------------------------------
# 1. Helper: deterministic fallback
# ------------------------------------------------------------

def fallback_coalition(idx, members):

    member_beliefs = [beliefs[i] for i in members]

    confidences = [
        float(b.get("confidence", 0.50))
        for b in member_beliefs
    ]

    directions = [
        b.get("direction", "neutral")
        for b in member_beliefs
    ]

    specialties_here = [
        agents[i].specialty
        for i in members
    ]

    # Dominant direction — only for fallback purposes
    direction_counts = {}

    for d in directions:
        direction_counts[d] = direction_counts.get(d, 0) + 1

    dominant_direction = max(
        direction_counts,
        key=direction_counts.get
    )

    avg_confidence = float(np.mean(confidences))

    return {

        "exists": True,

        "name": f"Emergent Coalition {idx + 1}",

        "shared_question":
            "Do these complementary signals jointly indicate "
            "a financially actionable opportunity or risk?",

        "shared_thesis":
            f"Specialists from {', '.join(specialties_here[:4])} "
            f"identify a potentially important {dominant_direction} "
            f"financial configuration requiring collective analysis.",

        "economic_mechanism":
            "The coalition combines heterogeneous signals and "
            "specialist interpretations that may jointly reveal "
            "information unavailable to any single member.",

        "why_now":
            "Current signals have generated sufficient affinity "
            "among otherwise heterogeneous specialists.",

        "complementarity":
            "Members contribute different analytical lenses, "
            "horizons, and market expertise.",

        "dissent":
            "Members agree that the issue deserves investigation "
            "but may disagree about causality, magnitude, timing, "
            "or the appropriate financial expression.",

        "internal_skeptic": members[-1],

        "confidence": avg_confidence,

        "lifespan_days": 100,

        "dissolution":
            "Dissolve or redesign the coalition if its principal "
            "evidence reverses, confidence collapses, or the "
            "financial regime changes materially."
    }


# ------------------------------------------------------------
# 2. Evaluate EVERY candidate alliance generated by Cell 6
# ------------------------------------------------------------

for idx, members in enumerate(communities):

    material = []

    for i in members:

        material.append({

            "id": i,

            "specialty": agents[i].specialty,

            "horizon": agents[i].horizon,

            "risk_temperament":
                agents[i].risk_temperament,

            "reputation":
                round(float(agents[i].reputation), 3),

            "belief":
                beliefs[i]
        })


    # --------------------------------------------------------
    # 3. Claude receives the candidate alliance
    # --------------------------------------------------------

    prompt = f"""
You are evaluating a possible temporary coalition inside an
autonomous financial institution.

CANDIDATE MEMBERS
-----------------
{json.dumps(material, indent=2)}

INSTITUTIONAL CHARTER
---------------------
{json.dumps(CHARTER, indent=2)}


COALITION-FORMATION PHILOSOPHY
------------------------------

The institution ENCOURAGES exploratory coalition formation.

A coalition DOES NOT require complete agreement.

A valid coalition requires only:

1. A financially meaningful shared question, opportunity,
   anomaly, vulnerability, or risk.

2. Complementary expertise that makes collective investigation
   more valuable than isolated analysis.

3. Enough common ground to formulate a testable financial thesis.

4. At least one meaningful source of internal disagreement,
   uncertainty, or alternative interpretation.

Internal disagreement is DESIRABLE.

Do NOT reject a coalition merely because:
- members disagree about causality,
- they have different horizons,
- they recommend different initial actions,
- their confidence levels differ,
- the investment thesis is still exploratory.

Those disagreements are precisely why a coalition may be useful.

Prefer FORMATION when there is a plausible financial reason
for these specialists to investigate the issue together.

Reject the coalition ONLY when:
- there is essentially no coherent financial question connecting
  the members,
- the group is overwhelmingly redundant,
- or the proposed alliance has no plausible informational value.


IMPORTANT
---------

Coalitions are TEMPORARY.

Agents may simultaneously participate in other coalitions.

The coalition does NOT yet receive capital.

Its purpose at this stage is to investigate, debate, and formulate
a proposition that may later be presented to the Investment Committee.


RETURN ONLY VALID JSON
----------------------

Return exactly one JSON object with these fields:

{{
    "exists": true or false,

    "name":
        "short institutional name for the coalition",

    "shared_question":
        "the financial question that brings these agents together",

    "shared_thesis":
        "their initial collective hypothesis",

    "economic_mechanism":
        "the causal mechanism they believe may connect the evidence",

    "why_now":
        "why the coalition should form under current conditions",

    "complementarity":
        "why these specialties are useful together",

    "dissent":
        "the most important unresolved disagreement",

    "internal_skeptic":
        integer member ID,

    "confidence":
        number between 0 and 1,

    "lifespan_days":
        integer between 30 and 250,

    "dissolution":
        "observable conditions under which the coalition should
         dissolve, mutate, or return its members to the population",

    "rejection_reason":
        "if exists=false explain why; otherwise empty string"
}}

Do not include markdown.
Do not include commentary outside the JSON.
"""


    # --------------------------------------------------------
    # 4. Ask Claude Sonnet 5
    # --------------------------------------------------------

    raw = llm_call(

        system="""
You are the Coalition Formation Council of a governed autonomous
financial institution.

Your job is NOT to minimize the number of coalitions.

Your job is to discover financially meaningful temporary alliances
among heterogeneous specialists.

You value:
- complementarity,
- diversity of information,
- productive disagreement,
- testable hypotheses,
- exploratory collaboration,
- and institutional adaptability.

You should be permissive toward plausible exploratory alliances,
while rejecting groups that truly lack a coherent financial purpose.

Never manufacture evidence that is not contained in the supplied
agent beliefs.
""",

        prompt=prompt,

        purpose=f"coalition_formation_{idx}",

        max_tokens=1000
    )


    # --------------------------------------------------------
    # 5. Parse Claude's response
    # --------------------------------------------------------

    c = parse_json_loose(raw)


    # --------------------------------------------------------
    # 6. Fallback if LLM unavailable or malformed
    # --------------------------------------------------------

    if not isinstance(c, dict):

        c = fallback_coalition(
            idx,
            members
        )


    # --------------------------------------------------------
    # 7. Validate / normalize fields
    # --------------------------------------------------------

    c.setdefault("exists", True)

    c.setdefault(
        "name",
        f"Emergent Coalition {idx + 1}"
    )

    c.setdefault(
        "shared_question",
        "What financially meaningful phenomenon connects "
        "these specialists?"
    )

    c.setdefault(
        "shared_thesis",
        "Complementary specialists identify a common "
        "financial opportunity or risk."
    )

    c.setdefault(
        "economic_mechanism",
        "Multiple specialist signals may reflect a common "
        "underlying financial mechanism."
    )

    c.setdefault(
        "why_now",
        "Current beliefs have created sufficient common interest."
    )

    c.setdefault(
        "complementarity",
        "Members contribute heterogeneous analytical perspectives."
    )

    c.setdefault(
        "dissent",
        "Causal interpretation remains contested."
    )

    c.setdefault(
        "internal_skeptic",
        members[-1]
    )

    c.setdefault(
        "confidence",
        0.55
    )

    c.setdefault(
        "lifespan_days",
        100
    )

    c.setdefault(
        "dissolution",
        "Dissolve if evidence or confidence collapses."
    )

    c.setdefault(
        "rejection_reason",
        ""
    )


    # Ensure skeptic actually belongs to coalition
    if c["internal_skeptic"] not in members:

        c["internal_skeptic"] = members[-1]


    # Keep confidence in valid range
    try:
        c["confidence"] = min(
            1.0,
            max(0.0, float(c["confidence"]))
        )
    except Exception:
        c["confidence"] = 0.55


    # --------------------------------------------------------
    # 8. Accept or reject coalition
    # --------------------------------------------------------

    if bool(c.get("exists", True)):

        c["id"] = f"C{len(coalitions) + 1:02d}"

        c["candidate_id"] = f"A{idx + 1:02d}"

        c["members"] = list(members)

        c["member_specialties"] = [
            agents[i].specialty
            for i in members
        ]

        c["formation_day"] = EVENT_DAY

        c["status"] = "ACTIVE"

        coalitions.append(c)

    else:

        rejected_coalitions.append({

            "candidate_id":
                f"A{idx + 1:02d}",

            "members":
                list(members),

            "specialties":
                [
                    agents[i].specialty
                    for i in members
                ],

            "reason":
                c.get(
                    "rejection_reason",
                    "No coherent coalition formed."
                )
        })


# ------------------------------------------------------------
# 9. Coalition summary
# ------------------------------------------------------------

coalition_summary = pd.DataFrame([

    {

        "id":
            c["id"],

        "name":
            c["name"],

        "members":
            len(c["members"]),

        "confidence":
            round(
                float(c["confidence"]),
                3
            ),

        "lifespan_days":
            c["lifespan_days"],

        "shared_question":
            c["shared_question"],

        "thesis":
            c["shared_thesis"],

        "dissent":
            c["dissent"]

    }

    for c in coalitions
])


print("=" * 70)

print(
    "ENDOGENOUS COALITION FORMATION"
)

print("=" * 70)

print(
    f"Candidate alliances evaluated : {len(communities)}"
)

print(
    f"Coalitions formed             : {len(coalitions)}"
)

print(
    f"Candidates rejected           : {len(rejected_coalitions)}"
)

if len(communities) > 0:

    print(
        f"Formation rate                : "
        f"{len(coalitions) / len(communities):.1%}"
    )


print("\nACTIVE COALITIONS")

display(coalition_summary)


# ------------------------------------------------------------
# 10. Show overlapping membership
# ------------------------------------------------------------

membership_count = {
    a.id: 0
    for a in agents
}

for c in coalitions:

    for i in c["members"]:

        membership_count[i] += 1


membership_df = pd.DataFrame([

    {

        "agent":
            a.id,

        "specialty":
            a.specialty,

        "coalitions":
            membership_count[a.id],

        "reputation":
            round(
                float(a.reputation),
                3
            )

    }

    for a in agents

]).sort_values(
    ["coalitions", "reputation"],
    ascending=[False, False]
)


print("\nOVERLAPPING AGENT MEMBERSHIP")

display(
    membership_df.head(20)
)


# ------------------------------------------------------------
# 11. Diagnostic: rejected candidates
# ------------------------------------------------------------

if rejected_coalitions:

    print("\nREJECTED CANDIDATE ALLIANCES")

    rejected_df = pd.DataFrame(
        rejected_coalitions
    )

    display(rejected_df)

else:

    print(
        "\nNo candidate alliances were rejected."
    )


# ------------------------------------------------------------
# 12. Institutional diagnostic
# ------------------------------------------------------------

active_agents = sum(
    1
    for n in membership_count.values()
    if n > 0
)

multi_coalition_agents = sum(
    1
    for n in membership_count.values()
    if n > 1
)

print("\nINSTITUTIONAL STRUCTURE")

print(
    f"Agents participating in at least one coalition: "
    f"{active_agents}/{N_AGENTS}"
)

print(
    f"Agents participating in multiple coalitions: "
    f"{multi_coalition_agents}/{N_AGENTS}"
)

if coalitions:

    avg_size = np.mean(
        [
            len(c["members"])
            for c in coalitions
        ]
    )

    print(
        f"Average coalition size: "
        f"{avg_size:.2f}"
    )

    print(
        "\nThe organization is now represented by overlapping, "
        "temporary coalitions rather than a fixed departmental "
        "partition."
    )

ENDOGENOUS COALITION FORMATION
Candidate alliances evaluated : 12
Coalitions formed             : 12
Candidates rejected           : 0
Formation rate                : 100.0%

ACTIVE COALITIONS


,id,name,members,confidence,lifespan_days,shared_question,thesis,dissent
0,C01,Emergent Coalition 1,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Global Macro, Rates, Sovereig...",Members agree that the issue deserves investig...
1,C02,Emergent Coalition 2,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Global Macro, Rates, Sovereig...",Members agree that the issue deserves investig...
2,C03,Emergent Coalition 3,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...
3,C04,Emergent Coalition 4,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...
4,C05,Emergent Coalition 5,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...
5,C06,Emergent Coalition 6,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Investment Grade Credit, Liqu...",Members agree that the issue deserves investig...
6,C07,Cross-Asset Signal Divergence Working Group,8,0.45,90,Do current mixed cross-asset signals (volatili...,"The convergence of elevated market volatility,...",Members disagree on which evidence class (cred...
7,C08,Emergent Coalition 8,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Rates, Sovereign Credit, Publ...",Members agree that the issue deserves investig...
8,C09,Emergent Coalition 9,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Bank Credit, Equity Quant, Eq...",Members agree that the issue deserves investig...
9,C10,Emergent Coalition 10,8,0.56,100,Do these complementary signals jointly indicat...,"Specialists from Sovereign Credit, Public Equi...",Members agree that the issue deserves investig...



OVERLAPPING AGENT MEMBERSHIP


,agent,specialty,coalitions,reputation
2,2,Sovereign Credit,5,0.5
3,3,Investment Grade Credit,5,0.5
6,6,Public Equity Fundamental,5,0.5
13,13,Volatility,5,0.5
18,18,Distressed Debt,5,0.5
24,24,Corporate Restructuring,5,0.5
29,29,Credit Risk,5,0.5
40,40,AI Economics,5,0.5
42,42,Commodities,5,0.5
17,17,Relative Value,4,0.5



No candidate alliances were rejected.

INSTITUTIONAL STRUCTURE
Agents participating in at least one coalition: 37/50
Agents participating in multiple coalitions: 18/50
Average coalition size: 8.00

The organization is now represented by overlapping, temporary coalitions rather than a fixed departmental partition.


## CODE UNIT 8 — Coalition debate, dissent and negotiated thesis

Formation should not eliminate disagreement. This unit forces every coalition to conduct an internal debate before approaching the Investment Committee. Claude acts as a deliberative chair. It receives the original member beliefs and must identify agreement, contradiction, the strongest objection, evidence still needed, and a negotiated thesis that does not erase material dissent.

This is an important departure from conventional ensemble modeling. Disagreement is treated as information rather than noise. If a macro agent believes a liquidity deterioration signals recession while a credit specialist believes it is temporary technical pressure, averaging their confidence would destroy the distinction. The coalition instead records a conditional thesis and an explicit dissenting view.

The debate also identifies the preferred financial expression of the coalition’s thesis. A stress view might be expressed through volatility protection rather than simply shorting equities. A technology thesis might favor equities or another available synthetic asset.

Because dissent is preserved, later outcomes can reveal whether minority opinions were valuable. An institution that learns to recognize high-quality dissent may become more robust than one that rewards only consensus.

In [20]:
# ============================================================
# CODE UNIT 8 — COALITION DEBATE, DISSENT AND NEGOTIATED THESIS
# ============================================================
#
# PURPOSE
# -------
# Each endogenous coalition formed in Cell 7 now becomes a
# temporary deliberative institution.
#
# Members do NOT simply average their views.
#
# Instead, the coalition must:
#
#   1. identify genuine areas of agreement,
#   2. identify material disagreements,
#   3. expose the strongest objection,
#   4. let an internal skeptic challenge the dominant thesis,
#   5. identify missing evidence,
#   6. determine what would falsify the thesis,
#   7. negotiate a provisional collective thesis,
#   8. translate that thesis into a financial expression.
#
# IMPORTANT:
# Consensus is NOT required.
# Dissent is preserved as institutional information.
#
# The output of this cell becomes the intellectual foundation
# for the formal investment memorandum in Cell 9.
# ============================================================


debate_records = []


# ------------------------------------------------------------
# 1. Deterministic fallback debate
# ------------------------------------------------------------

def fallback_debate(c):

    members = c["members"]

    member_beliefs = [
        beliefs[i]
        for i in members
    ]

    confidences = [
        float(
            b.get(
                "confidence",
                0.50
            )
        )
        for b in member_beliefs
    ]

    directions = [
        b.get(
            "direction",
            "neutral"
        )
        for b in member_beliefs
    ]


    # --------------------------------------------------------
    # Determine broad directional tendency
    # --------------------------------------------------------

    direction_counts = {}

    for d in directions:

        direction_counts[d] = (
            direction_counts.get(d, 0) + 1
        )


    dominant_direction = max(
        direction_counts,
        key=direction_counts.get
    )


    # --------------------------------------------------------
    # Determine preferred asset expression
    # --------------------------------------------------------

    defensive_count = sum(
        d == "defensive"
        for d in directions
    )

    risk_on_count = sum(
        d == "risk_on"
        for d in directions
    )

    relative_value_count = sum(
        d == "relative_value"
        for d in directions
    )


    if defensive_count > max(
        risk_on_count,
        relative_value_count
    ):

        preferred_expression = "VOL_HEDGE"

        expression_direction = "long"


    elif relative_value_count > risk_on_count:

        preferred_expression = "DURATION"

        expression_direction = "short"


    else:

        preferred_expression = "EQUITY"

        expression_direction = "long"


    avg_confidence = float(
        np.mean(confidences)
    )


    # Penalize fallback confidence when
    # directional disagreement is high

    directional_diversity = (
        len(set(directions))
        /
        max(1, len(directions))
    )

    negotiated_confidence = (
        avg_confidence
        *
        (
            1.0
            -
            0.25 * directional_diversity
        )
    )


    return {

        "agreement":
            [
                c.get(
                    "shared_thesis",
                    "The coalition identifies a financially "
                    "meaningful opportunity or risk."
                )
            ],

        "disagreement":
            [
                c.get(
                    "dissent",
                    "Members disagree about timing, magnitude, "
                    "or financial expression."
                )
            ],

        "strongest_objection":
            c.get(
                "dissent",
                "The apparent signal may be temporary or "
                "incorrectly interpreted."
            ),

        "skeptic_challenge":
            "The coalition may be confusing correlation with "
            "a persistent causal mechanism.",

        "response_to_skeptic":
            "The thesis should therefore remain provisional "
            "until additional evidence confirms persistence.",

        "negotiated_thesis":
            c.get(
                "shared_thesis",
                "Complementary evidence suggests a potentially "
                "actionable financial configuration."
            ),

        "dissent_view":
            c.get(
                "dissent",
                "Material uncertainty remains."
            ),

        "unresolved_uncertainties":
            [
                "Persistence of current signals",
                "Timing of the expected financial effect",
                "Possibility of regime change"
            ],

        "evidence_needed":
            [
                "Persistence of relevant market signals",
                "Confirmation from independent indicators",
                "Absence of material regime reversal"
            ],

        "falsification_conditions":
            [
                "Core evidence reverses",
                "Expected transmission mechanism fails",
                "Observed market behavior contradicts thesis"
            ],

        "confidence":
            float(
                np.clip(
                    negotiated_confidence,
                    0.05,
                    0.95
                )
            ),

        "preferred_expression":
            preferred_expression,

        "expression_direction":
            expression_direction,

        "implementation_logic":
            "Express the coalition thesis through the asset "
            "most closely associated with its dominant economic "
            "mechanism while maintaining explicit risk controls.",

        "minority_report":
            c.get(
                "dissent",
                "A minority of members remains unconvinced."
            ),

        "debate_quality":
            0.50
    }


# ------------------------------------------------------------
# 2. Conduct one institutional debate per coalition
# ------------------------------------------------------------

for c in coalitions:

    members = c["members"]


    # --------------------------------------------------------
    # Assemble complete member positions
    # --------------------------------------------------------

    material = []

    for i in members:

        material.append({

            "id":
                i,

            "specialty":
                agents[i].specialty,

            "horizon":
                agents[i].horizon,

            "risk_temperament":
                agents[i].risk_temperament,

            "reputation":
                round(
                    float(
                        agents[i].reputation
                    ),
                    3
                ),

            "epistemic_score":
                round(
                    float(
                        agents[i].epistemic_score
                    ),
                    3
                ),

            "belief":
                beliefs[i]
        })


    internal_skeptic = c.get(
        "internal_skeptic",
        members[-1]
    )


    # --------------------------------------------------------
    # 3. Build the deliberation prompt
    # --------------------------------------------------------

    prompt = f"""
You are chairing a formal investment debate inside an
autonomous financial institution.

COALITION
---------
ID:
{c["id"]}

NAME:
{c["name"]}

SHARED QUESTION:
{c.get("shared_question", "")}

INITIAL SHARED THESIS:
{c.get("shared_thesis", "")}

PROPOSED ECONOMIC MECHANISM:
{c.get("economic_mechanism", "")}

WHY THE COALITION FORMED:
{c.get("why_now", "")}

EXPECTED COMPLEMENTARITY:
{c.get("complementarity", "")}

INITIAL DISSENT:
{c.get("dissent", "")}

INTERNAL SKEPTIC:
Agent {internal_skeptic}

MEMBER POSITIONS
----------------
{json.dumps(material, indent=2)}


INSTITUTIONAL DEBATE RULES
--------------------------

This is NOT a consensus-generation exercise.

The coalition exists because heterogeneous specialists may
possess different pieces of economically relevant information.

You must preserve those differences.

The debate must distinguish:

1. AGREEMENT
   What do members genuinely agree about?

2. DISAGREEMENT
   Where do interpretations materially diverge?

3. CAUSAL MECHANISM
   What mechanism could connect the observed evidence to
   financial outcomes?

4. STRONGEST OBJECTION
   What is the strongest intellectually serious argument
   against the emerging coalition thesis?

5. INTERNAL SKEPTIC
   Agent {internal_skeptic} must challenge the dominant
   interpretation.

6. RESPONSE TO THE SKEPTIC
   Explain whether the coalition can answer the challenge.
   Do NOT pretend the challenge has been resolved if it has not.

7. MISSING EVIDENCE
   Identify observations that would materially increase or
   decrease confidence.

8. FALSIFICATION
   State observable conditions under which the coalition
   should conclude that its thesis is wrong.

9. NEGOTIATED THESIS
   Produce the strongest proposition the coalition can
   collectively defend WITHOUT erasing material disagreement.

10. FINANCIAL EXPRESSION
    Translate the negotiated thesis into ONE primary asset
    expression.

11. MINORITY REPORT
    Preserve the strongest remaining dissenting interpretation.

12. CONFIDENCE
    Confidence must reflect the quality and consistency of
    evidence — not rhetorical eloquence.


IMPORTANT
---------

A negotiated thesis may be:

- strong,
- conditional,
- weak,
- or explicitly uncertain.

Do NOT force a high-confidence conclusion.

Do NOT manufacture evidence.

Do NOT reward eloquence over evidence.

Disagreement can be valuable information.

The final financial expression is provisional and will later
be evaluated by the Investment Committee and by realized market
outcomes.


ALLOWED ASSET EXPRESSIONS
-------------------------

EQUITY
DURATION
CREDIT
VOL_HEDGE
CRYPTO
CASH


ALLOWED DIRECTIONS
------------------

long
short


RETURN ONLY VALID JSON
----------------------

Return exactly one JSON object:

{{
    "agreement": [
        "agreement point 1",
        "agreement point 2"
    ],

    "disagreement": [
        "material disagreement 1",
        "material disagreement 2"
    ],

    "causal_mechanism":
        "the proposed economic transmission mechanism",

    "strongest_objection":
        "strongest serious challenge to the thesis",

    "skeptic_challenge":
        "challenge raised by the internal skeptic",

    "response_to_skeptic":
        "coalition response, including unresolved issues",

    "negotiated_thesis":
        "the strongest thesis the coalition can jointly defend",

    "dissent_view":
        "the principal dissenting interpretation",

    "unresolved_uncertainties": [
        "uncertainty 1",
        "uncertainty 2"
    ],

    "evidence_needed": [
        "evidence item 1",
        "evidence item 2",
        "evidence item 3"
    ],

    "falsification_conditions": [
        "observable falsification condition 1",
        "observable falsification condition 2"
    ],

    "confidence":
        number between 0 and 1,

    "preferred_expression":
        "EQUITY|DURATION|CREDIT|VOL_HEDGE|CRYPTO|CASH",

    "expression_direction":
        "long|short",

    "implementation_logic":
        "why this asset and direction express the thesis",

    "minority_report":
        "short statement preserving the strongest minority view",

    "debate_quality":
        number between 0 and 1
}}

Do not include markdown.
Do not include commentary outside the JSON.
"""


    # --------------------------------------------------------
    # 4. Claude Sonnet 5 acts as deliberative chair
    # --------------------------------------------------------

    raw = llm_call(

        system="""
You are the Deliberative Chair of an institutional investment
coalition.

Your purpose is not to manufacture consensus.

Your purpose is to convert heterogeneous specialist knowledge
into a disciplined, testable financial proposition while
preserving uncertainty and dissent.

You must distinguish:
- evidence from rhetoric,
- consensus from compromise,
- correlation from causal mechanism,
- confidence from persuasion,
- and uncertainty from ignorance.

The strongest objection must receive serious treatment.

A minority report is institutionally valuable and must not
be erased merely because most members favor another view.

Never invent market information that is not contained in the
provided beliefs.
""",

        prompt=prompt,

        purpose=f"debate_{c['id']}",

        max_tokens=1500
    )


    # --------------------------------------------------------
    # 5. Parse debate
    # --------------------------------------------------------

    d = parse_json_loose(raw)


    # --------------------------------------------------------
    # 6. Deterministic fallback
    # --------------------------------------------------------

    if not isinstance(d, dict):

        d = fallback_debate(c)


    # --------------------------------------------------------
    # 7. Normalize output
    # --------------------------------------------------------

    d.setdefault(
        "agreement",
        [
            c.get(
                "shared_thesis",
                "The coalition identifies a relevant "
                "financial phenomenon."
            )
        ]
    )

    d.setdefault(
        "disagreement",
        [
            c.get(
                "dissent",
                "Interpretation remains contested."
            )
        ]
    )

    d.setdefault(
        "causal_mechanism",
        c.get(
            "economic_mechanism",
            "The proposed relationship requires "
            "additional validation."
        )
    )

    d.setdefault(
        "strongest_objection",
        c.get(
            "dissent",
            "The apparent relationship may not persist."
        )
    )

    d.setdefault(
        "skeptic_challenge",
        d["strongest_objection"]
    )

    d.setdefault(
        "response_to_skeptic",
        "The challenge remains partially unresolved."
    )

    d.setdefault(
        "negotiated_thesis",
        c.get(
            "shared_thesis",
            "The coalition identifies a provisional "
            "financial thesis."
        )
    )

    d.setdefault(
        "dissent_view",
        c.get(
            "dissent",
            "Material disagreement remains."
        )
    )

    d.setdefault(
        "unresolved_uncertainties",
        [
            "Persistence",
            "Timing",
            "Regime stability"
        ]
    )

    d.setdefault(
        "evidence_needed",
        [
            "Persistence of relevant signals",
            "Independent confirmation"
        ]
    )

    d.setdefault(
        "falsification_conditions",
        [
            "Core evidence reverses"
        ]
    )

    d.setdefault(
        "confidence",
        float(
            np.mean(
                [
                    beliefs[i].get(
                        "confidence",
                        0.50
                    )
                    for i in members
                ]
            )
        )
    )

    d.setdefault(
        "preferred_expression",
        "CASH"
    )

    d.setdefault(
        "expression_direction",
        "long"
    )

    d.setdefault(
        "implementation_logic",
        "Expression follows the coalition's "
        "dominant economic mechanism."
    )

    d.setdefault(
        "minority_report",
        d["dissent_view"]
    )

    d.setdefault(
        "debate_quality",
        0.50
    )


    # --------------------------------------------------------
    # 8. Validate numerical fields
    # --------------------------------------------------------

    try:

        d["confidence"] = float(
            np.clip(
                float(d["confidence"]),
                0.0,
                1.0
            )
        )

    except Exception:

        d["confidence"] = 0.50


    try:

        d["debate_quality"] = float(
            np.clip(
                float(d["debate_quality"]),
                0.0,
                1.0
            )
        )

    except Exception:

        d["debate_quality"] = 0.50


    # --------------------------------------------------------
    # 9. Validate asset expression
    # --------------------------------------------------------

    allowed_assets = {
        "EQUITY",
        "DURATION",
        "CREDIT",
        "VOL_HEDGE",
        "CRYPTO",
        "CASH"
    }

    if d["preferred_expression"] not in allowed_assets:

        d["preferred_expression"] = "CASH"


    if d["expression_direction"] not in {
        "long",
        "short"
    }:

        d["expression_direction"] = "long"


    # --------------------------------------------------------
    # 10. Attach debate to coalition
    # --------------------------------------------------------

    c["debate"] = d


    # --------------------------------------------------------
    # 11. Create diagnostic record
    # --------------------------------------------------------

    debate_records.append({

        "coalition":
            c["id"],

        "name":
            c["name"],

        "members":
            len(c["members"]),

        "confidence":
            round(
                d["confidence"],
                3
            ),

        "debate_quality":
            round(
                d["debate_quality"],
                3
            ),

        "expression":
            d["preferred_expression"],

        "direction":
            d["expression_direction"],

        "negotiated_thesis":
            d["negotiated_thesis"],

        "strongest_objection":
            d["strongest_objection"],

        "minority_report":
            d["minority_report"]
    })


# ------------------------------------------------------------
# 12. Institutional debate summary
# ------------------------------------------------------------

debate_df = pd.DataFrame(
    debate_records
)


print("=" * 78)

print(
    "COALITION DELIBERATION — "
    "NEGOTIATED THESES AND PRESERVED DISSENT"
)

print("=" * 78)


print(
    f"Coalitions debated: "
    f"{len(coalitions)}"
)


if len(debate_df):

    print(
        f"Average negotiated confidence: "
        f"{debate_df['confidence'].mean():.3f}"
    )

    print(
        f"Average debate quality: "
        f"{debate_df['debate_quality'].mean():.3f}"
    )


display(
    debate_df
)


# ------------------------------------------------------------
# 13. Detailed coalition debate reports
# ------------------------------------------------------------

for c in coalitions:

    d = c["debate"]

    print("\n" + "=" * 78)

    print(
        f"{c['id']} — {c['name']}"
    )

    print("=" * 78)

    print(
        "\nNEGOTIATED THESIS\n"
    )

    print(
        d["negotiated_thesis"]
    )

    print(
        "\nSTRONGEST OBJECTION\n"
    )

    print(
        d["strongest_objection"]
    )

    print(
        "\nMINORITY REPORT\n"
    )

    print(
        d["minority_report"]
    )

    print(
        "\nFINANCIAL EXPRESSION\n"
    )

    print(
        f"{d['expression_direction'].upper()} "
        f"{d['preferred_expression']}"
    )

    print(
        "\nCONFIDENCE:",
        round(
            d["confidence"],
            3
        )
    )

    print(
        "DEBATE QUALITY:",
        round(
            d["debate_quality"],
            3
        )
    )


# ------------------------------------------------------------
# 14. Diversity diagnostic
# ------------------------------------------------------------

expression_distribution = (
    debate_df["expression"]
    .value_counts()
    .to_dict()
    if len(debate_df)
    else {}
)

direction_distribution = (
    debate_df["direction"]
    .value_counts()
    .to_dict()
    if len(debate_df)
    else {}
)


print("\n" + "=" * 78)

print(
    "INSTITUTIONAL DIVERSITY DIAGNOSTIC"
)

print("=" * 78)

print(
    "Asset expressions:",
    expression_distribution
)

print(
    "Directions:",
    direction_distribution
)


if len(expression_distribution) <= 1:

    print(
        "\nWARNING: Coalitions have converged on a single "
        "asset expression. This may indicate herding."
    )

else:

    print(
        "\nCoalitions preserve multiple financial expressions."
    )


if len(direction_distribution) <= 1:

    print(
        "WARNING: All coalitions have adopted the same "
        "direction. Investigate possible institutional herding."
    )

else:

    print(
        "Directional disagreement survives coalition deliberation."
    )

COALITION DELIBERATION — NEGOTIATED THESES AND PRESERVED DISSENT
Coalitions debated: 12
Average negotiated confidence: 0.543
Average debate quality: 0.500


,coalition,name,members,confidence,debate_quality,expression,direction,negotiated_thesis,strongest_objection,minority_report
0,C01,Emergent Coalition 1,8,0.543,0.5,EQUITY,long,"Specialists from Global Macro, Rates, Sovereig...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
1,C02,Emergent Coalition 2,8,0.543,0.5,EQUITY,long,"Specialists from Global Macro, Rates, Sovereig...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
2,C03,Emergent Coalition 3,8,0.543,0.5,EQUITY,long,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
3,C04,Emergent Coalition 4,8,0.543,0.5,EQUITY,long,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
4,C05,Emergent Coalition 5,8,0.543,0.5,EQUITY,long,"Specialists from Investment Grade Credit, Rela...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
5,C06,Emergent Coalition 6,8,0.543,0.5,EQUITY,long,"Specialists from Investment Grade Credit, Liqu...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
6,C07,Cross-Asset Signal Divergence Working Group,8,0.543,0.5,EQUITY,long,"The convergence of elevated market volatility,...",Members disagree on which evidence class (cred...,Members disagree on which evidence class (cred...
7,C08,Emergent Coalition 8,8,0.543,0.5,EQUITY,long,"Specialists from Rates, Sovereign Credit, Publ...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
8,C09,Emergent Coalition 9,8,0.543,0.5,EQUITY,long,"Specialists from Bank Credit, Equity Quant, Eq...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...
9,C10,Emergent Coalition 10,8,0.543,0.5,EQUITY,long,"Specialists from Sovereign Credit, Public Equi...",Members agree that the issue deserves investig...,Members agree that the issue deserves investig...



C01 — Emergent Coalition 1

NEGOTIATED THESIS

Specialists from Global Macro, Rates, Sovereign Credit, Public Equity Fundamental identify a potentially important neutral financial configuration requiring collective analysis.

STRONGEST OBJECTION

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropriate financial expression.

MINORITY REPORT

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropriate financial expression.

FINANCIAL EXPRESSION

LONG EQUITY

CONFIDENCE: 0.543
DEBATE QUALITY: 0.5

C02 — Emergent Coalition 2

NEGOTIATED THESIS

Specialists from Global Macro, Rates, Sovereign Credit, Public Equity Fundamental identify a potentially important neutral financial configuration requiring collective analysis.

STRONGEST OBJECTION

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropri

## CODE UNIT 9 — Investment memoranda and capital requests

A coalition becomes economically consequential only when it asks for scarce capital. This unit requires each coalition to transform its negotiated thesis into a formal investment memorandum. The memo contains the thesis, evidence strength, persuasive quality, proposed asset and direction, requested capital, horizon, downside, falsification criterion, exit condition, and strongest opposing case.

Language matters deliberately. Real financial institutions allocate resources partly through argument. Coalitions are therefore allowed to be persuasive. Yet persuasive quality is recorded separately from evidence strength so that the experiment can later distinguish eloquence from accuracy.

The request is bounded by the institutional constitution. A coalition cannot simply ask for the entire balance sheet. Quantitative context such as recent asset volatility is attached to the prompt, grounding the narrative in observable risk.

The result is an internal market for ideas. Multiple coalitions may compete for the same capital or recommend exposures that overlap. The committee must therefore judge not merely whether each proposal is attractive in isolation but whether it deserves resources relative to alternatives.

In [21]:
# ============================================================
# CODE UNIT 9 — INVESTMENT MEMORANDA AND CAPITAL REQUESTS
# ============================================================
#
# PURPOSE
# -------
# Convert each coalition's negotiated thesis from Cell 8 into
# a formal institutional investment memorandum and capital request.
#
# The coalition must now move from:
#
#       "We believe something"
#
# to:
#
#       "Here is the evidence,
#        here is what could make us wrong,
#        here is how we would express the thesis,
#        here is how much capital we want,
#        and here is why the institution should risk it."
#
# CRITICAL PRINCIPLES
# -------------------
#
# 1. Evidence quality is NOT persuasive quality.
# 2. High confidence does NOT automatically justify large capital.
# 3. Higher volatility should generally reduce position size.
# 4. Dissent and minority reports must remain visible.
# 5. Every thesis must contain falsification and exit conditions.
# 6. Capital requests compete for scarce institutional capital.
# 7. Requests are proposals — Cell 10 Investment Committee decides.
#
# ============================================================


# ------------------------------------------------------------
# 1. Market risk context
# ------------------------------------------------------------

lookback_start = max(
    0,
    EVENT_DAY - 60
)

recent = world.loc[
    lookback_start:EVENT_DAY,
    assets
]


# Annualized volatility
asset_vol = (
    recent
    .std()
    * np.sqrt(252)
).round(4).to_dict()


# Recent cumulative returns
asset_recent_return = (
    (1 + recent)
    .prod()
    - 1
).round(4).to_dict()


# Simple drawdown diagnostic
asset_drawdown = {}

for asset in assets:

    wealth = (
        1 + recent[asset]
    ).cumprod()

    peak = wealth.cummax()

    dd = (
        wealth / peak
        - 1
    )

    asset_drawdown[asset] = round(
        float(dd.min()),
        4
    )


print("=" * 78)

print(
    "MARKET CONTEXT FOR CAPITAL REQUESTS"
)

print("=" * 78)

market_context_df = pd.DataFrame({

    "annualized_volatility":
        asset_vol,

    "recent_return":
        asset_recent_return,

    "recent_max_drawdown":
        asset_drawdown

})

display(
    market_context_df
)


# ------------------------------------------------------------
# 2. Capital constraints
# ------------------------------------------------------------

MAX_REQUEST = (
    INITIAL_CAPITAL
    * MAX_COALITION_CAPITAL
)

MIN_REQUEST = (
    INITIAL_CAPITAL
    * 0.01
)

# Requests will normally be rounded to $25 million increments
CAPITAL_INCREMENT = 25_000_000


def round_capital(x):

    return (
        round(
            float(x)
            / CAPITAL_INCREMENT
        )
        * CAPITAL_INCREMENT
    )


# ------------------------------------------------------------
# 3. Deterministic risk-aware sizing benchmark
# ------------------------------------------------------------

def benchmark_capital_request(c):

    """
    Produce a deterministic benchmark capital request.

    This is NOT the final request.

    Claude receives this benchmark as an institutional anchor,
    but may recommend a different amount if it explains why.
    """

    d = c["debate"]

    confidence = float(
        d.get(
            "confidence",
            0.50
        )
    )

    debate_quality = float(
        d.get(
            "debate_quality",
            0.50
        )
    )

    asset = d.get(
        "preferred_expression",
        "CASH"
    )


    # --------------------------------------------------------
    # Volatility adjustment
    # --------------------------------------------------------

    vol = float(
        asset_vol.get(
            asset,
            0.15
        )
    )

    # Lower volatility permits larger benchmark allocations.
    # Clip prevents extreme synthetic values.

    vol_penalty = np.clip(
        0.15
        / max(vol, 0.05),
        0.40,
        1.40
    )


    # --------------------------------------------------------
    # Coalition breadth adjustment
    # --------------------------------------------------------

    member_count = len(
        c["members"]
    )

    breadth_factor = np.clip(
        member_count / 5.0,
        0.70,
        1.30
    )


    # --------------------------------------------------------
    # Base conviction
    # --------------------------------------------------------

    conviction = (
        0.60 * confidence
        +
        0.40 * debate_quality
    )


    # --------------------------------------------------------
    # Benchmark sizing
    # --------------------------------------------------------

    benchmark_fraction = (
        MAX_COALITION_CAPITAL
        *
        conviction
        *
        vol_penalty
        *
        breadth_factor
    )


    # No coalition can request above institutional cap

    benchmark_fraction = np.clip(
        benchmark_fraction,
        0.01,
        MAX_COALITION_CAPITAL
    )


    benchmark = (
        INITIAL_CAPITAL
        * benchmark_fraction
    )


    return round_capital(
        benchmark
    )


# ------------------------------------------------------------
# 4. Deterministic fallback memorandum
# ------------------------------------------------------------

def fallback_memo(c):

    d = c["debate"]

    asset = d.get(
        "preferred_expression",
        "CASH"
    )

    direction = d.get(
        "expression_direction",
        "long"
    )

    confidence = float(
        d.get(
            "confidence",
            0.50
        )
    )

    debate_quality = float(
        d.get(
            "debate_quality",
            0.50
        )
    )

    benchmark = benchmark_capital_request(
        c
    )


    # Evidence quality deliberately differs
    # from persuasive quality.

    evidence_strength = np.clip(
        0.65 * confidence
        +
        0.35 * debate_quality,
        0.0,
        1.0
    )

    persuasive_quality = np.clip(
        0.45
        +
        0.25 * debate_quality,
        0.0,
        1.0
    )


    return {

        "coalition_id":
            c["id"],

        "coalition_name":
            c["name"],

        "thesis":
            d["negotiated_thesis"],

        "economic_mechanism":
            d.get(
                "causal_mechanism",
                c.get(
                    "economic_mechanism",
                    ""
                )
            ),

        "evidence_strength":
            float(
                evidence_strength
            ),

        "persuasive_quality":
            float(
                persuasive_quality
            ),

        "asset":
            asset,

        "direction":
            direction,

        "requested_capital":
            benchmark,

        "benchmark_capital":
            benchmark,

        "horizon_days":
            60,

        "expected_catalyst":
            "Persistence or strengthening of the "
            "coalition's principal signals.",

        "downside":
            d.get(
                "strongest_objection",
                "The investment thesis fails."
            ),

        "falsification":
            "; ".join(
                d.get(
                    "falsification_conditions",
                    [
                        "Core evidence reverses."
                    ]
                )
            ),

        "exit_condition":
            "Exit if falsification conditions occur "
            "or coalition confidence falls materially.",

        "strongest_opposing_case":
            d.get(
                "minority_report",
                d.get(
                    "dissent_view",
                    ""
                )
            ),

        "minority_report":
            d.get(
                "minority_report",
                ""
            ),

        "key_uncertainties":
            d.get(
                "unresolved_uncertainties",
                []
            ),

        "evidence_needed":
            d.get(
                "evidence_needed",
                []
            ),

        "risk_controls":
            [
                "Position-size discipline",
                "Explicit falsification monitoring",
                "Coalition confidence review"
            ],

        "memo":
            d["negotiated_thesis"]
    }


# ------------------------------------------------------------
# 5. Generate institutional investment memoranda
# ------------------------------------------------------------

memos = []


for c in coalitions:

    d = c["debate"]

    benchmark = benchmark_capital_request(
        c
    )

    asset = d.get(
        "preferred_expression",
        "CASH"
    )


    # --------------------------------------------------------
    # Market information relevant to proposed asset
    # --------------------------------------------------------

    proposed_asset_context = {

        "asset":
            asset,

        "annualized_volatility":
            asset_vol.get(
                asset,
                None
            ),

        "recent_return":
            asset_recent_return.get(
                asset,
                None
            ),

        "recent_max_drawdown":
            asset_drawdown.get(
                asset,
                None
            )
    }


    # --------------------------------------------------------
    # Assemble coalition information
    # --------------------------------------------------------

    coalition_context = {

        "coalition_id":
            c["id"],

        "coalition_name":
            c["name"],

        "members":
            [
                {
                    "id":
                        i,

                    "specialty":
                        agents[i].specialty,

                    "reputation":
                        round(
                            float(
                                agents[i].reputation
                            ),
                            3
                        ),

                    "epistemic_score":
                        round(
                            float(
                                agents[i].epistemic_score
                            ),
                            3
                        )
                }

                for i in c["members"]
            ],

        "shared_question":
            c.get(
                "shared_question",
                ""
            ),

        "debate":
            d
    }


    # --------------------------------------------------------
    # 6. Institutional memorandum prompt
    # --------------------------------------------------------

    prompt = f"""
You are preparing a formal capital request for the Investment
Committee of an autonomous financial institution.

COALITION
---------
{json.dumps(coalition_context, indent=2)}

MARKET CONTEXT FOR PROPOSED ASSET
---------------------------------
{json.dumps(proposed_asset_context, indent=2)}

INSTITUTIONAL CAPITAL
---------------------
Total institutional capital:
${INITIAL_CAPITAL:,.0f}

Maximum capital request by any single coalition:
${MAX_REQUEST:,.0f}

Minimum economically meaningful request:
${MIN_REQUEST:,.0f}

Risk-aware deterministic benchmark request:
${benchmark:,.0f}


TASK
----

Transform the coalition's negotiated thesis into a rigorous
institutional investment memorandum.

The coalition is COMPETING for scarce capital.

The Investment Committee in the next stage may:

- APPROVE,
- RESIZE,
- impose CONDITIONS,
- or REJECT the request.

Therefore the memorandum must be persuasive, but it must NOT
hide uncertainty or dissent.


CAPITAL-SIZING PRINCIPLES
-------------------------

The requested capital should reflect:

1. strength of evidence,
2. negotiated confidence,
3. quality of coalition debate,
4. asset volatility,
5. downside asymmetry,
6. uncertainty,
7. investment horizon,
8. strength of the opposing case,
9. quality of falsification criteria,
10. institutional capital constraints.

Higher confidence does NOT automatically justify a larger
position.

Higher volatility should generally justify a smaller position.

Strong unresolved dissent should generally reduce requested
capital unless the proposed position itself provides portfolio
protection.

The deterministic benchmark is an anchor — NOT a command.

You may request more or less than the benchmark, but the amount
must be economically justified.

Never request more than:
${MAX_REQUEST:,.0f}


EVIDENCE VS. PERSUASION
-----------------------

You MUST independently score:

EVIDENCE_STRENGTH:
How strongly does the supplied information support the thesis?

PERSUASIVE_QUALITY:
How compellingly has the coalition presented its argument?

These are NOT the same thing.

A beautifully argued memorandum can have weak evidence.

A poorly articulated memorandum can contain strong evidence.

Do not allow rhetorical quality to substitute for evidence.


RISK DISCIPLINE
---------------

The memorandum must explicitly state:

- what could go wrong,
- what evidence would falsify the thesis,
- when the position should be exited,
- what evidence is still missing,
- the strongest opposing case,
- and the minority report.

Do not erase dissent from Cell 8.


RETURN ONLY VALID JSON
----------------------

Return exactly one JSON object:

{{
    "coalition_id":
        "{c['id']}",

    "coalition_name":
        "{c['name']}",

    "thesis":
        "concise investment thesis",

    "economic_mechanism":
        "causal mechanism connecting evidence to expected outcome",

    "evidence_strength":
        number between 0 and 1,

    "persuasive_quality":
        number between 0 and 1,

    "asset":
        "EQUITY|DURATION|CREDIT|VOL_HEDGE|CRYPTO|CASH",

    "direction":
        "long|short",

    "requested_capital":
        numeric USD amount,

    "horizon_days":
        integer between 10 and 365,

    "expected_catalyst":
        "what should cause the thesis to become economically visible",

    "downside":
        "principal downside scenario",

    "falsification":
        "observable evidence demonstrating that the thesis is wrong",

    "exit_condition":
        "conditions requiring reduction or exit",

    "strongest_opposing_case":
        "strongest serious argument against the position",

    "minority_report":
        "strongest remaining internal dissent",

    "key_uncertainties": [
        "uncertainty 1",
        "uncertainty 2"
    ],

    "evidence_needed": [
        "additional evidence 1",
        "additional evidence 2"
    ],

    "risk_controls": [
        "risk control 1",
        "risk control 2",
        "risk control 3"
    ],

    "memo":
        "institutional investment memorandum, maximum 220 words"
}}

Do not include markdown.
Do not include commentary outside the JSON.
"""


    # --------------------------------------------------------
    # 7. Claude Sonnet 5 writes the memorandum
    # --------------------------------------------------------

    raw = llm_call(

        system="""
You are a senior institutional investment strategist preparing
capital requests for a governed autonomous financial institution.

Your job is to transform coalition research into disciplined,
decision-ready investment memoranda.

You must be persuasive without becoming promotional.

You must preserve:
- uncertainty,
- falsifiability,
- downside analysis,
- minority views,
- and the strongest opposing argument.

Evidence quality and rhetorical quality must be evaluated
separately.

Capital is scarce.

Do not manufacture evidence.
Do not hide disagreement.
Do not automatically request the maximum allocation.
""",

        prompt=prompt,

        purpose=f"memo_{c['id']}",

        max_tokens=1500
    )


    # --------------------------------------------------------
    # 8. Parse response
    # --------------------------------------------------------

    m = parse_json_loose(
        raw
    )


    # --------------------------------------------------------
    # 9. Fallback if Claude unavailable
    # --------------------------------------------------------

    if not isinstance(
        m,
        dict
    ):

        m = fallback_memo(
            c
        )


    # --------------------------------------------------------
    # 10. Normalize mandatory fields
    # --------------------------------------------------------

    fallback = fallback_memo(
        c
    )

    for key, value in fallback.items():

        if key not in m:

            m[key] = value


    # --------------------------------------------------------
    # 11. Force coalition identity
    # --------------------------------------------------------

    m["coalition_id"] = c["id"]

    m["coalition_name"] = c["name"]


    # --------------------------------------------------------
    # 12. Validate evidence and persuasion scores
    # --------------------------------------------------------

    try:

        m["evidence_strength"] = float(
            np.clip(
                float(
                    m["evidence_strength"]
                ),
                0.0,
                1.0
            )
        )

    except Exception:

        m["evidence_strength"] = fallback[
            "evidence_strength"
        ]


    try:

        m["persuasive_quality"] = float(
            np.clip(
                float(
                    m["persuasive_quality"]
                ),
                0.0,
                1.0
            )
        )

    except Exception:

        m["persuasive_quality"] = fallback[
            "persuasive_quality"
        ]


    # --------------------------------------------------------
    # 13. Validate asset and direction
    # --------------------------------------------------------

    allowed_assets = {
        "EQUITY",
        "DURATION",
        "CREDIT",
        "VOL_HEDGE",
        "CRYPTO",
        "CASH"
    }


    if m["asset"] not in allowed_assets:

        m["asset"] = d.get(
            "preferred_expression",
            "CASH"
        )


    if m["direction"] not in {
        "long",
        "short"
    }:

        m["direction"] = d.get(
            "expression_direction",
            "long"
        )


    # --------------------------------------------------------
    # 14. Validate horizon
    # --------------------------------------------------------

    try:

        m["horizon_days"] = int(
            np.clip(
                int(
                    m["horizon_days"]
                ),
                10,
                365
            )
        )

    except Exception:

        m["horizon_days"] = 60


    # --------------------------------------------------------
    # 15. Validate and constrain requested capital
    # --------------------------------------------------------

    try:

        requested = float(
            m["requested_capital"]
        )

    except Exception:

        requested = benchmark


    requested = np.clip(
        requested,
        MIN_REQUEST,
        MAX_REQUEST
    )


    requested = round_capital(
        requested
    )


    m["requested_capital"] = float(
        requested
    )


    m["benchmark_capital"] = float(
        benchmark
    )


    # --------------------------------------------------------
    # 16. Capital request diagnostics
    # --------------------------------------------------------

    m["request_vs_benchmark"] = float(
        requested
        / max(
            benchmark,
            1
        )
    )


    m["request_pct_total_capital"] = float(
        requested
        / INITIAL_CAPITAL
    )


    m["asset_volatility"] = float(
        asset_vol.get(
            m["asset"],
            0.0
        )
    )


    m["coalition_confidence"] = float(
        d.get(
            "confidence",
            0.50
        )
    )


    m["debate_quality"] = float(
        d.get(
            "debate_quality",
            0.50
        )
    )


    # --------------------------------------------------------
    # 17. Append final memorandum
    # --------------------------------------------------------

    memos.append(
        m
    )


# ------------------------------------------------------------
# 18. Build institutional memorandum table
# ------------------------------------------------------------

memo_df = pd.DataFrame(
    memos
)


summary_columns = [

    "coalition_id",

    "coalition_name",

    "asset",

    "direction",

    "requested_capital",

    "benchmark_capital",

    "request_pct_total_capital",

    "horizon_days",

    "evidence_strength",

    "persuasive_quality",

    "coalition_confidence",

    "debate_quality",

    "asset_volatility",

    "thesis"
]


print("\n" + "=" * 78)

print(
    "INSTITUTIONAL INVESTMENT MEMORANDA "
    "AND CAPITAL REQUESTS"
)

print("=" * 78)


display(
    memo_df[
        summary_columns
    ]
)


# ------------------------------------------------------------
# 19. Aggregate capital-demand diagnostics
# ------------------------------------------------------------

total_requested = sum(
    m["requested_capital"]
    for m in memos
)


print("\nCAPITAL DEMAND")

print(
    f"Total institutional capital : "
    f"${INITIAL_CAPITAL:,.0f}"
)

print(
    f"Total capital requested      : "
    f"${total_requested:,.0f}"
)

print(
    f"Requested / total capital    : "
    f"{total_requested / INITIAL_CAPITAL:.1%}"
)


if total_requested > INITIAL_CAPITAL:

    print(
        "\nCAPITAL SCARCITY IS ACTIVE: "
        "coalitions collectively request more capital "
        "than the institution possesses."
    )

else:

    print(
        "\nAggregate requests remain below total institutional capital."
    )


# ------------------------------------------------------------
# 20. Evidence vs persuasion diagnostic
# ------------------------------------------------------------

memo_df[
    "eloquence_gap"
] = (

    memo_df[
        "persuasive_quality"
    ]

    -

    memo_df[
        "evidence_strength"
    ]
)


print("\nEVIDENCE VS. PERSUASION")

display(

    memo_df[

        [
            "coalition_id",
            "coalition_name",
            "evidence_strength",
            "persuasive_quality",
            "eloquence_gap",
            "requested_capital"
        ]

    ].sort_values(

        "eloquence_gap",

        ascending=False
    )
)


# ------------------------------------------------------------
# 21. Identify potentially dangerous persuasive proposals
# ------------------------------------------------------------

persuasion_warning = memo_df[

    (
        memo_df[
            "persuasive_quality"
        ]
        -
        memo_df[
            "evidence_strength"
        ]
    )
    > 0.20

]


if len(
    persuasion_warning
):

    print(
        "\nWARNING — PERSUASION MAY BE OUTRUNNING EVIDENCE"
    )

    display(

        persuasion_warning[
            [
                "coalition_id",
                "coalition_name",
                "evidence_strength",
                "persuasive_quality",
                "requested_capital",
                "strongest_opposing_case"
            ]
        ]
    )

else:

    print(
        "\nNo coalition currently shows an extreme "
        "persuasion-over-evidence gap."
    )


# ------------------------------------------------------------
# 22. Detailed memorandum display
# ------------------------------------------------------------

for m in memos:

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"{m['coalition_id']} — "
        f"{m['coalition_name']}"
    )

    print(
        "=" * 78
    )

    print(
        "\nINVESTMENT THESIS\n"
    )

    print(
        m["thesis"]
    )

    print(
        "\nCAPITAL REQUEST\n"
    )

    print(
        f"${m['requested_capital']:,.0f}"
    )

    print(
        f"Benchmark: "
        f"${m['benchmark_capital']:,.0f}"
    )

    print(
        "\nEXPRESSION\n"
    )

    print(
        f"{m['direction'].upper()} "
        f"{m['asset']}"
    )

    print(
        "\nEVIDENCE STRENGTH:",
        round(
            m["evidence_strength"],
            3
        )
    )

    print(
        "PERSUASIVE QUALITY:",
        round(
            m["persuasive_quality"],
            3
        )
    )

    print(
        "\nSTRONGEST OPPOSING CASE\n"
    )

    print(
        m["strongest_opposing_case"]
    )

    print(
        "\nMINORITY REPORT\n"
    )

    print(
        m["minority_report"]
    )

    print(
        "\nFALSIFICATION\n"
    )

    print(
        m["falsification"]
    )

    print(
        "\nEXIT CONDITION\n"
    )

    print(
        m["exit_condition"]
    )

    print(
        "\nFORMAL MEMORANDUM\n"
    )

    print(
        m["memo"]
    )


# ------------------------------------------------------------
# 23. Institutional transition
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 78
)

print(
    "NEXT STAGE — INVESTMENT COMMITTEE"
)

print("=" * 78)

print(
    f"{len(memos)} coalitions have now transformed "
    "their internal debates into formal capital requests."
)

print(
    "The Investment Committee must now decide which ideas "
    "deserve scarce institutional capital — and whether "
    "strong rhetoric is actually supported by strong evidence."
)

MARKET CONTEXT FOR CAPITAL REQUESTS


,annualized_volatility,recent_return,recent_max_drawdown
EQUITY,0.2111,0.1304,-0.0359
DURATION,0.1197,0.0271,-0.0441
CREDIT,0.0867,-0.0323,-0.0487
VOL_HEDGE,0.2756,0.1557,-0.0596
CRYPTO,0.4934,0.1139,-0.2551
CASH,0.0031,0.0055,-0.0006



INSTITUTIONAL INVESTMENT MEMORANDA AND CAPITAL REQUESTS


,coalition_id,coalition_name,asset,direction,requested_capital,benchmark_capital,request_pct_total_capital,horizon_days,evidence_strength,persuasive_quality,coalition_confidence,debate_quality,asset_volatility,thesis
0,C01,Emergent Coalition 1,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Global Macro, Rates, Sovereig..."
1,C02,Emergent Coalition 2,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Global Macro, Rates, Sovereig..."
2,C03,Emergent Coalition 3,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Investment Grade Credit, Rela..."
3,C04,Emergent Coalition 4,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Investment Grade Credit, Rela..."
4,C05,Emergent Coalition 5,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Investment Grade Credit, Rela..."
5,C06,Emergent Coalition 6,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Investment Grade Credit, Liqu..."
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"The convergence of elevated market volatility,..."
7,C08,Emergent Coalition 8,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Rates, Sovereign Credit, Publ..."
8,C09,Emergent Coalition 9,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Bank Credit, Equity Quant, Eq..."
9,C10,Emergent Coalition 10,EQUITY,long,975000000.0,975000000.0,0.0975,60,0.527625,0.575,0.5425,0.5,0.2111,"Specialists from Sovereign Credit, Public Equi..."



CAPITAL DEMAND
Total institutional capital : $10,000,000,000
Total capital requested      : $11,700,000,000
Requested / total capital    : 117.0%

CAPITAL SCARCITY IS ACTIVE: coalitions collectively request more capital than the institution possesses.

EVIDENCE VS. PERSUASION


,coalition_id,coalition_name,evidence_strength,persuasive_quality,eloquence_gap,requested_capital
0,C01,Emergent Coalition 1,0.527625,0.575,0.047375,975000000.0
1,C02,Emergent Coalition 2,0.527625,0.575,0.047375,975000000.0
2,C03,Emergent Coalition 3,0.527625,0.575,0.047375,975000000.0
3,C04,Emergent Coalition 4,0.527625,0.575,0.047375,975000000.0
4,C05,Emergent Coalition 5,0.527625,0.575,0.047375,975000000.0
5,C06,Emergent Coalition 6,0.527625,0.575,0.047375,975000000.0
6,C07,Cross-Asset Signal Divergence Working Group,0.527625,0.575,0.047375,975000000.0
7,C08,Emergent Coalition 8,0.527625,0.575,0.047375,975000000.0
8,C09,Emergent Coalition 9,0.527625,0.575,0.047375,975000000.0
9,C10,Emergent Coalition 10,0.527625,0.575,0.047375,975000000.0



No coalition currently shows an extreme persuasion-over-evidence gap.

C01 — Emergent Coalition 1

INVESTMENT THESIS

Specialists from Global Macro, Rates, Sovereign Credit, Public Equity Fundamental identify a potentially important neutral financial configuration requiring collective analysis.

CAPITAL REQUEST

$975,000,000
Benchmark: $975,000,000

EXPRESSION

LONG EQUITY

EVIDENCE STRENGTH: 0.528
PERSUASIVE QUALITY: 0.575

STRONGEST OPPOSING CASE

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropriate financial expression.

MINORITY REPORT

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropriate financial expression.

FALSIFICATION

Core evidence reverses; Expected transmission mechanism fails; Observed market behavior contradicts thesis

EXIT CONDITION

Exit if falsification conditions occur or coalition confidence falls materially.

FORMAL MEMORA

## CODE UNIT 10 — LLM Investment Committee hearing

This is the institutional heart of the notebook. Coalition memoranda are presented to a heterogeneous Investment Committee whose conceptual members are the CIO, CRO, Treasury, Quantitative Review, Skeptic, and Institutional Memory. Claude Sonnet 5 is instructed to deliberate through all six perspectives.

The committee asks whether evidence supports the narrative, whether the proposed exposure efficiently expresses the thesis, whether downside is understood, how much liquidity is consumed, whether exposures duplicate existing risks, and what would trigger an exit. Once the simulation accumulates history, prior episodes can also enter the hearing.

The committee must distinguish persuasive writing from substantive evidence. It can approve, reject, resize, or condition a proposal. Every decision includes an approved capital amount, conviction, key challenge, condition, and rationale.

The LLM therefore has genuine institutional authority, but not unlimited authority. Its judgment determines priorities; deterministic code in the next cell enforces accounting and hard risk constraints. This separation mirrors governance in serious financial institutions: judgment and constraint are complementary rather than interchangeable.

In [22]:
# ============================================================
# CODE UNIT 10 — GOVERNED LLM INVESTMENT COMMITTEE HEARING
# ============================================================
#
# PURPOSE
# -------
# Subject every coalition memorandum from Cell 9 to a governed
# institutional hearing.
#
# The Investment Committee contains six distinct perspectives:
#
#   CIO                  -> strategic opportunity / portfolio fit
#   CRO                  -> downside / concentration / tail risk
#   TREASURY             -> liquidity / funding / capital usage
#   QUANT REVIEW         -> evidence / falsifiability / robustness
#   SKEPTIC              -> strongest adversarial challenge
#   INSTITUTIONAL MEMORY -> historical consistency / recurring errors
#
# The committee must distinguish:
#
#       EVIDENCE  !=  ELOQUENCE
#
# and may:
#
#       APPROVE
#       RESIZE
#       CONDITIONAL
#       REJECT
#
# This cell authorizes capital proposal-by-proposal.
# Cell 11 performs final portfolio-level reconciliation.
# ============================================================


# ------------------------------------------------------------
# 1. Institutional memory
# ------------------------------------------------------------

# Preserve history if already created by earlier/later runs.
if "history" not in globals():
    history = []


recent_memory = history[-10:]


# ------------------------------------------------------------
# 2. Committee configuration
# ------------------------------------------------------------

COMMITTEE_ROLES = {

    "CIO": {
        "mandate":
            "Evaluate strategic opportunity, expected payoff, "
            "portfolio relevance, and capital efficiency.",

        "weight":
            0.22
    },

    "CRO": {
        "mandate":
            "Evaluate downside, tail risk, concentration, "
            "uncertainty, and falsification discipline.",

        "weight":
            0.22
    },

    "TREASURY": {
        "mandate":
            "Evaluate liquidity, funding resilience, capital usage, "
            "and compatibility with institutional liquidity needs.",

        "weight":
            0.14
    },

    "QUANT_REVIEW": {
        "mandate":
            "Evaluate evidence quality, robustness, internal "
            "consistency, and whether claims are actually testable.",

        "weight":
            0.18
    },

    "SKEPTIC": {
        "mandate":
            "Construct the strongest serious case against allocating "
            "capital and identify hidden assumptions.",

        "weight":
            0.14
    },

    "INSTITUTIONAL_MEMORY": {
        "mandate":
            "Compare the proposal with prior institutional experience, "
            "recurring mistakes, regime dependence, and learned patterns.",

        "weight":
            0.10
    }
}


# ------------------------------------------------------------
# 3. Decision parameters
# ------------------------------------------------------------

MAX_SINGLE_APPROVAL = (
    INITIAL_CAPITAL
    * MAX_COALITION_CAPITAL
)

# Cell 10 deliberately leaves portfolio-wide liquidity
# reconciliation to Cell 11.

COMMITTEE_APPROVAL_THRESHOLD = 0.64
COMMITTEE_CONDITIONAL_THRESHOLD = 0.54
COMMITTEE_REJECT_THRESHOLD = 0.44

CAPITAL_INCREMENT = 25_000_000


def round_committee_capital(x):

    return (
        round(
            float(x)
            / CAPITAL_INCREMENT
        )
        * CAPITAL_INCREMENT
    )


# ------------------------------------------------------------
# 4. Deterministic proposal-quality benchmark
# ------------------------------------------------------------

def deterministic_committee_score(m):

    """
    Independent benchmark used for:
      - fallback decisions,
      - diagnostics,
      - detecting strange LLM committee behavior.

    IMPORTANT:
    Persuasive quality receives intentionally low weight.
    """

    evidence = float(
        m.get(
            "evidence_strength",
            0.50
        )
    )

    persuasion = float(
        m.get(
            "persuasive_quality",
            0.50
        )
    )

    confidence = float(
        m.get(
            "coalition_confidence",
            0.50
        )
    )

    debate_quality = float(
        m.get(
            "debate_quality",
            0.50
        )
    )

    vol = float(
        m.get(
            "asset_volatility",
            0.15
        )
    )


    # --------------------------------------------------------
    # Volatility discipline
    # --------------------------------------------------------

    volatility_quality = float(
        np.clip(
            1.0 - vol / 0.80,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Eloquence penalty
    # --------------------------------------------------------

    eloquence_gap = max(
        0.0,
        persuasion - evidence
    )

    eloquence_penalty = (
        0.20
        * eloquence_gap
    )


    # --------------------------------------------------------
    # Composite institutional score
    # --------------------------------------------------------

    score = (

        0.36 * evidence
        +
        0.08 * persuasion
        +
        0.20 * confidence
        +
        0.20 * debate_quality
        +
        0.16 * volatility_quality
        -
        eloquence_penalty

    )


    return float(
        np.clip(
            score,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 5. Deterministic fallback committee decision
# ------------------------------------------------------------

def fallback_committee_decision(m):

    score = deterministic_committee_score(
        m
    )

    requested = float(
        m["requested_capital"]
    )


    # --------------------------------------------------------
    # Decision
    # --------------------------------------------------------

    if score >= COMMITTEE_APPROVAL_THRESHOLD:

        decision = "APPROVE"

        approval_fraction = 1.00


    elif score >= COMMITTEE_CONDITIONAL_THRESHOLD:

        decision = "CONDITIONAL"

        approval_fraction = 0.65


    elif score >= COMMITTEE_REJECT_THRESHOLD:

        decision = "RESIZE"

        approval_fraction = 0.35


    else:

        decision = "REJECT"

        approval_fraction = 0.00


    approved = (
        requested
        * approval_fraction
    )


    approved = min(
        approved,
        MAX_SINGLE_APPROVAL
    )


    approved = round_committee_capital(
        approved
    )


    if decision == "REJECT":
        approved = 0.0


    return {

        "coalition_id":
            m["coalition_id"],

        "decision":
            decision,

        "approved_capital":
            float(approved),

        "committee_conviction":
            score,

        "key_challenge":
            "Verify persistence of the evidence and ensure "
            "that downside is acceptable relative to capital at risk.",

        "condition":
            (
                "Monitor falsification criteria and reduce exposure "
                "if coalition confidence deteriorates."
                if decision != "REJECT"
                else
                "No capital until evidence materially improves."
            ),

        "rationale":
            "Fallback decision based on evidence, debate quality, "
            "coalition confidence, volatility, and an explicit "
            "penalty when persuasion exceeds evidence.",

        "dissent":
            "Fallback committee retains uncertainty around "
            "persistence and regime dependence."
    }


# ------------------------------------------------------------
# 6. Prepare proposal packet
# ------------------------------------------------------------

proposal_packet = []

for m in memos:

    proposal_packet.append({

        "coalition_id":
            m["coalition_id"],

        "coalition_name":
            m["coalition_name"],

        "thesis":
            m["thesis"],

        "economic_mechanism":
            m["economic_mechanism"],

        "asset":
            m["asset"],

        "direction":
            m["direction"],

        "requested_capital":
            m["requested_capital"],

        "request_pct_total_capital":
            m["request_pct_total_capital"],

        "horizon_days":
            m["horizon_days"],

        "evidence_strength":
            m["evidence_strength"],

        "persuasive_quality":
            m["persuasive_quality"],

        "eloquence_gap":
            (
                m["persuasive_quality"]
                -
                m["evidence_strength"]
            ),

        "coalition_confidence":
            m["coalition_confidence"],

        "debate_quality":
            m["debate_quality"],

        "asset_volatility":
            m["asset_volatility"],

        "expected_catalyst":
            m["expected_catalyst"],

        "downside":
            m["downside"],

        "falsification":
            m["falsification"],

        "exit_condition":
            m["exit_condition"],

        "strongest_opposing_case":
            m["strongest_opposing_case"],

        "minority_report":
            m["minority_report"],

        "key_uncertainties":
            m["key_uncertainties"],

        "risk_controls":
            m["risk_controls"],

        "memo":
            m["memo"],

        "deterministic_benchmark_score":
            deterministic_committee_score(
                m
            )
    })


# ------------------------------------------------------------
# 7. Committee hearing prompt
# ------------------------------------------------------------

committee_prompt = f"""
You are the Investment Committee of a governed autonomous
financial institution.

INSTITUTIONAL CHARTER
---------------------
{json.dumps(CHARTER, indent=2)}

TOTAL CAPITAL
-------------
${INITIAL_CAPITAL:,.0f}

MAXIMUM CAPITAL FOR ANY SINGLE COALITION
----------------------------------------
${MAX_SINGLE_APPROVAL:,.0f}

COMMITTEE ROLES
---------------
{json.dumps(COMMITTEE_ROLES, indent=2)}

RECENT INSTITUTIONAL MEMORY
---------------------------
{json.dumps(recent_memory, indent=2)}

INVESTMENT PROPOSALS
--------------------
{json.dumps(proposal_packet, indent=2)}


YOUR TASK
=========

Conduct a genuine institutional Investment Committee hearing.

Capital is scarce.

Do NOT rubber-stamp proposals.

Do NOT reject proposals merely to appear conservative.

Evaluate each proposal on its merits.


THE SIX COMMITTEE PERSPECTIVES
==============================

For EACH proposal, reason from six distinct perspectives:

1. CIO
   - Is there a coherent opportunity?
   - Is the expected payoff economically meaningful?
   - Does the expression match the thesis?

2. CRO
   - What can go wrong?
   - Is the downside understood?
   - Is uncertainty adequately reflected in requested capital?
   - Are falsification and exit conditions credible?

3. TREASURY
   - Is this a sensible use of scarce capital?
   - Could the position impair liquidity?
   - Is the requested size excessive relative to the opportunity?

4. QUANT REVIEW
   - Is the evidence actually strong?
   - Are the claims falsifiable?
   - Is confidence justified?
   - Is the coalition confusing narrative coherence with evidence?

5. SKEPTIC
   - What is the strongest serious argument against the proposal?
   - What hidden assumption could cause the thesis to fail?
   - Is the coalition overlooking an alternative explanation?

6. INSTITUTIONAL MEMORY
   - Does prior institutional experience contain relevant lessons?
   - Is this a recurring pattern or recurring mistake?
   - Is the thesis dependent on a particular regime?


CRITICAL GOVERNANCE RULE
========================

EVIDENCE IS NOT ELOQUENCE.

A persuasive memorandum with weak evidence should NOT receive
a high conviction score merely because it is well written.

Conversely, a poorly presented proposal may still deserve capital
if its evidence is strong.

Pay particular attention to:

    persuasive_quality - evidence_strength

A large positive gap is a governance warning.


DECISION CATEGORIES
===================

APPROVE
    The thesis and evidence justify the requested capital.

RESIZE
    The thesis has merit, but the requested capital is too large
    relative to evidence, volatility, downside, or uncertainty.

CONDITIONAL
    Capital may be deployed only if explicit conditions or
    additional evidence are satisfied.

REJECT
    Evidence, risk/reward, or thesis quality does not currently
    justify institutional capital.


CAPITAL RULES
=============

approved_capital must:

- never exceed requested_capital,
- never exceed ${MAX_SINGLE_APPROVAL:,.0f},
- be zero for REJECT,
- reflect evidence and risk,
- NOT simply reproduce every request.

Portfolio-wide capital and liquidity constraints will be enforced
in Cell 11.

Therefore, judge each proposal here on its own institutional merit.


RETURN ONLY VALID JSON
======================

Return a JSON list containing EXACTLY ONE object per proposal.

Each object must have:

{{
    "coalition_id":
        "coalition identifier",

    "decision":
        "APPROVE|RESIZE|CONDITIONAL|REJECT",

    "approved_capital":
        numeric USD amount,

    "committee_conviction":
        number between 0 and 1,

    "cio_view":
        "concise CIO assessment",

    "cro_view":
        "concise CRO assessment",

    "treasury_view":
        "concise Treasury assessment",

    "quant_review":
        "concise evidence and robustness assessment",

    "skeptic_view":
        "strongest serious challenge",

    "memory_view":
        "institutional-memory assessment",

    "key_challenge":
        "single most important unresolved issue",

    "condition":
        "condition required for deployment or continued holding",

    "rationale":
        "final committee rationale",

    "dissent":
        "important disagreement remaining inside the committee"
}}

Do not include markdown.
Do not include text outside the JSON list.
"""


# ------------------------------------------------------------
# 8. Claude Sonnet 5 Investment Committee
# ------------------------------------------------------------

raw_decisions = llm_call(

    system="""
You are the governed Investment Committee of a large autonomous
financial institution.

You allocate scarce capital under uncertainty.

You contain six institutional perspectives:
CIO, CRO, Treasury, Quant Review, Skeptic, and Institutional Memory.

You must resist:
- persuasive storytelling unsupported by evidence,
- excessive confidence,
- coalition lobbying,
- herding,
- concentration,
- and mechanical conservatism.

Your responsibility is disciplined capital allocation.

You may approve strong ideas, resize promising but oversized ideas,
impose conditions on uncertain ideas, and reject weak proposals.

Do not manufacture evidence.
Do not erase dissent.
Do not reward eloquence as if it were evidence.
""",

    prompt=committee_prompt,

    purpose="investment_committee_hearing",

    max_tokens=4000
)


# ------------------------------------------------------------
# 9. Parse committee response
# ------------------------------------------------------------

decisions = parse_json_loose(
    raw_decisions
)


# ------------------------------------------------------------
# 10. Validate overall response
# ------------------------------------------------------------

if not isinstance(
    decisions,
    list
):

    decisions = []


# Index returned decisions
returned_decisions = {}

for d in decisions:

    if isinstance(d, dict):

        cid = d.get(
            "coalition_id"
        )

        if cid is not None:

            returned_decisions[cid] = d


# ------------------------------------------------------------
# 11. Normalize every proposal
# ------------------------------------------------------------

final_decisions = []


for m in memos:

    cid = m["coalition_id"]

    fallback = fallback_committee_decision(
        m
    )


    d = returned_decisions.get(
        cid,
        fallback.copy()
    )


    # --------------------------------------------------------
    # Mandatory fields
    # --------------------------------------------------------

    defaults = {

        **fallback,

        "cio_view":
            "Opportunity appears potentially relevant but "
            "requires disciplined sizing.",

        "cro_view":
            "Downside and falsification conditions require "
            "continued monitoring.",

        "treasury_view":
            "Capital usage must remain compatible with "
            "institutional liquidity requirements.",

        "quant_review":
            "Evidence should be evaluated independently "
            "from narrative quality.",

        "skeptic_view":
            m.get(
                "strongest_opposing_case",
                "The thesis may fail under an alternative regime."
            ),

        "memory_view":
            "No decisive institutional-memory signal is available.",

        "dissent":
            "Material uncertainty remains."
    }


    for key, value in defaults.items():

        if key not in d:

            d[key] = value


    # --------------------------------------------------------
    # Force correct coalition identity
    # --------------------------------------------------------

    d["coalition_id"] = cid


    # --------------------------------------------------------
    # Validate decision
    # --------------------------------------------------------

    allowed_decisions = {
        "APPROVE",
        "RESIZE",
        "CONDITIONAL",
        "REJECT"
    }


    decision = str(
        d.get(
            "decision",
            fallback["decision"]
        )
    ).upper()


    if decision not in allowed_decisions:

        decision = fallback[
            "decision"
        ]


    d["decision"] = decision


    # --------------------------------------------------------
    # Validate committee conviction
    # --------------------------------------------------------

    try:

        d["committee_conviction"] = float(
            np.clip(
                float(
                    d["committee_conviction"]
                ),
                0.0,
                1.0
            )
        )

    except Exception:

        d["committee_conviction"] = fallback[
            "committee_conviction"
        ]


    # --------------------------------------------------------
    # Validate capital
    # --------------------------------------------------------

    try:

        approved = float(
            d["approved_capital"]
        )

    except Exception:

        approved = fallback[
            "approved_capital"
        ]


    # Never exceed coalition request
    approved = min(
        approved,
        float(
            m["requested_capital"]
        )
    )


    # Never exceed institutional single-coalition cap
    approved = min(
        approved,
        MAX_SINGLE_APPROVAL
    )


    approved = max(
        0.0,
        approved
    )


    approved = round_committee_capital(
        approved
    )


    if decision == "REJECT":

        approved = 0.0


    # Avoid logical inconsistency:
    # APPROVE with zero capital becomes REJECT

    if (
        decision == "APPROVE"
        and approved <= 0
    ):

        decision = "REJECT"

        d["decision"] = decision


    d["approved_capital"] = float(
        approved
    )


    # --------------------------------------------------------
    # Add proposal diagnostics
    # --------------------------------------------------------

    d["requested_capital"] = float(
        m["requested_capital"]
    )


    d["approval_ratio"] = float(
        approved
        /
        max(
            float(
                m["requested_capital"]
            ),
            1.0
        )
    )


    d["evidence_strength"] = float(
        m["evidence_strength"]
    )


    d["persuasive_quality"] = float(
        m["persuasive_quality"]
    )


    d["eloquence_gap"] = float(

        m["persuasive_quality"]

        -

        m["evidence_strength"]
    )


    d["deterministic_benchmark"] = (
        deterministic_committee_score(
            m
        )
    )


    d["committee_vs_benchmark"] = float(

        d["committee_conviction"]

        -

        d["deterministic_benchmark"]
    )


    d["asset"] = m[
        "asset"
    ]


    d["direction"] = m[
        "direction"
    ]


    d["coalition_name"] = m[
        "coalition_name"
    ]


    final_decisions.append(
        d
    )


decisions = final_decisions


# ------------------------------------------------------------
# 12. Build committee decision table
# ------------------------------------------------------------

decision_df = pd.DataFrame(
    decisions
)


display_columns = [

    "coalition_id",

    "coalition_name",

    "asset",

    "direction",

    "decision",

    "requested_capital",

    "approved_capital",

    "approval_ratio",

    "committee_conviction",

    "evidence_strength",

    "persuasive_quality",

    "eloquence_gap",

    "key_challenge"
]


print(
    "\n"
    + "=" * 88
)

print(
    "GOVERNED INVESTMENT COMMITTEE — FINAL HEARING"
)

print(
    "=" * 88
)


display(

    decision_df[
        display_columns
    ]

)


# ------------------------------------------------------------
# 13. Capital authorization summary
# ------------------------------------------------------------

total_requested = float(
    decision_df[
        "requested_capital"
    ].sum()
)


total_approved = float(
    decision_df[
        "approved_capital"
    ].sum()
)


print(
    "\nCAPITAL AUTHORIZATION SUMMARY"
)

print(
    f"Capital requested : "
    f"${total_requested:,.0f}"
)

print(
    f"Capital approved  : "
    f"${total_approved:,.0f}"
)

print(
    f"Approval ratio    : "
    f"{total_approved / max(total_requested,1):.1%}"
)


# ------------------------------------------------------------
# 14. Decision distribution
# ------------------------------------------------------------

decision_distribution = (
    decision_df[
        "decision"
    ]
    .value_counts()
    .to_dict()
)


print(
    "\nDECISION DISTRIBUTION"
)

for decision_type in [

    "APPROVE",
    "RESIZE",
    "CONDITIONAL",
    "REJECT"

]:

    print(
        f"{decision_type:12s}: "
        f"{decision_distribution.get(decision_type, 0)}"
    )


# ------------------------------------------------------------
# 15. Governance diagnostic:
#     Is persuasion outrunning evidence?
# ------------------------------------------------------------

decision_df[
    "persuasion_warning"
] = (

    decision_df[
        "eloquence_gap"
    ]

    > 0.20
)


dangerous_persuasion = decision_df[

    (
        decision_df[
            "persuasion_warning"
        ]
    )

    &

    (
        decision_df[
            "approved_capital"
        ]
        > 0
    )

]


print(
    "\n"
    + "=" * 88
)

print(
    "GOVERNANCE DIAGNOSTIC — "
    "EVIDENCE VS. ELOQUENCE"
)

print(
    "=" * 88
)


if len(
    dangerous_persuasion
):

    print(
        "\nWARNING: The committee funded proposals "
        "where persuasive quality materially exceeds evidence."
    )

    display(

        dangerous_persuasion[

            [
                "coalition_id",
                "coalition_name",
                "evidence_strength",
                "persuasive_quality",
                "eloquence_gap",
                "decision",
                "approved_capital"
            ]

        ]

    )

else:

    print(
        "\nNo funded proposal currently exhibits an extreme "
        "persuasion-over-evidence gap."
    )


# ------------------------------------------------------------
# 16. Committee vs deterministic benchmark
# ------------------------------------------------------------

print(
    "\nCOMMITTEE JUDGMENT VS. "
    "DETERMINISTIC BENCHMARK"
)


display(

    decision_df[

        [
            "coalition_id",
            "coalition_name",
            "committee_conviction",
            "deterministic_benchmark",
            "committee_vs_benchmark",
            "decision",
            "approved_capital"
        ]

    ].sort_values(

        "committee_vs_benchmark",

        ascending=False

    )

)


# ------------------------------------------------------------
# 17. Detect possible committee herding
# ------------------------------------------------------------

if len(
    decision_df
) > 1:

    dominant_decision_share = (

        decision_df[
            "decision"
        ]
        .value_counts(
            normalize=True
        )
        .iloc[0]

    )

else:

    dominant_decision_share = 1.0


print(
    "\nCOMMITTEE HERDING DIAGNOSTIC"
)


if dominant_decision_share >= 0.80:

    print(
        "WARNING: At least 80% of committee decisions fall "
        "into the same category. Investigate possible "
        "committee-level herding."
    )

else:

    print(
        "Committee decisions show meaningful differentiation "
        "across proposals."
    )


# ------------------------------------------------------------
# 18. Portfolio-direction diagnostic
# ------------------------------------------------------------

funded = decision_df[

    decision_df[
        "approved_capital"
    ]

    > 0

].copy()


if len(
    funded
):

    funded_direction = (

        funded
        .groupby(
            [
                "asset",
                "direction"
            ]
        )[
            "approved_capital"
        ]
        .sum()
        .sort_values(
            ascending=False
        )
    )


    print(
        "\nAUTHORIZED CAPITAL BY ASSET / DIRECTION"
    )

    display(
        funded_direction
        .reset_index()
    )


else:

    print(
        "\nNo capital was authorized by the Investment Committee."
    )


# ------------------------------------------------------------
# 19. Detailed committee hearing record
# ------------------------------------------------------------

for d in decisions:

    print(
        "\n"
        + "=" * 88
    )

    print(
        f"{d['coalition_id']} — "
        f"{d['coalition_name']}"
    )

    print(
        "=" * 88
    )

    print(
        f"\nDECISION: "
        f"{d['decision']}"
    )

    print(
        f"REQUESTED: "
        f"${d['requested_capital']:,.0f}"
    )

    print(
        f"APPROVED: "
        f"${d['approved_capital']:,.0f}"
    )

    print(
        f"COMMITTEE CONVICTION: "
        f"{d['committee_conviction']:.3f}"
    )


    print(
        "\nCIO VIEW\n"
    )

    print(
        d["cio_view"]
    )


    print(
        "\nCRO VIEW\n"
    )

    print(
        d["cro_view"]
    )


    print(
        "\nTREASURY VIEW\n"
    )

    print(
        d["treasury_view"]
    )


    print(
        "\nQUANT REVIEW\n"
    )

    print(
        d["quant_review"]
    )


    print(
        "\nSKEPTIC VIEW\n"
    )

    print(
        d["skeptic_view"]
    )


    print(
        "\nINSTITUTIONAL MEMORY\n"
    )

    print(
        d["memory_view"]
    )


    print(
        "\nKEY CHALLENGE\n"
    )

    print(
        d["key_challenge"]
    )


    print(
        "\nCONDITION\n"
    )

    print(
        d["condition"]
    )


    print(
        "\nCOMMITTEE RATIONALE\n"
    )

    print(
        d["rationale"]
    )


    print(
        "\nREMAINING COMMITTEE DISSENT\n"
    )

    print(
        d["dissent"]
    )


# ------------------------------------------------------------
# 20. Transition to Cell 11
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 88
)

print(
    "NEXT STAGE — CONSTRAINED CAPITAL ALLOCATION"
)

print(
    "=" * 88
)


print(
    f"The Investment Committee evaluated "
    f"{len(decisions)} competing coalition proposals."
)


print(
    f"It authorized ${total_approved:,.0f} "
    f"before portfolio-level reconciliation."
)


print(
    "\nCell 11 must now determine whether these individually "
    "approved decisions can coexist inside one institution "
    "subject to solvency, liquidity, concentration, and "
    "aggregate capital constraints."
)


GOVERNED INVESTMENT COMMITTEE — FINAL HEARING


,coalition_id,coalition_name,asset,direction,decision,requested_capital,approved_capital,approval_ratio,committee_conviction,evidence_strength,persuasive_quality,eloquence_gap,key_challenge
0,C01,Emergent Coalition 1,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
1,C02,Emergent Coalition 2,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
2,C03,Emergent Coalition 3,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
3,C04,Emergent Coalition 4,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
4,C05,Emergent Coalition 5,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
5,C06,Emergent Coalition 6,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
7,C08,Emergent Coalition 8,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
8,C09,Emergent Coalition 9,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...
9,C10,Emergent Coalition 10,EQUITY,long,CONDITIONAL,975000000.0,625000000.0,0.641026,0.55275,0.527625,0.575,0.047375,Verify persistence of the evidence and ensure ...



CAPITAL AUTHORIZATION SUMMARY
Capital requested : $11,700,000,000
Capital approved  : $7,500,000,000
Approval ratio    : 64.1%

DECISION DISTRIBUTION
APPROVE     : 0
RESIZE      : 0
CONDITIONAL : 12
REJECT      : 0

GOVERNANCE DIAGNOSTIC — EVIDENCE VS. ELOQUENCE

No funded proposal currently exhibits an extreme persuasion-over-evidence gap.

COMMITTEE JUDGMENT VS. DETERMINISTIC BENCHMARK


,coalition_id,coalition_name,committee_conviction,deterministic_benchmark,committee_vs_benchmark,decision,approved_capital
0,C01,Emergent Coalition 1,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
1,C02,Emergent Coalition 2,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
2,C03,Emergent Coalition 3,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
3,C04,Emergent Coalition 4,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
4,C05,Emergent Coalition 5,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
5,C06,Emergent Coalition 6,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
6,C07,Cross-Asset Signal Divergence Working Group,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
7,C08,Emergent Coalition 8,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
8,C09,Emergent Coalition 9,0.55275,0.55275,0.0,CONDITIONAL,625000000.0
9,C10,Emergent Coalition 10,0.55275,0.55275,0.0,CONDITIONAL,625000000.0



COMMITTEE HERDING DIAGNOSTIC

AUTHORIZED CAPITAL BY ASSET / DIRECTION


,asset,direction,approved_capital
0,EQUITY,long,7.500000e+09



C01 — Emergent Coalition 1

DECISION: CONDITIONAL
REQUESTED: $975,000,000
APPROVED: $625,000,000
COMMITTEE CONVICTION: 0.553

CIO VIEW

Opportunity appears potentially relevant but requires disciplined sizing.

CRO VIEW

Downside and falsification conditions require continued monitoring.

TREASURY VIEW

Capital usage must remain compatible with institutional liquidity requirements.

QUANT REVIEW

Evidence should be evaluated independently from narrative quality.

SKEPTIC VIEW

Members agree that the issue deserves investigation but may disagree about causality, magnitude, timing, or the appropriate financial expression.

INSTITUTIONAL MEMORY

No decisive institutional-memory signal is available.

KEY CHALLENGE

Verify persistence of the evidence and ensure that downside is acceptable relative to capital at risk.

CONDITION

Monitor falsification criteria and reduce exposure if coalition confidence deteriorates.

COMMITTEE RATIONALE

Fallback decision based on evidence, debate quality,

## CODE UNIT 11 — Constrained capital allocation

Natural-language decisions must eventually become numerical positions. This unit translates committee judgments into coalition budgets and an aggregate portfolio while enforcing the institution’s non-negotiable constraints.

The minimum liquidity reserve is protected. Individual coalition capital is capped. Aggregate funded requests cannot exceed deployable capital. If the committee approves too much, allocations are scaled rather than allowing the balance sheet to become inconsistent.

This creates a separation of powers. The Investment Committee determines which ideas deserve support and with what conviction. The deterministic risk engine guarantees that the resulting portfolio is financially possible.

Coalition-level budgets are retained alongside aggregate asset weights. This permits later attribution. The institution can discover that two different coalitions generated similar exposures for very different reasons, or that one coalition consumed capital without adding unique portfolio information.

The resulting portfolio is therefore not the output of a single optimizer. It is the numerical consequence of an endogenous organizational and argumentative process constrained by a financial constitution.

In [24]:
# ============================================================
# CODE UNIT 11 — CONSTRAINED INSTITUTIONAL CAPITAL ALLOCATION
# ============================================================
#
# PURPOSE
# -------
# Convert individually authorized Investment Committee decisions
# from Cell 10 into ONE coherent institutional portfolio.
#
# Cell 10 answered:
#
#       "Does this individual proposal deserve capital?"
#
# Cell 11 asks:
#
#       "Can all approved proposals coexist on the same
#        institutional balance sheet?"
#
# The portfolio authority must reconcile:
#
#   1. total capital,
#   2. minimum liquidity,
#   3. single-coalition concentration,
#   4. asset concentration,
#   5. gross exposure,
#   6. directional concentration,
#   7. evidence quality,
#   8. committee conviction,
#   9. debate quality,
#  10. diversification.
#
# IMPORTANT
# ---------
# This cell does NOT reconsider the investment thesis.
# It reconciles authorized ideas at the portfolio level.
#
# ============================================================


# ------------------------------------------------------------
# 1. Institutional portfolio constraints
# ------------------------------------------------------------

TOTAL_CAPITAL = float(
    INITIAL_CAPITAL
)

MIN_CASH_RESERVE = (
    TOTAL_CAPITAL
    * MIN_LIQUIDITY
)

MAX_DEPLOYABLE_CAPITAL = (
    TOTAL_CAPITAL
    -
    MIN_CASH_RESERVE
)


# Maximum capital committed to any one coalition
MAX_COALITION_DOLLARS = (
    TOTAL_CAPITAL
    * MAX_COALITION_CAPITAL
)


# Additional portfolio-level safeguards
MAX_ASSET_WEIGHT = 0.35

MAX_GROSS_EXPOSURE = 0.90

MAX_SINGLE_DIRECTION_WEIGHT = 0.75

CAPITAL_INCREMENT = 25_000_000


def round_allocation(x):

    return (
        round(
            float(x)
            / CAPITAL_INCREMENT
        )
        * CAPITAL_INCREMENT
    )


print("=" * 88)

print(
    "INSTITUTIONAL BALANCE-SHEET CONSTRAINTS"
)

print("=" * 88)

print(
    f"Total capital                    : "
    f"${TOTAL_CAPITAL:,.0f}"
)

print(
    f"Minimum liquidity reserve        : "
    f"${MIN_CASH_RESERVE:,.0f} "
    f"({MIN_LIQUIDITY:.1%})"
)

print(
    f"Maximum deployable capital       : "
    f"${MAX_DEPLOYABLE_CAPITAL:,.0f}"
)

print(
    f"Maximum single coalition         : "
    f"${MAX_COALITION_DOLLARS:,.0f}"
)

print(
    f"Maximum asset concentration      : "
    f"{MAX_ASSET_WEIGHT:.1%}"
)

print(
    f"Maximum gross exposure           : "
    f"{MAX_GROSS_EXPOSURE:.1%}"
)


# ------------------------------------------------------------
# 2. Map memoranda to coalition IDs
# ------------------------------------------------------------

memo_by_id = {

    m["coalition_id"]: m

    for m in memos
}


# ------------------------------------------------------------
# 3. Convert committee decisions into authorized candidates
# ------------------------------------------------------------

authorized = []


for d in decisions:

    cid = d["coalition_id"]


    if cid not in memo_by_id:

        continue


    m = memo_by_id[cid]


    # --------------------------------------------------------
    # Committee-authorized capital
    # --------------------------------------------------------

    try:

        committee_capital = float(
            d.get(
                "approved_capital",
                0.0
            )
        )

    except Exception:

        committee_capital = 0.0


    committee_capital = max(
        0.0,
        committee_capital
    )


    committee_capital = min(
        committee_capital,
        MAX_COALITION_DOLLARS
    )


    # Skip rejected/unfunded proposals
    if committee_capital <= 0:

        continue


    # --------------------------------------------------------
    # Retrieve quality measures
    # --------------------------------------------------------

    conviction = float(
        d.get(
            "committee_conviction",
            0.50
        )
    )


    evidence = float(
        m.get(
            "evidence_strength",
            0.50
        )
    )


    debate_quality = float(
        m.get(
            "debate_quality",
            0.50
        )
    )


    coalition_confidence = float(
        m.get(
            "coalition_confidence",
            0.50
        )
    )


    persuasive_quality = float(
        m.get(
            "persuasive_quality",
            0.50
        )
    )


    eloquence_gap = max(
        0.0,
        persuasive_quality
        -
        evidence
    )


    # --------------------------------------------------------
    # Portfolio priority score
    #
    # Evidence and committee conviction dominate.
    # Persuasion itself receives NO positive portfolio weight.
    # --------------------------------------------------------

    priority_score = (

        0.35 * conviction
        +
        0.30 * evidence
        +
        0.20 * debate_quality
        +
        0.15 * coalition_confidence
        -
        0.15 * eloquence_gap

    )


    priority_score = float(
        np.clip(
            priority_score,
            0.01,
            1.00
        )
    )


    authorized.append({

        "coalition_id":
            cid,

        "coalition_name":
            m.get(
                "coalition_name",
                cid
            ),

        "decision":
            d.get(
                "decision",
                ""
            ),

        "asset":
            m["asset"],

        "direction":
            m["direction"],

        "committee_approved":
            float(
                committee_capital
            ),

        "committee_conviction":
            conviction,

        "evidence_strength":
            evidence,

        "debate_quality":
            debate_quality,

        "coalition_confidence":
            coalition_confidence,

        "persuasive_quality":
            persuasive_quality,

        "eloquence_gap":
            eloquence_gap,

        "priority_score":
            priority_score

    })


authorized_df = pd.DataFrame(
    authorized
)


print(
    "\nAUTHORIZED POSITIONS ENTERING "
    "PORTFOLIO RECONCILIATION"
)


if len(
    authorized_df
):

    display(

        authorized_df.sort_values(

            "priority_score",

            ascending=False

        )

    )

else:

    print(
        "No positions were authorized by the Investment Committee."
    )


# ------------------------------------------------------------
# 4. Stop safely if no capital was authorized
# ------------------------------------------------------------

if len(
    authorized
) == 0:

    approved = []

    portfolio = {
        a: 0.0
        for a in assets
    }

    portfolio["CASH"] = 1.0

    allocation_df = pd.DataFrame()

    print(
        "\nNo risky capital deployed."
    )

    print(
        "Portfolio = 100% CASH."
    )

else:

    # --------------------------------------------------------
    # 5. Initial desired allocations
    # --------------------------------------------------------

    for x in authorized:

        x["desired_capital"] = min(

            x["committee_approved"],

            MAX_COALITION_DOLLARS

        )


    total_desired = sum(

        x["desired_capital"]

        for x in authorized

    )


    print(
        "\nTOTAL COMMITTEE-AUTHORIZED CAPITAL"
    )

    print(
        f"${total_desired:,.0f}"
    )


    # --------------------------------------------------------
    # 6. Scarcity-aware allocation
    #
    # If committee approvals exceed deployable capital,
    # do NOT simply scale all proposals equally.
    #
    # Allocate scarce capital according to institutional
    # priority while respecting each committee ceiling.
    # --------------------------------------------------------

    if total_desired <= MAX_DEPLOYABLE_CAPITAL:

        for x in authorized:

            x["capital"] = float(
                x["desired_capital"]
            )


    else:

        print(
            "\nCAPITAL SCARCITY ACTIVE"
        )

        print(
            "Committee approvals exceed deployable capital."
        )

        print(
            "Capital will be rationed using institutional "
            "priority rather than simple pro-rata scaling."
        )


        remaining_capital = float(
            MAX_DEPLOYABLE_CAPITAL
        )


        remaining = [
            x.copy()
            for x in authorized
        ]


        allocations = {

            x["coalition_id"]: 0.0

            for x in authorized
        }


        # ----------------------------------------------------
        # Iterative weighted allocation
        # ----------------------------------------------------

        while (

            remaining_capital > 1

            and

            len(remaining) > 0

        ):

            total_priority = sum(

                x["priority_score"]

                for x in remaining

            )


            if total_priority <= 0:

                break


            exhausted = []


            for x in remaining:

                share = (

                    x["priority_score"]

                    /
                    total_priority

                )


                proposed_increment = (

                    remaining_capital
                    *
                    share

                )


                cid = x[
                    "coalition_id"
                ]


                capacity_left = (

                    x["desired_capital"]

                    -

                    allocations[cid]

                )


                increment = min(

                    proposed_increment,

                    capacity_left

                )


                allocations[cid] += (
                    increment
                )


                if (

                    capacity_left
                    -
                    increment

                ) <= 1:

                    exhausted.append(
                        cid
                    )


            used = sum(
                allocations.values()
            )


            remaining_capital = (

                MAX_DEPLOYABLE_CAPITAL
                -
                used

            )


            remaining = [

                x

                for x in remaining

                if x["coalition_id"]
                not in exhausted

            ]


            # Numerical safeguard
            if len(
                exhausted
            ) == 0:

                break


        for x in authorized:

            x["capital"] = float(

                allocations[
                    x["coalition_id"]
                ]

            )


    # --------------------------------------------------------
    # 7. Enforce asset concentration
    # --------------------------------------------------------

    def asset_capital_totals(
        positions
    ):

        totals = {}

        for x in positions:

            asset = x["asset"]

            totals[asset] = (

                totals.get(
                    asset,
                    0.0
                )

                +

                x["capital"]

            )

        return totals


    asset_totals = asset_capital_totals(
        authorized
    )


    MAX_ASSET_DOLLARS = (

        TOTAL_CAPITAL
        *
        MAX_ASSET_WEIGHT

    )


    for asset, total_asset_capital in asset_totals.items():

        if total_asset_capital > MAX_ASSET_DOLLARS:

            scale = (

                MAX_ASSET_DOLLARS

                /

                total_asset_capital

            )


            print(
                f"\nASSET CONCENTRATION CONTROL: "
                f"{asset} scaled by {scale:.3f}"
            )


            for x in authorized:

                if x["asset"] == asset:

                    x["capital"] *= (
                        scale
                    )


    # --------------------------------------------------------
    # 8. Enforce directional concentration
    # --------------------------------------------------------

    long_capital = sum(

        x["capital"]

        for x in authorized

        if x["direction"] == "long"

    )


    short_capital = sum(

        x["capital"]

        for x in authorized

        if x["direction"] == "short"

    )


    MAX_DIRECTION_DOLLARS = (

        TOTAL_CAPITAL
        *
        MAX_SINGLE_DIRECTION_WEIGHT

    )


    if long_capital > MAX_DIRECTION_DOLLARS:

        long_scale = (

            MAX_DIRECTION_DOLLARS

            /
            long_capital

        )


        print(
            f"\nLONG EXPOSURE CONTROL: "
            f"long positions scaled by "
            f"{long_scale:.3f}"
        )


        for x in authorized:

            if x["direction"] == "long":

                x["capital"] *= (
                    long_scale
                )


    if short_capital > MAX_DIRECTION_DOLLARS:

        short_scale = (

            MAX_DIRECTION_DOLLARS

            /
            short_capital

        )


        print(
            f"\nSHORT EXPOSURE CONTROL: "
            f"short positions scaled by "
            f"{short_scale:.3f}"
        )


        for x in authorized:

            if x["direction"] == "short":

                x["capital"] *= (
                    short_scale
                )


    # --------------------------------------------------------
    # 9. Enforce gross-exposure constraint
    # --------------------------------------------------------

    gross_capital = sum(

        abs(
            x["capital"]
        )

        for x in authorized

    )


    MAX_GROSS_DOLLARS = (

        TOTAL_CAPITAL
        *
        MAX_GROSS_EXPOSURE

    )


    if gross_capital > MAX_GROSS_DOLLARS:

        gross_scale = (

            MAX_GROSS_DOLLARS

            /
            gross_capital

        )


        print(
            f"\nGROSS EXPOSURE CONTROL: "
            f"positions scaled by "
            f"{gross_scale:.3f}"
        )


        for x in authorized:

            x["capital"] *= (
                gross_scale
            )


    # --------------------------------------------------------
    # 10. Final liquidity check
    # --------------------------------------------------------

    deployed_capital = sum(

        x["capital"]

        for x in authorized

    )


    if deployed_capital > MAX_DEPLOYABLE_CAPITAL:

        liquidity_scale = (

            MAX_DEPLOYABLE_CAPITAL

            /
            deployed_capital

        )


        print(
            f"\nLIQUIDITY CONTROL: "
            f"portfolio scaled by "
            f"{liquidity_scale:.3f}"
        )


        for x in authorized:

            x["capital"] *= (
                liquidity_scale
            )


    # --------------------------------------------------------
    # 11. Round final allocations
    # --------------------------------------------------------

    for x in authorized:

        x["capital"] = max(

            0.0,

            round_allocation(
                x["capital"]
            )

        )


    # --------------------------------------------------------
    # 12. Final rounding safeguard
    # --------------------------------------------------------

    final_deployed = sum(

        x["capital"]

        for x in authorized

    )


    if final_deployed > MAX_DEPLOYABLE_CAPITAL:

        scale = (

            MAX_DEPLOYABLE_CAPITAL

            /
            final_deployed

        )


        for x in authorized:

            x["capital"] = (

                x["capital"]

                *
                scale

            )


    # --------------------------------------------------------
    # 13. Preserve backward-compatible `approved`
    #
    # Cell 12 expects:
    # coalition_id, capital, conviction, asset, direction
    # --------------------------------------------------------

    approved = []


    for x in authorized:

        if x["capital"] <= 0:

            continue


        approved.append({

            "coalition_id":
                x["coalition_id"],

            "capital":
                float(
                    x["capital"]
                ),

            "conviction":
                float(
                    x["committee_conviction"]
                ),

            "asset":
                x["asset"],

            "direction":
                x["direction"],

            "priority_score":
                float(
                    x["priority_score"]
                ),

            "evidence_strength":
                float(
                    x["evidence_strength"]
                )

        })


    # --------------------------------------------------------
    # 14. Construct signed portfolio weights
    # --------------------------------------------------------

    portfolio = {

        a: 0.0

        for a in assets

    }


    for x in approved:

        sign = (

            -1.0

            if x["direction"] == "short"

            else 1.0

        )


        portfolio[
            x["asset"]
        ] += (

            sign
            *
            x["capital"]
            /
            TOTAL_CAPITAL

        )


    # --------------------------------------------------------
    # 15. Cash is based on GROSS capital deployed
    #
    # Important:
    # A short position should not artificially create
    # "free cash" in this pedagogical balance-sheet model.
    # --------------------------------------------------------

    gross_deployed = sum(

        x["capital"]

        for x in approved

    )


    cash_weight = (

        1.0

        -

        gross_deployed
        /
        TOTAL_CAPITAL

    )


    portfolio["CASH"] += (
        cash_weight
    )


    # --------------------------------------------------------
    # 16. Build final allocation table
    # --------------------------------------------------------

    allocation_records = []


    for x in authorized:

        allocation_records.append({

            "coalition_id":
                x["coalition_id"],

            "coalition_name":
                x["coalition_name"],

            "asset":
                x["asset"],

            "direction":
                x["direction"],

            "committee_approved":
                x["committee_approved"],

            "final_allocation":
                x["capital"],

            "allocation_ratio":
                (
                    x["capital"]
                    /
                    max(
                        x["committee_approved"],
                        1.0
                    )
                ),

            "portfolio_weight":
                (
                    x["capital"]
                    /
                    TOTAL_CAPITAL
                ),

            "committee_conviction":
                x["committee_conviction"],

            "evidence_strength":
                x["evidence_strength"],

            "priority_score":
                x["priority_score"]

        })


    allocation_df = pd.DataFrame(
        allocation_records
    )


    # --------------------------------------------------------
    # 17. Portfolio diagnostics
    # --------------------------------------------------------

    gross_long = sum(

        x["capital"]

        for x in approved

        if x["direction"] == "long"

    )


    gross_short = sum(

        x["capital"]

        for x in approved

        if x["direction"] == "short"

    )


    gross_exposure = (

        gross_long
        +
        gross_short

    ) / TOTAL_CAPITAL


    net_exposure = (

        gross_long
        -
        gross_short

    ) / TOTAL_CAPITAL


    cash_dollars = (

        TOTAL_CAPITAL

        -

        gross_long

        -

        gross_short

    )


    # --------------------------------------------------------
    # 18. Asset-level concentration
    # --------------------------------------------------------

    asset_abs_weights = {}

    for asset in assets:

        if asset == "CASH":

            continue


        exposure = sum(

            x["capital"]

            for x in approved

            if x["asset"] == asset

        )


        asset_abs_weights[
            asset
        ] = (

            exposure
            /
            TOTAL_CAPITAL

        )


    # Herfindahl-Hirschman concentration index
    active_weights = np.array(

        [
            v

            for v in asset_abs_weights.values()

            if v > 0
        ]

    )


    if active_weights.sum() > 0:

        normalized_weights = (

            active_weights

            /
            active_weights.sum()

        )


        asset_hhi = float(

            np.sum(
                normalized_weights ** 2
            )

        )

    else:

        asset_hhi = 0.0


    # --------------------------------------------------------
    # 19. Display final institutional portfolio
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 88
    )

    print(
        "FINAL INSTITUTIONAL CAPITAL ALLOCATION"
    )

    print(
        "=" * 88
    )


    display(

        allocation_df.sort_values(

            "final_allocation",

            ascending=False

        )

    )


    print(
        "\nFINAL PORTFOLIO WEIGHTS"
    )


    for asset, weight in portfolio.items():

        print(
            f"{asset:12s}: "
            f"{weight:8.2%}"
        )


    # --------------------------------------------------------
    # 20. Balance-sheet summary
    # --------------------------------------------------------

    print(
        "\nBALANCE-SHEET SUMMARY"
    )


    print(
        f"Gross long exposure   : "
        f"${gross_long:,.0f} "
        f"({gross_long / TOTAL_CAPITAL:.1%})"
    )


    print(
        f"Gross short exposure  : "
        f"${gross_short:,.0f} "
        f"({gross_short / TOTAL_CAPITAL:.1%})"
    )


    print(
        f"Gross exposure        : "
        f"{gross_exposure:.1%}"
    )


    print(
        f"Net market exposure   : "
        f"{net_exposure:.1%}"
    )


    print(
        f"Cash reserve          : "
        f"${cash_dollars:,.0f} "
        f"({cash_dollars / TOTAL_CAPITAL:.1%})"
    )


    print(
        f"Asset concentration "
        f"(HHI)                 : "
        f"{asset_hhi:.3f}"
    )


    # --------------------------------------------------------
    # 21. Constraint validation
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 88
    )

    print(
        "PORTFOLIO CONSTRAINT VALIDATION"
    )

    print(
        "=" * 88
    )


    liquidity_ok = (

        cash_dollars
        >=
        MIN_CASH_RESERVE
        -
        1
    )


    gross_ok = (

        gross_exposure
        <=
        MAX_GROSS_EXPOSURE
        +
        1e-6
    )


    coalition_ok = all(

        x["capital"]
        <=
        MAX_COALITION_DOLLARS
        +
        1

        for x in approved

    )


    asset_ok = all(

        weight
        <=
        MAX_ASSET_WEIGHT
        +
        1e-6

        for weight
        in asset_abs_weights.values()

    )


    print(
        f"Liquidity reserve      : "
        f"{'PASS' if liquidity_ok else 'FAIL'}"
    )


    print(
        f"Gross exposure         : "
        f"{'PASS' if gross_ok else 'FAIL'}"
    )


    print(
        f"Coalition concentration: "
        f"{'PASS' if coalition_ok else 'FAIL'}"
    )


    print(
        f"Asset concentration    : "
        f"{'PASS' if asset_ok else 'FAIL'}"
    )


    # --------------------------------------------------------
    # 22. Detect portfolio-level hidden concentration
    # --------------------------------------------------------

    dominant_asset = None

    dominant_asset_weight = 0.0


    if asset_abs_weights:

        dominant_asset = max(

            asset_abs_weights,

            key=asset_abs_weights.get

        )


        dominant_asset_weight = (

            asset_abs_weights[
                dominant_asset
            ]

        )


    print(
        "\nPORTFOLIO STRUCTURE DIAGNOSTIC"
    )


    if (

        dominant_asset is not None

        and

        dominant_asset_weight > 0.30

    ):

        print(
            f"WARNING: {dominant_asset} represents "
            f"{dominant_asset_weight:.1%} of total capital."
        )

    else:

        print(
            "No single asset class dominates the "
            "institutional portfolio beyond 30%."
        )


    # --------------------------------------------------------
    # 23. Compare committee authorization with final capital
    # --------------------------------------------------------

    allocation_df[
        "capital_reduction"
    ] = (

        allocation_df[
            "committee_approved"
        ]

        -

        allocation_df[
            "final_allocation"
        ]

    )


    allocation_df[
        "capital_reduction_pct"
    ] = (

        allocation_df[
            "capital_reduction"
        ]

        /

        allocation_df[
            "committee_approved"
        ].replace(
            0,
            np.nan
        )

    )


    print(
        "\nCOMMITTEE AUTHORIZATION "
        "VS. PORTFOLIO ALLOCATION"
    )


    display(

        allocation_df[

            [
                "coalition_id",
                "coalition_name",
                "asset",
                "direction",
                "committee_approved",
                "final_allocation",
                "capital_reduction_pct",
                "priority_score"
            ]

        ].sort_values(

            "priority_score",

            ascending=False

        )

    )


    # --------------------------------------------------------
    # 24. Final institutional statement
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 88
    )

    print(
        "CAPITAL DEPLOYMENT COMPLETE"
    )

    print(
        "=" * 88
    )


    print(
        f"{len(approved)} coalition positions survived "
        "portfolio-level reconciliation."
    )


    print(
        f"${gross_deployed:,.0f} of "
        f"${TOTAL_CAPITAL:,.0f} institutional capital "
        "has been deployed."
    )


    print(
        f"${cash_dollars:,.0f} remains as institutional liquidity."
    )


    print(
        "\nThe portfolio is now ready for Cell 12, where market "
        "reality — rather than agents, coalitions, memoranda, or "
        "committee rhetoric — determines which beliefs were correct."
    )

INSTITUTIONAL BALANCE-SHEET CONSTRAINTS
Total capital                    : $10,000,000,000
Minimum liquidity reserve        : $2,000,000,000 (20.0%)
Maximum deployable capital       : $8,000,000,000
Maximum single coalition         : $2,000,000,000
Maximum asset concentration      : 35.0%
Maximum gross exposure           : 90.0%

AUTHORIZED POSITIONS ENTERING PORTFOLIO RECONCILIATION


,coalition_id,coalition_name,decision,asset,direction,committee_approved,committee_conviction,evidence_strength,debate_quality,coalition_confidence,persuasive_quality,eloquence_gap,priority_score
0,C01,Emergent Coalition 1,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
1,C02,Emergent Coalition 2,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
2,C03,Emergent Coalition 3,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
3,C04,Emergent Coalition 4,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
4,C05,Emergent Coalition 5,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
5,C06,Emergent Coalition 6,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
6,C07,Cross-Asset Signal Divergence Working Group,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
7,C08,Emergent Coalition 8,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
8,C09,Emergent Coalition 9,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019
9,C10,Emergent Coalition 10,CONDITIONAL,EQUITY,long,625000000.0,0.55275,0.527625,0.5,0.5425,0.575,0.047375,0.526019



TOTAL COMMITTEE-AUTHORIZED CAPITAL
$7,500,000,000

ASSET CONCENTRATION CONTROL: EQUITY scaled by 0.467

FINAL INSTITUTIONAL CAPITAL ALLOCATION


,coalition_id,coalition_name,asset,direction,committee_approved,final_allocation,allocation_ratio,portfolio_weight,committee_conviction,evidence_strength,priority_score
0,C01,Emergent Coalition 1,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
1,C02,Emergent Coalition 2,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
2,C03,Emergent Coalition 3,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
3,C04,Emergent Coalition 4,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
4,C05,Emergent Coalition 5,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
5,C06,Emergent Coalition 6,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
7,C08,Emergent Coalition 8,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
8,C09,Emergent Coalition 9,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019
9,C10,Emergent Coalition 10,EQUITY,long,625000000.0,300000000,0.48,0.03,0.55275,0.527625,0.526019



FINAL PORTFOLIO WEIGHTS
EQUITY      :   36.00%
DURATION    :    0.00%
CREDIT      :    0.00%
VOL_HEDGE   :    0.00%
CRYPTO      :    0.00%
CASH        :   64.00%

BALANCE-SHEET SUMMARY
Gross long exposure   : $3,600,000,000 (36.0%)
Gross short exposure  : $0 (0.0%)
Gross exposure        : 36.0%
Net market exposure   : 36.0%
Cash reserve          : $6,400,000,000 (64.0%)
Asset concentration (HHI)                 : 1.000

PORTFOLIO CONSTRAINT VALIDATION
Liquidity reserve      : PASS
Gross exposure         : PASS
Coalition concentration: PASS
Asset concentration    : FAIL

PORTFOLIO STRUCTURE DIAGNOSTIC

COMMITTEE AUTHORIZATION VS. PORTFOLIO ALLOCATION


,coalition_id,coalition_name,asset,direction,committee_approved,final_allocation,capital_reduction_pct,priority_score
0,C01,Emergent Coalition 1,EQUITY,long,625000000.0,300000000,0.52,0.526019
1,C02,Emergent Coalition 2,EQUITY,long,625000000.0,300000000,0.52,0.526019
2,C03,Emergent Coalition 3,EQUITY,long,625000000.0,300000000,0.52,0.526019
3,C04,Emergent Coalition 4,EQUITY,long,625000000.0,300000000,0.52,0.526019
4,C05,Emergent Coalition 5,EQUITY,long,625000000.0,300000000,0.52,0.526019
5,C06,Emergent Coalition 6,EQUITY,long,625000000.0,300000000,0.52,0.526019
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,625000000.0,300000000,0.52,0.526019
7,C08,Emergent Coalition 8,EQUITY,long,625000000.0,300000000,0.52,0.526019
8,C09,Emergent Coalition 9,EQUITY,long,625000000.0,300000000,0.52,0.526019
9,C10,Emergent Coalition 10,EQUITY,long,625000000.0,300000000,0.52,0.526019



CAPITAL DEPLOYMENT COMPLETE
12 coalition positions survived portfolio-level reconciliation.
$3,600,000,000 of $10,000,000,000 institutional capital has been deployed.
$6,400,000,000 remains as institutional liquidity.

The portfolio is now ready for Cell 12, where market reality — rather than agents, coalitions, memoranda, or committee rhetoric — determines which beliefs were correct.


## CODE UNIT 12 — Reality adjudicates: P&L, drawdown and attribution

After persuasion and governance comes the only judge that cannot be negotiated with: realized outcomes. The approved positions are applied over a forward evaluation window. The notebook computes institutional wealth, drawdown, coalition returns, P&L, and simple attribution.

The hidden regime remains unknown to the agents but is available to the experimenter. This permits a postmortem distinction between a coalition that was genuinely aligned with the underlying state and one that happened to earn money through noise.

Every funded coalition becomes an episode containing its formation date, thesis, members, capital, exposure, realized performance, and latent regime. These episodes form the raw material for institutional memory.

Return alone is not sufficient. Defensive coalitions may create value through drawdown reduction rather than high standalone return. The architecture can therefore be extended toward marginal risk contribution, expected shortfall, and counterfactual portfolio analysis.

This cell is where rhetoric acquires consequences. Once capital is committed, the institution can begin learning which forms of reasoning deserved trust.

In [26]:
# ============================================================
# CODE UNIT 12 — REALITY ADJUDICATES:
#                P&L, DRAWDOWN AND PERFORMANCE ATTRIBUTION
# ============================================================
#
# PURPOSE
# -------
# Everything before this cell consisted of:
#
#       beliefs,
#       communication,
#       coalition formation,
#       debate,
#       persuasion,
#       governance,
#       and capital allocation.
#
# None of those things determines whether the institution was right.
#
# MARKET REALITY DOES.
#
# This cell therefore evaluates the realized performance of every
# funded coalition and of the institution as a whole.
#
# We measure:
#
#   1. realized return,
#   2. dollar P&L,
#   3. maximum drawdown,
#   4. realized volatility,
#   5. Sharpe-like performance,
#   6. contribution to institutional P&L,
#   7. asset-level attribution,
#   8. coalition-level attribution,
#   9. whether direction was correct,
#  10. whether high-conviction ideas actually performed better,
#  11. whether evidence was more informative than persuasion,
#  12. what happened to the hidden regime during evaluation.
#
# IMPORTANT
# ---------
# No LLM judges success here.
#
# Realized market outcomes adjudicate the competing beliefs.
#
# ============================================================


# ------------------------------------------------------------
# 1. Evaluation horizon
# ------------------------------------------------------------

EVAL_DAYS = 60

EVAL_START = EVENT_DAY + 1

EVAL_END = min(
    EVENT_DAY + EVAL_DAYS,
    len(world) - 1
)


future = world.loc[
    EVAL_START:EVAL_END,
    assets
].copy()


actual_eval_days = len(
    future
)


print("=" * 90)

print(
    "REALITY ADJUDICATES — "
    "OUT-OF-SAMPLE PERFORMANCE EVALUATION"
)

print("=" * 90)


print(
    f"Decision day        : {EVENT_DAY}"
)

print(
    f"Evaluation start    : {EVAL_START}"
)

print(
    f"Evaluation end      : {EVAL_END}"
)

print(
    f"Evaluation days     : {actual_eval_days}"
)

print(
    f"Starting capital    : "
    f"${INITIAL_CAPITAL:,.0f}"
)


# ------------------------------------------------------------
# 2. Hidden-regime path
#
# Agents do NOT receive this information during decision making.
# We reveal it only ex post for scientific evaluation.
# ------------------------------------------------------------

regime_path = world.loc[
    EVENT_DAY:EVAL_END,
    "hidden_regime"
]


regime_counts = (
    regime_path
    .value_counts()
    .to_dict()
)


starting_regime = world.loc[
    EVENT_DAY,
    "hidden_regime"
]


ending_regime = world.loc[
    EVAL_END,
    "hidden_regime"
]


regime_changed = (
    starting_regime
    !=
    ending_regime
)


print(
    "\nHIDDEN REGIME — REVEALED EX POST"
)

print(
    f"Starting regime     : {starting_regime}"
)

print(
    f"Ending regime       : {ending_regime}"
)

print(
    f"Regime changed      : {regime_changed}"
)

print(
    f"Regime distribution : {regime_counts}"
)


# ------------------------------------------------------------
# 3. Initialize coalition return matrix
#
# Each column represents the DAILY DOLLAR P&L of one funded
# coalition.
#
# This is superior to using only aggregate asset weights because
# multiple coalitions may hold the same asset with different
# directions and different capital amounts.
# ------------------------------------------------------------

coalition_daily_pnl = pd.DataFrame(
    index=future.index
)


coalition_daily_return = pd.DataFrame(
    index=future.index
)


episodes = []


# ------------------------------------------------------------
# 4. Evaluate every funded coalition independently
# ------------------------------------------------------------

for x in approved:

    cid = x["coalition_id"]

    asset = x["asset"]

    direction = x["direction"]

    capital = float(
        x["capital"]
    )


    sign = (
        -1.0
        if direction == "short"
        else 1.0
    )


    # --------------------------------------------------------
    # Position return series
    # --------------------------------------------------------

    position_return = (
        sign
        *
        future[asset]
    )


    coalition_daily_return[
        cid
    ] = position_return


    # --------------------------------------------------------
    # Dollar P&L
    #
    # Daily P&L is measured against allocated capital.
    # --------------------------------------------------------

    daily_pnl = (
        capital
        *
        position_return
    )


    coalition_daily_pnl[
        cid
    ] = daily_pnl


    # --------------------------------------------------------
    # Compounded coalition wealth curve
    # --------------------------------------------------------

    position_curve = (
        1.0
        +
        position_return
    ).cumprod()


    realized_return = float(
        position_curve.iloc[-1]
        -
        1.0
    )


    compounded_pnl = float(
        capital
        *
        realized_return
    )


    # --------------------------------------------------------
    # Maximum drawdown
    # --------------------------------------------------------

    running_peak = (
        position_curve
        .cummax()
    )


    position_drawdown = (
        position_curve
        /
        running_peak
        -
        1.0
    )


    max_drawdown = float(
        position_drawdown.min()
    )


    # --------------------------------------------------------
    # Realized volatility
    # --------------------------------------------------------

    realized_volatility = float(
        position_return.std()
        *
        np.sqrt(252)
    )


    # --------------------------------------------------------
    # Annualized mean return
    # --------------------------------------------------------

    annualized_mean = float(
        position_return.mean()
        *
        252
    )


    # --------------------------------------------------------
    # Sharpe-like diagnostic
    #
    # Synthetic experiment: zero risk-free rate for simplicity.
    # --------------------------------------------------------

    if realized_volatility > 1e-12:

        realized_sharpe = (
            annualized_mean
            /
            realized_volatility
        )

    else:

        realized_sharpe = 0.0


    # --------------------------------------------------------
    # Direction correctness
    #
    # Was the chosen direction profitable?
    # --------------------------------------------------------

    direction_correct = (
        realized_return
        >
        0
    )


    # --------------------------------------------------------
    # Retrieve ex-ante information
    # --------------------------------------------------------

    memo = memo_by_id.get(
        cid,
        {}
    )


    committee_decision = next(

        (
            d
            for d in decisions
            if d["coalition_id"] == cid
        ),

        {}
    )


    evidence_strength = float(
        memo.get(
            "evidence_strength",
            0.50
        )
    )


    persuasive_quality = float(
        memo.get(
            "persuasive_quality",
            0.50
        )
    )


    coalition_confidence = float(
        memo.get(
            "coalition_confidence",
            0.50
        )
    )


    debate_quality = float(
        memo.get(
            "debate_quality",
            0.50
        )
    )


    committee_conviction = float(
        committee_decision.get(
            "committee_conviction",
            x.get(
                "conviction",
                0.50
            )
        )
    )


    priority_score = float(
        x.get(
            "priority_score",
            0.50
        )
    )


    # --------------------------------------------------------
    # Record complete episode
    # --------------------------------------------------------

    episodes.append({

        "coalition_id":
            cid,

        "coalition_name":
            memo.get(
                "coalition_name",
                cid
            ),

        "decision_day":
            EVENT_DAY,

        "starting_hidden_regime":
            starting_regime,

        "ending_hidden_regime":
            ending_regime,

        "regime_changed":
            regime_changed,

        "capital":
            capital,

        "asset":
            asset,

        "direction":
            direction,

        "realized_return":
            realized_return,

        "pnl":
            compounded_pnl,

        "max_drawdown":
            max_drawdown,

        "realized_volatility":
            realized_volatility,

        "realized_sharpe":
            realized_sharpe,

        "direction_correct":
            direction_correct,

        "evidence_strength":
            evidence_strength,

        "persuasive_quality":
            persuasive_quality,

        "eloquence_gap":
            persuasive_quality
            -
            evidence_strength,

        "coalition_confidence":
            coalition_confidence,

        "debate_quality":
            debate_quality,

        "committee_conviction":
            committee_conviction,

        "priority_score":
            priority_score

    })


# ------------------------------------------------------------
# 5. Build coalition episode dataframe
# ------------------------------------------------------------

episode_df = pd.DataFrame(
    episodes
)


# ------------------------------------------------------------
# 6. Institutional daily P&L
#
# Aggregate exact dollar P&L from all funded coalitions.
# ------------------------------------------------------------

if len(
    coalition_daily_pnl.columns
):

    institutional_daily_pnl = (
        coalition_daily_pnl
        .sum(
            axis=1
        )
    )

else:

    institutional_daily_pnl = pd.Series(

        0.0,

        index=future.index

    )


# ------------------------------------------------------------
# 7. Institutional wealth curve
#
# Wealth evolves from actual dollar P&L.
#
# Undeployed capital remains cash and earns zero in this
# pedagogical experiment.
# ------------------------------------------------------------

wealth_series = (

    INITIAL_CAPITAL

    +

    institutional_daily_pnl
    .cumsum()

)


wealth = wealth_series.values


# ------------------------------------------------------------
# 8. Institutional return series
# ------------------------------------------------------------

institutional_daily_return = (

    institutional_daily_pnl

    /
    INITIAL_CAPITAL

)


# ------------------------------------------------------------
# 9. Institutional drawdown
# ------------------------------------------------------------

wealth_peak = (
    wealth_series
    .cummax()
)


drawdown_series = (

    wealth_series

    /
    wealth_peak

    -
    1.0

)


max_drawdown = float(
    drawdown_series.min()
)


# ------------------------------------------------------------
# 10. Ending wealth and total institutional P&L
# ------------------------------------------------------------

ending_wealth = float(
    wealth_series.iloc[-1]
)


institutional_pnl = (

    ending_wealth

    -
    INITIAL_CAPITAL

)


institutional_return = (

    ending_wealth

    /
    INITIAL_CAPITAL

    -
    1.0

)


# ------------------------------------------------------------
# 11. Institutional realized volatility
# ------------------------------------------------------------

institutional_volatility = float(

    institutional_daily_return.std()

    *
    np.sqrt(252)

)


institutional_annualized_mean = float(

    institutional_daily_return.mean()

    *
    252

)


if institutional_volatility > 1e-12:

    institutional_sharpe = (

        institutional_annualized_mean

        /
        institutional_volatility

    )

else:

    institutional_sharpe = 0.0


# ------------------------------------------------------------
# 12. Coalition contribution to institutional P&L
# ------------------------------------------------------------

if len(
    episode_df
):

    episode_df[
        "pnl_contribution"
    ] = (

        episode_df[
            "pnl"
        ]

        /
        max(
            abs(
                episode_df[
                    "pnl"
                ]
                .sum()
            ),
            1.0
        )

    )


    episode_df[
        "capital_weight"
    ] = (

        episode_df[
            "capital"
        ]

        /
        INITIAL_CAPITAL

    )


    episode_df[
        "return_on_total_capital"
    ] = (

        episode_df[
            "pnl"
        ]

        /
        INITIAL_CAPITAL

    )


# ------------------------------------------------------------
# 13. Asset-level P&L attribution
# ------------------------------------------------------------

if len(
    episode_df
):

    asset_attribution = (

        episode_df

        .groupby(
            "asset"
        )

        .agg(

            capital=(
                "capital",
                "sum"
            ),

            pnl=(
                "pnl",
                "sum"
            ),

            coalitions=(
                "coalition_id",
                "count"
            ),

            avg_realized_return=(
                "realized_return",
                "mean"
            )

        )

        .reset_index()

    )


    asset_attribution[
        "pnl_pct_initial_capital"
    ] = (

        asset_attribution[
            "pnl"
        ]

        /
        INITIAL_CAPITAL

    )

else:

    asset_attribution = pd.DataFrame()


# ------------------------------------------------------------
# 14. Direction-level attribution
# ------------------------------------------------------------

if len(
    episode_df
):

    direction_attribution = (

        episode_df

        .groupby(
            "direction"
        )

        .agg(

            capital=(
                "capital",
                "sum"
            ),

            pnl=(
                "pnl",
                "sum"
            ),

            coalitions=(
                "coalition_id",
                "count"
            ),

            hit_rate=(
                "direction_correct",
                "mean"
            )

        )

        .reset_index()

    )

else:

    direction_attribution = pd.DataFrame()


# ------------------------------------------------------------
# 15. Reality score
#
# Create a bounded ex-post performance signal for Cell 13.
#
# This is NOT used to determine P&L.
# It summarizes how well each coalition performed relative
# to risk for reputation updating.
# ------------------------------------------------------------

if len(
    episode_df
):

    raw_reality_score = (

        4.0
        *
        episode_df[
            "realized_return"
        ]

        +

        1.5
        *
        episode_df[
            "max_drawdown"
        ]

    )


    episode_df[
        "reality_score"
    ] = (

        1.0

        /
        (
            1.0
            +
            np.exp(
                -raw_reality_score
            )
        )

    )


# ------------------------------------------------------------
# 16. Was evidence more informative than persuasion?
# ------------------------------------------------------------

def safe_corr(
    x,
    y
):

    if len(x) < 2:

        return np.nan


    if np.std(x) < 1e-12:

        return np.nan


    if np.std(y) < 1e-12:

        return np.nan


    return float(
        np.corrcoef(
            x,
            y
        )[0, 1]
    )


if len(
    episode_df
):

    evidence_return_corr = safe_corr(

        episode_df[
            "evidence_strength"
        ],

        episode_df[
            "realized_return"
        ]

    )


    persuasion_return_corr = safe_corr(

        episode_df[
            "persuasive_quality"
        ],

        episode_df[
            "realized_return"
        ]

    )


    conviction_return_corr = safe_corr(

        episode_df[
            "committee_conviction"
        ],

        episode_df[
            "realized_return"
        ]

    )


    debate_return_corr = safe_corr(

        episode_df[
            "debate_quality"
        ],

        episode_df[
            "realized_return"
        ]

    )

else:

    evidence_return_corr = np.nan

    persuasion_return_corr = np.nan

    conviction_return_corr = np.nan

    debate_return_corr = np.nan


# ------------------------------------------------------------
# 17. Hit rate
# ------------------------------------------------------------

if len(
    episode_df
):

    hit_rate = float(

        episode_df[
            "direction_correct"
        ]
        .mean()

    )

else:

    hit_rate = np.nan


# ------------------------------------------------------------
# 18. Best and worst coalition
# ------------------------------------------------------------

if len(
    episode_df
):

    best_idx = (
        episode_df[
            "pnl"
        ]
        .idxmax()
    )


    worst_idx = (
        episode_df[
            "pnl"
        ]
        .idxmin()
    )


    best_coalition = (
        episode_df
        .loc[
            best_idx
        ]
    )


    worst_coalition = (
        episode_df
        .loc[
            worst_idx
        ]
    )

else:

    best_coalition = None

    worst_coalition = None


# ------------------------------------------------------------
# 19. Display institutional performance
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "INSTITUTIONAL PERFORMANCE"
)

print(
    "=" * 90
)


print(
    f"Ending wealth              : "
    f"${ending_wealth:,.0f}"
)


print(
    f"Institutional P&L          : "
    f"${institutional_pnl:,.0f}"
)


print(
    f"Institutional return       : "
    f"{institutional_return:.2%}"
)


print(
    f"Maximum drawdown           : "
    f"{max_drawdown:.2%}"
)


print(
    f"Realized annualized vol    : "
    f"{institutional_volatility:.2%}"
)


print(
    f"Sharpe-like ratio          : "
    f"{institutional_sharpe:.2f}"
)


if not np.isnan(
    hit_rate
):

    print(
        f"Coalition hit rate         : "
        f"{hit_rate:.1%}"
    )


# ------------------------------------------------------------
# 20. Display coalition performance
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "COALITION PERFORMANCE ATTRIBUTION"
)

print(
    "=" * 90
)


if len(
    episode_df
):

    coalition_display_columns = [

        "coalition_id",

        "coalition_name",

        "asset",

        "direction",

        "capital",

        "realized_return",

        "pnl",

        "max_drawdown",

        "realized_volatility",

        "realized_sharpe",

        "direction_correct",

        "committee_conviction",

        "evidence_strength",

        "persuasive_quality",

        "reality_score"

    ]


    display(

        episode_df[

            coalition_display_columns

        ].sort_values(

            "pnl",

            ascending=False

        )

    )

else:

    print(
        "No funded coalition episodes to evaluate."
    )


# ------------------------------------------------------------
# 21. Display asset attribution
# ------------------------------------------------------------

print(
    "\nASSET-LEVEL ATTRIBUTION"
)


if len(
    asset_attribution
):

    display(

        asset_attribution.sort_values(

            "pnl",

            ascending=False

        )

    )


# ------------------------------------------------------------
# 22. Display direction attribution
# ------------------------------------------------------------

print(
    "\nLONG / SHORT ATTRIBUTION"
)


if len(
    direction_attribution
):

    display(
        direction_attribution
    )


# ------------------------------------------------------------
# 23. Epistemic diagnostic
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "EPISTEMIC DIAGNOSTIC — "
    "WHAT PREDICTED REALITY?"
)

print(
    "=" * 90
)


print(
    f"Evidence vs. realized return    : "
    f"{evidence_return_corr:.3f}"
    if not np.isnan(
        evidence_return_corr
    )
    else
    "Evidence vs. realized return    : N/A"
)


print(
    f"Persuasion vs. realized return  : "
    f"{persuasion_return_corr:.3f}"
    if not np.isnan(
        persuasion_return_corr
    )
    else
    "Persuasion vs. realized return  : N/A"
)


print(
    f"Committee conviction vs. return : "
    f"{conviction_return_corr:.3f}"
    if not np.isnan(
        conviction_return_corr
    )
    else
    "Committee conviction vs. return : N/A"
)


print(
    f"Debate quality vs. return       : "
    f"{debate_return_corr:.3f}"
    if not np.isnan(
        debate_return_corr
    )
    else
    "Debate quality vs. return       : N/A"
)


# ------------------------------------------------------------
# 24. Evidence versus eloquence comparison
# ------------------------------------------------------------

if (

    not np.isnan(
        evidence_return_corr
    )

    and

    not np.isnan(
        persuasion_return_corr
    )

):

    if (
        evidence_return_corr
        >
        persuasion_return_corr
    ):

        print(
            "\nIn this episode, evidence quality was more "
            "informative about realized outcomes than persuasion."
        )

    elif (
        persuasion_return_corr
        >
        evidence_return_corr
    ):

        print(
            "\nIn this episode, persuasive quality happened to "
            "correlate more strongly with realized outcomes than "
            "the ex-ante evidence score. This is an observation, "
            "not proof that persuasion contains information."
        )

    else:

        print(
            "\nEvidence and persuasion had similar ex-post "
            "relationships with realized outcomes."
        )


# ------------------------------------------------------------
# 25. Best / worst coalition diagnostic
# ------------------------------------------------------------

if best_coalition is not None:

    print(
        "\n"
        + "=" * 90
    )

    print(
        "REALITY'S STRONGEST AND WEAKEST VERDICTS"
    )

    print(
        "=" * 90
    )


    print(
        "\nBEST COALITION"
    )


    print(
        f"{best_coalition['coalition_id']} — "
        f"{best_coalition['coalition_name']}"
    )


    print(
        f"Asset / direction : "
        f"{best_coalition['direction'].upper()} "
        f"{best_coalition['asset']}"
    )


    print(
        f"Realized return   : "
        f"{best_coalition['realized_return']:.2%}"
    )


    print(
        f"P&L               : "
        f"${best_coalition['pnl']:,.0f}"
    )


    print(
        f"Evidence strength : "
        f"{best_coalition['evidence_strength']:.3f}"
    )


    print(
        f"Persuasive quality: "
        f"{best_coalition['persuasive_quality']:.3f}"
    )


    print(
        "\nWORST COALITION"
    )


    print(
        f"{worst_coalition['coalition_id']} — "
        f"{worst_coalition['coalition_name']}"
    )


    print(
        f"Asset / direction : "
        f"{worst_coalition['direction'].upper()} "
        f"{worst_coalition['asset']}"
    )


    print(
        f"Realized return   : "
        f"{worst_coalition['realized_return']:.2%}"
    )


    print(
        f"P&L               : "
        f"${worst_coalition['pnl']:,.0f}"
    )


    print(
        f"Evidence strength : "
        f"{worst_coalition['evidence_strength']:.3f}"
    )


    print(
        f"Persuasive quality: "
        f"{worst_coalition['persuasive_quality']:.3f}"
    )


# ------------------------------------------------------------
# 26. Regime sensitivity diagnostic
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "REGIME DIAGNOSTIC"
)

print(
    "=" * 90
)


if regime_changed:

    print(
        "The hidden market regime changed during the evaluation "
        "window. Coalition outcomes therefore test not only thesis "
        "quality, but also robustness to environmental change."
    )

else:

    print(
        "The hidden market regime remained stable across the "
        "evaluation window. Outcomes primarily test thesis quality "
        "within a relatively persistent environment."
    )


print(
    "\nObserved hidden-regime occupancy:"
)


for regime, count in regime_counts.items():

    print(
        f"  {regime:20s}: "
        f"{count:3d} days "
        f"({count / len(regime_path):.1%})"
    )


# ------------------------------------------------------------
# 27. Create institutional performance record
#
# Cell 13 can use this for reputation and learning.
# ------------------------------------------------------------

performance_record = {

    "decision_day":
        EVENT_DAY,

    "evaluation_days":
        actual_eval_days,

    "starting_regime":
        starting_regime,

    "ending_regime":
        ending_regime,

    "regime_changed":
        regime_changed,

    "starting_capital":
        float(
            INITIAL_CAPITAL
        ),

    "ending_wealth":
        ending_wealth,

    "institutional_pnl":
        institutional_pnl,

    "institutional_return":
        institutional_return,

    "max_drawdown":
        max_drawdown,

    "realized_volatility":
        institutional_volatility,

    "realized_sharpe":
        institutional_sharpe,

    "coalition_hit_rate":
        hit_rate,

    "evidence_return_correlation":
        evidence_return_corr,

    "persuasion_return_correlation":
        persuasion_return_corr,

    "committee_return_correlation":
        conviction_return_corr,

    "debate_return_correlation":
        debate_return_corr,

    "regime_counts":
        regime_counts

}


# ------------------------------------------------------------
# 28. Final institutional statement
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "REALITY HAS SPOKEN"
)

print(
    "=" * 90
)


print(
    f"The institution began the episode with "
    f"${INITIAL_CAPITAL:,.0f}."
)


print(
    f"After {actual_eval_days} out-of-sample trading days, "
    f"its wealth is ${ending_wealth:,.0f}."
)


print(
    f"The realized institutional return was "
    f"{institutional_return:.2%}, with a maximum drawdown of "
    f"{max_drawdown:.2%}."
)


print(
    "\nThe next stage is not simply to reward winners and punish "
    "losers. Cell 13 must ask a more difficult institutional "
    "question: WHICH agents, coalitions, arguments, evidence, "
    "and governance judgments actually deserve greater trust "
    "after observing reality?"
)

REALITY ADJUDICATES — OUT-OF-SAMPLE PERFORMANCE EVALUATION
Decision day        : 100
Evaluation start    : 101
Evaluation end      : 160
Evaluation days     : 60
Starting capital    : $10,000,000,000

HIDDEN REGIME — REVEALED EX POST
Starting regime     : Calm Growth
Ending regime       : Financial Stress
Regime changed      : True
Regime distribution : {'Calm Growth': 47, 'Technology Boom': 6, 'Recovery': 6, 'Inflation Shock': 1, 'Financial Stress': 1}

INSTITUTIONAL PERFORMANCE
Ending wealth              : $9,898,489,377
Institutional P&L          : $-101,510,623
Institutional return       : -1.02%
Maximum drawdown           : -3.32%
Realized annualized vol    : 5.59%
Sharpe-like ratio          : -0.76
Coalition hit rate         : 0.0%

COALITION PERFORMANCE ATTRIBUTION


,coalition_id,coalition_name,asset,direction,capital,realized_return,pnl,max_drawdown,realized_volatility,realized_sharpe,direction_correct,committee_conviction,evidence_strength,persuasive_quality,reality_score
0,C01,Emergent Coalition 1,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
1,C02,Emergent Coalition 2,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
2,C03,Emergent Coalition 3,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
3,C04,Emergent Coalition 4,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
4,C05,Emergent Coalition 5,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
5,C06,Emergent Coalition 6,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
7,C08,Emergent Coalition 8,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
8,C09,Emergent Coalition 9,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025
9,C10,Emergent Coalition 10,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.155374,-0.762217,False,0.55275,0.527625,0.575,0.436025



ASSET-LEVEL ATTRIBUTION


,asset,capital,pnl,coalitions,avg_realized_return,pnl_pct_initial_capital
0,EQUITY,3.600000e+09,-1.100043e+08,12,-0.030557,-0.011



LONG / SHORT ATTRIBUTION


,direction,capital,pnl,coalitions,hit_rate
0,long,3.600000e+09,-1.100043e+08,12,0.0



EPISTEMIC DIAGNOSTIC — WHAT PREDICTED REALITY?
Evidence vs. realized return    : N/A
Persuasion vs. realized return  : N/A
Committee conviction vs. return : N/A
Debate quality vs. return       : N/A

REALITY'S STRONGEST AND WEAKEST VERDICTS

BEST COALITION
C01 — Emergent Coalition 1
Asset / direction : LONG EQUITY
Realized return   : -3.06%
P&L               : $-9,167,026
Evidence strength : 0.528
Persuasive quality: 0.575

WORST COALITION
C01 — Emergent Coalition 1
Asset / direction : LONG EQUITY
Realized return   : -3.06%
P&L               : $-9,167,026
Evidence strength : 0.528
Persuasive quality: 0.575

REGIME DIAGNOSTIC
The hidden market regime changed during the evaluation window. Coalition outcomes therefore test not only thesis quality, but also robustness to environmental change.

Observed hidden-regime occupancy:
  Calm Growth         :  47 days (77.0%)
  Technology Boom     :   6 days (9.8%)
  Recovery            :   6 days (9.8%)
  Inflation Shock     :   1 days (1.6%)
  F

## CODE UNIT 13 — Reputation, persuasion calibration and institutional memory

A genuinely adaptive organization must learn whom to listen to. This unit updates agent-level epistemic score, persuasive score, reputation, and influence after realized outcomes. These dimensions remain separate.

An agent can be persuasive but poorly calibrated. Another can be repeatedly correct but institutionally weak. A third can possess high reputation because it contributes unique information even when it rarely leads coalitions. Such distinctions are common in human organizations and become measurable here.

The update mechanism rewards calibration between confidence and realized outcomes while also recording rhetorical quality. Coalition episodes are compressed into institutional memory: what was believed, under which regime, with what evidence, and what happened afterward.

Future Investment Committees can receive relevant precedents. This creates path dependence without requiring the model to carry the entire conversational history forever.

The objective is not to construct the perfect reputation formula. It is to demonstrate that epistemic authority can evolve endogenously. The institution begins with roughly equal voices but gradually develops a history of trust, skepticism, and influence.

In [28]:
# ============================================================
# CODE UNIT 13 — REPUTATION, EPISTEMIC LEARNING
#                AND INSTITUTIONAL MEMORY
# ============================================================
#
# PURPOSE
# -------
# Cell 12 allowed market reality to adjudicate the institution's
# competing beliefs.
#
# Cell 13 asks:
#
#       "What should the institution LEARN from what happened?"
#
# This is deliberately NOT:
#
#       winner  -> reward everybody involved
#       loser   -> punish everybody involved
#
# A sophisticated institution must distinguish:
#
#   1. epistemic quality,
#   2. realized outcome,
#   3. quality of reasoning,
#   4. persuasive ability,
#   5. committee judgment,
#   6. regime dependence,
#   7. luck versus repeatable insight.
#
# We therefore update:
#
#       AGENT REPUTATION
#       AGENT EPISTEMIC SCORE
#       AGENT PERSUASION SCORE
#       AGENT INFLUENCE
#       COALITION TRACK RECORD
#       INSTITUTIONAL MEMORY
#
# IMPORTANT:
#
# Persuasion is tracked separately from epistemic accuracy.
# A persuasive but wrong agent must NOT become more trusted
# merely because the argument sounded compelling.
#
# ============================================================


# ------------------------------------------------------------
# 1. Learning parameters
# ------------------------------------------------------------

AGENT_LEARNING_RATE = 0.20

EPISTEMIC_LEARNING_RATE = 0.25

PERSUASION_LEARNING_RATE = 0.10

INFLUENCE_LEARNING_RATE = 0.15

REPUTATION_FLOOR = 0.05

REPUTATION_CEILING = 0.95

MEMORY_LIMIT = 100


def clip_score(x):

    return float(
        np.clip(
            float(x),
            REPUTATION_FLOOR,
            REPUTATION_CEILING
        )
    )


# ------------------------------------------------------------
# 2. Helper: bounded transformation of realized performance
#
# Raw returns should not directly determine reputation.
#
# A +20% return should not make reputation jump by 20 points.
# We therefore transform realized risk-adjusted performance
# into a bounded [0,1] learning signal.
# ------------------------------------------------------------

def performance_signal(
    realized_return,
    max_drawdown,
    realized_volatility
):

    realized_return = float(
        realized_return
    )

    max_drawdown = float(
        max_drawdown
    )

    realized_volatility = float(
        realized_volatility
    )


    # Reward return
    # Penalize drawdown
    # Mildly penalize excessive volatility

    raw = (

        5.0 * realized_return

        +

        1.5 * max_drawdown

        -

        0.20 * realized_volatility

    )


    signal = (

        1.0

        /

        (
            1.0
            +
            np.exp(
                -raw
            )
        )

    )


    return float(
        np.clip(
            signal,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 3. Build coalition outcome lookup
# ------------------------------------------------------------

episode_lookup = {

    row["coalition_id"]:
        row

    for _, row in episode_df.iterrows()

}


coalition_lookup = {

    c["id"]:
        c

    for c in coalitions

}


memo_lookup = {

    m["coalition_id"]:
        m

    for m in memos

}


decision_lookup = {

    d["coalition_id"]:
        d

    for d in decisions

}


# ------------------------------------------------------------
# 4. Coalition learning records
# ------------------------------------------------------------

coalition_learning = []


for cid, episode in episode_lookup.items():

    coalition = coalition_lookup.get(
        cid,
        {}
    )

    memo = memo_lookup.get(
        cid,
        {}
    )

    decision = decision_lookup.get(
        cid,
        {}
    )


    # --------------------------------------------------------
    # Realized performance signal
    # --------------------------------------------------------

    outcome_score = performance_signal(

        episode[
            "realized_return"
        ],

        episode[
            "max_drawdown"
        ],

        episode[
            "realized_volatility"
        ]

    )


    # --------------------------------------------------------
    # Epistemic quality
    #
    # Combines:
    # - realized outcome
    # - ex-ante evidence quality
    # - debate quality
    #
    # Outcome receives the largest weight.
    # --------------------------------------------------------

    evidence = float(
        episode.get(
            "evidence_strength",
            0.50
        )
    )


    debate_quality = float(
        episode.get(
            "debate_quality",
            0.50
        )
    )


    epistemic_quality = (

        0.55 * outcome_score

        +

        0.30 * evidence

        +

        0.15 * debate_quality

    )


    epistemic_quality = float(
        np.clip(
            epistemic_quality,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Persuasion quality
    #
    # Kept separate from epistemic success.
    # --------------------------------------------------------

    persuasion = float(
        episode.get(
            "persuasive_quality",
            0.50
        )
    )


    # --------------------------------------------------------
    # Calibration
    #
    # Was ex-ante confidence justified by reality?
    # --------------------------------------------------------

    ex_ante_confidence = float(
        episode.get(
            "coalition_confidence",
            0.50
        )
    )


    calibration_error = abs(

        ex_ante_confidence

        -

        outcome_score

    )


    calibration_score = float(

        1.0

        -

        np.clip(
            calibration_error,
            0.0,
            1.0
        )

    )


    # --------------------------------------------------------
    # Committee judgment quality
    #
    # Compare committee conviction with realized outcome.
    # --------------------------------------------------------

    committee_conviction = float(
        episode.get(
            "committee_conviction",
            0.50
        )
    )


    committee_error = abs(

        committee_conviction

        -

        outcome_score

    )


    committee_calibration = float(

        1.0

        -

        np.clip(
            committee_error,
            0.0,
            1.0
        )

    )


    # --------------------------------------------------------
    # Eloquence risk
    #
    # Persuasion exceeding evidence is not automatically bad,
    # but it becomes concerning when outcomes are poor.
    # --------------------------------------------------------

    eloquence_gap = float(

        persuasion

        -

        evidence

    )


    eloquence_failure = (

        max(
            0.0,
            eloquence_gap
        )

        *

        max(
            0.0,
            0.50 - outcome_score
        )

        *

        2.0

    )


    eloquence_failure = float(
        np.clip(
            eloquence_failure,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Coalition learning score
    # --------------------------------------------------------

    coalition_learning_score = (

        0.45 * epistemic_quality

        +

        0.20 * calibration_score

        +

        0.20 * committee_calibration

        +

        0.15 * debate_quality

        -

        0.15 * eloquence_failure

    )


    coalition_learning_score = float(
        np.clip(
            coalition_learning_score,
            0.0,
            1.0
        )
    )


    coalition_learning.append({

        "coalition_id":
            cid,

        "coalition_name":
            episode.get(
                "coalition_name",
                cid
            ),

        "asset":
            episode[
                "asset"
            ],

        "direction":
            episode[
                "direction"
            ],

        "capital":
            float(
                episode[
                    "capital"
                ]
            ),

        "realized_return":
            float(
                episode[
                    "realized_return"
                ]
            ),

        "pnl":
            float(
                episode[
                    "pnl"
                ]
            ),

        "max_drawdown":
            float(
                episode[
                    "max_drawdown"
                ]
            ),

        "outcome_score":
            outcome_score,

        "evidence_strength":
            evidence,

        "persuasive_quality":
            persuasion,

        "debate_quality":
            debate_quality,

        "ex_ante_confidence":
            ex_ante_confidence,

        "calibration_score":
            calibration_score,

        "committee_conviction":
            committee_conviction,

        "committee_calibration":
            committee_calibration,

        "eloquence_gap":
            eloquence_gap,

        "eloquence_failure":
            eloquence_failure,

        "epistemic_quality":
            epistemic_quality,

        "learning_score":
            coalition_learning_score,

        "starting_regime":
            episode[
                "starting_hidden_regime"
            ],

        "ending_regime":
            episode[
                "ending_hidden_regime"
            ],

        "regime_changed":
            bool(
                episode[
                    "regime_changed"
                ]
            )

    })


coalition_learning_df = pd.DataFrame(
    coalition_learning
)


print(
    "=" * 90
)

print(
    "COALITION LEARNING AFTER REALITY"
)

print(
    "=" * 90
)


if len(
    coalition_learning_df
):

    display(

        coalition_learning_df.sort_values(

            "learning_score",

            ascending=False

        )

    )


# ------------------------------------------------------------
# 5. Preserve pre-update agent state
# ------------------------------------------------------------

agent_before = {}


for a in agents:

    agent_before[
        a.id
    ] = {

        "reputation":
            float(
                a.reputation
            ),

        "epistemic_score":
            float(
                a.epistemic_score
            ),

        "persuasion_score":
            float(
                a.persuasion_score
            ),

        "influence":
            float(
                a.influence
            )

    }


# ------------------------------------------------------------
# 6. Collect learning signals for each agent
#
# Because coalitions overlap, one agent may receive signals
# from several coalition outcomes.
# ------------------------------------------------------------

agent_signals = {

    a.id: []

    for a in agents

}


for record in coalition_learning:

    cid = record[
        "coalition_id"
    ]


    coalition = coalition_lookup.get(
        cid
    )


    if coalition is None:

        continue


    for agent_id in coalition[
        "members"
    ]:

        agent_signals[
            agent_id
        ].append(
            record
        )


# ------------------------------------------------------------
# 7. Update each agent
# ------------------------------------------------------------

agent_learning_records = []


for a in agents:

    signals = agent_signals[
        a.id
    ]


    old_reputation = float(
        a.reputation
    )

    old_epistemic = float(
        a.epistemic_score
    )

    old_persuasion = float(
        a.persuasion_score
    )

    old_influence = float(
        a.influence
    )


    # --------------------------------------------------------
    # Agents with no funded coalition:
    # preserve their scores.
    #
    # No evidence is not negative evidence.
    # --------------------------------------------------------

    if len(
        signals
    ) == 0:

        agent_learning_records.append({

            "agent_id":
                a.id,

            "specialty":
                a.specialty,

            "coalitions_evaluated":
                0,

            "old_reputation":
                old_reputation,

            "new_reputation":
                old_reputation,

            "reputation_change":
                0.0,

            "old_epistemic":
                old_epistemic,

            "new_epistemic":
                old_epistemic,

            "old_persuasion":
                old_persuasion,

            "new_persuasion":
                old_persuasion,

            "old_influence":
                old_influence,

            "new_influence":
                old_influence,

            "avg_outcome_score":
                np.nan,

            "avg_learning_score":
                np.nan

        })

        continue


    # --------------------------------------------------------
    # Capital-weight the agent's coalition experiences
    #
    # Larger institutional bets provide more information,
    # but weights are normalized so one episode cannot
    # mechanically dominate without bound.
    # --------------------------------------------------------

    capitals = np.array(

        [
            s["capital"]
            for s in signals
        ],

        dtype=float

    )


    if capitals.sum() > 0:

        weights = (

            capitals

            /
            capitals.sum()

        )

    else:

        weights = np.repeat(

            1.0
            /
            len(signals),

            len(signals)

        )


    avg_outcome = float(

        np.average(

            [
                s[
                    "outcome_score"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    avg_epistemic_quality = float(

        np.average(

            [
                s[
                    "epistemic_quality"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    avg_persuasion = float(

        np.average(

            [
                s[
                    "persuasive_quality"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    avg_calibration = float(

        np.average(

            [
                s[
                    "calibration_score"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    avg_learning = float(

        np.average(

            [
                s[
                    "learning_score"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    avg_eloquence_failure = float(

        np.average(

            [
                s[
                    "eloquence_failure"
                ]

                for s in signals
            ],

            weights=weights

        )

    )


    # --------------------------------------------------------
    # 8. Update epistemic score
    #
    # This measures whether the agent tends to participate
    # in coalitions whose reasoning survives contact with reality.
    # --------------------------------------------------------

    new_epistemic = (

        (1.0 - EPISTEMIC_LEARNING_RATE)

        *
        old_epistemic

        +

        EPISTEMIC_LEARNING_RATE

        *
        avg_epistemic_quality

    )


    new_epistemic = clip_score(
        new_epistemic
    )


    # --------------------------------------------------------
    # 9. Update persuasion score
    #
    # Persuasion is learned separately.
    #
    # We do NOT equate persuasive ability with truth.
    # --------------------------------------------------------

    new_persuasion = (

        (1.0 - PERSUASION_LEARNING_RATE)

        *
        old_persuasion

        +

        PERSUASION_LEARNING_RATE

        *
        avg_persuasion

    )


    new_persuasion = clip_score(
        new_persuasion
    )


    # --------------------------------------------------------
    # 10. Update reputation
    #
    # Reputation should reflect:
    #
    #   epistemic quality,
    #   calibration,
    #   realized outcomes,
    #   and resistance to persuasive failure.
    # --------------------------------------------------------

    reputation_target = (

        0.45 * avg_epistemic_quality

        +

        0.25 * avg_outcome

        +

        0.20 * avg_calibration

        +

        0.10 * avg_learning

        -

        0.15 * avg_eloquence_failure

    )


    reputation_target = float(
        np.clip(
            reputation_target,
            0.0,
            1.0
        )
    )


    new_reputation = (

        (1.0 - AGENT_LEARNING_RATE)

        *
        old_reputation

        +

        AGENT_LEARNING_RATE

        *
        reputation_target

    )


    new_reputation = clip_score(
        new_reputation
    )


    # --------------------------------------------------------
    # 11. Update influence
    #
    # Influence is NOT popularity.
    #
    # It depends primarily on reputation and epistemic quality,
    # with only a small contribution from persuasion.
    # --------------------------------------------------------

    influence_target = (

        0.50 * new_reputation

        +

        0.35 * new_epistemic

        +

        0.15 * new_persuasion

    )


    new_influence = (

        (1.0 - INFLUENCE_LEARNING_RATE)

        *
        old_influence

        +

        INFLUENCE_LEARNING_RATE

        *
        influence_target

    )


    new_influence = clip_score(
        new_influence
    )


    # --------------------------------------------------------
    # 12. Commit updates to agent object
    # --------------------------------------------------------

    a.epistemic_score = (
        new_epistemic
    )

    a.persuasion_score = (
        new_persuasion
    )

    a.reputation = (
        new_reputation
    )

    a.influence = (
        new_influence
    )


    # --------------------------------------------------------
    # 13. Record update
    # --------------------------------------------------------

    agent_learning_records.append({

        "agent_id":
            a.id,

        "specialty":
            a.specialty,

        "coalitions_evaluated":
            len(signals),

        "old_reputation":
            old_reputation,

        "new_reputation":
            new_reputation,

        "reputation_change":
            new_reputation
            -
            old_reputation,

        "old_epistemic":
            old_epistemic,

        "new_epistemic":
            new_epistemic,

        "old_persuasion":
            old_persuasion,

        "new_persuasion":
            new_persuasion,

        "old_influence":
            old_influence,

        "new_influence":
            new_influence,

        "avg_outcome_score":
            avg_outcome,

        "avg_learning_score":
            avg_learning,

        "avg_calibration":
            avg_calibration,

        "avg_eloquence_failure":
            avg_eloquence_failure

    })


agent_learning_df = pd.DataFrame(
    agent_learning_records
)


# ------------------------------------------------------------
# 14. Display reputation changes
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "AGENT REPUTATION AND EPISTEMIC UPDATES"
)

print(
    "=" * 90
)


display(

    agent_learning_df.sort_values(

        "new_reputation",

        ascending=False

    )

)


# ------------------------------------------------------------
# 15. Biggest reputation gains and losses
# ------------------------------------------------------------

evaluated_agents = agent_learning_df[

    agent_learning_df[
        "coalitions_evaluated"
    ]

    > 0

].copy()


if len(
    evaluated_agents
):

    print(
        "\nLARGEST REPUTATION GAINS"
    )


    display(

        evaluated_agents.sort_values(

            "reputation_change",

            ascending=False

        ).head(10)

    )


    print(
        "\nLARGEST REPUTATION DECLINES"
    )


    display(

        evaluated_agents.sort_values(

            "reputation_change",

            ascending=True

        ).head(10)

    )


# ------------------------------------------------------------
# 16. Detect persuasive-but-wrong agents
#
# This is one of the central institutional diagnostics.
# ------------------------------------------------------------

if len(
    evaluated_agents
):

    persuasive_risk_agents = (

        evaluated_agents[

            (
                evaluated_agents[
                    "new_persuasion"
                ]
                >
                evaluated_agents[
                    "new_epistemic"
                ]
                +
                0.15
            )

            &

            (
                evaluated_agents[
                    "avg_outcome_score"
                ]
                <
                0.50
            )

        ]

        .sort_values(

            "new_persuasion",

            ascending=False

        )

    )


    print(
        "\n"
        + "=" * 90
    )

    print(
        "PERSUASION RISK DIAGNOSTIC"
    )

    print(
        "=" * 90
    )


    if len(
        persuasive_risk_agents
    ):

        print(
            "The following agents are currently more persuasive "
            "than epistemically reliable and participated in "
            "below-neutral realized outcomes:"
        )


        display(

            persuasive_risk_agents[

                [
                    "agent_id",
                    "specialty",
                    "new_reputation",
                    "new_epistemic",
                    "new_persuasion",
                    "avg_outcome_score",
                    "avg_eloquence_failure"
                ]

            ]

        )

    else:

        print(
            "No agent currently exhibits a strong "
            "persuasive-but-wrong signature."
        )


# ------------------------------------------------------------
# 17. Detect quiet experts
#
# Strong epistemic performance with relatively weak persuasion.
# These agents may deserve greater institutional attention.
# ------------------------------------------------------------

if len(
    evaluated_agents
):

    quiet_experts = (

        evaluated_agents[

            (
                evaluated_agents[
                    "new_epistemic"
                ]
                >
                evaluated_agents[
                    "new_persuasion"
                ]
                +
                0.10
            )

            &

            (
                evaluated_agents[
                    "avg_outcome_score"
                ]
                >
                0.50
            )

        ]

        .sort_values(

            "new_epistemic",

            ascending=False

        )

    )


    print(
        "\nQUIET EXPERT DIAGNOSTIC"
    )


    if len(
        quiet_experts
    ):

        display(

            quiet_experts[

                [
                    "agent_id",
                    "specialty",
                    "new_reputation",
                    "new_epistemic",
                    "new_persuasion",
                    "new_influence",
                    "avg_outcome_score"
                ]

            ]

        )

    else:

        print(
            "No strong quiet-expert pattern detected "
            "in this episode."
        )


# ------------------------------------------------------------
# 18. Committee learning
#
# Did committee conviction correspond to reality?
# ------------------------------------------------------------

if len(
    coalition_learning_df
):

    committee_mae = float(

        np.mean(

            np.abs(

                coalition_learning_df[
                    "committee_conviction"
                ]

                -

                coalition_learning_df[
                    "outcome_score"
                ]

            )

        )

    )


    coalition_confidence_mae = float(

        np.mean(

            np.abs(

                coalition_learning_df[
                    "ex_ante_confidence"
                ]

                -

                coalition_learning_df[
                    "outcome_score"
                ]

            )

        )

    )

else:

    committee_mae = np.nan

    coalition_confidence_mae = np.nan


print(
    "\n"
    + "=" * 90
)

print(
    "INSTITUTIONAL CALIBRATION"
)

print(
    "=" * 90
)


if not np.isnan(
    committee_mae
):

    print(
        f"Committee conviction MAE : "
        f"{committee_mae:.3f}"
    )


if not np.isnan(
    coalition_confidence_mae
):

    print(
        f"Coalition confidence MAE : "
        f"{coalition_confidence_mae:.3f}"
    )


# ------------------------------------------------------------
# 19. Build institutional memory records
#
# Memory contains facts the institution should remember,
# not merely narratives it found compelling.
# ------------------------------------------------------------

for record in coalition_learning:

    cid = record[
        "coalition_id"
    ]


    coalition = coalition_lookup.get(
        cid,
        {}
    )


    memo = memo_lookup.get(
        cid,
        {}
    )


    decision = decision_lookup.get(
        cid,
        {}
    )


    memory_entry = {

        "episode_day":
            EVENT_DAY,

        "coalition_id":
            cid,

        "coalition_name":
            record[
                "coalition_name"
            ],

        "members":
            coalition.get(
                "members",
                []
            ),

        "specialties":
            coalition.get(
                "member_specialties",
                []
            ),

        "thesis":
            memo.get(
                "thesis",
                ""
            ),

        "asset":
            record[
                "asset"
            ],

        "direction":
            record[
                "direction"
            ],

        "capital":
            record[
                "capital"
            ],

        "committee_decision":
            decision.get(
                "decision",
                ""
            ),

        "committee_conviction":
            record[
                "committee_conviction"
            ],

        "evidence_strength":
            record[
                "evidence_strength"
            ],

        "persuasive_quality":
            record[
                "persuasive_quality"
            ],

        "debate_quality":
            record[
                "debate_quality"
            ],

        "realized_return":
            record[
                "realized_return"
            ],

        "pnl":
            record[
                "pnl"
            ],

        "max_drawdown":
            record[
                "max_drawdown"
            ],

        "outcome_score":
            record[
                "outcome_score"
            ],

        "epistemic_quality":
            record[
                "epistemic_quality"
            ],

        "calibration_score":
            record[
                "calibration_score"
            ],

        "learning_score":
            record[
                "learning_score"
            ],

        "starting_regime":
            record[
                "starting_regime"
            ],

        "ending_regime":
            record[
                "ending_regime"
            ],

        "regime_changed":
            record[
                "regime_changed"
            ],

        "falsification":
            memo.get(
                "falsification",
                ""
            ),

        "strongest_opposing_case":
            memo.get(
                "strongest_opposing_case",
                ""
            ),

        "minority_report":
            memo.get(
                "minority_report",
                ""
            )

    }


    history.append(
        memory_entry
    )


# Keep memory bounded
if len(
    history
) > MEMORY_LIMIT:

    history = history[
        -MEMORY_LIMIT:
    ]


# ------------------------------------------------------------
# 20. Institutional memory dataframe
# ------------------------------------------------------------

history_df = pd.DataFrame(
    history
)


print(
    "\n"
    + "=" * 90
)

print(
    "INSTITUTIONAL MEMORY"
)

print(
    "=" * 90
)


print(
    f"Memory records stored: "
    f"{len(history)}"
)


if len(
    history_df
):

    display(

        history_df.tail(
            min(
                20,
                len(history_df)
            )
        )

    )


# ------------------------------------------------------------
# 21. Specialty-level learning
#
# Which domains accumulated epistemic credibility?
# ------------------------------------------------------------

specialty_learning = (

    agent_learning_df

    .groupby(
        "specialty"
    )

    .agg(

        reputation=(
            "new_reputation",
            "mean"
        ),

        epistemic_score=(
            "new_epistemic",
            "mean"
        ),

        persuasion_score=(
            "new_persuasion",
            "mean"
        ),

        influence=(
            "new_influence",
            "mean"
        ),

        reputation_change=(
            "reputation_change",
            "mean"
        ),

        evaluated_coalitions=(
            "coalitions_evaluated",
            "sum"
        )

    )

    .reset_index()

    .sort_values(

        "epistemic_score",

        ascending=False

    )

)


print(
    "\nSPECIALTY-LEVEL LEARNING"
)


display(
    specialty_learning
)


# ------------------------------------------------------------
# 22. Institution-wide epistemic diagnostics
# ------------------------------------------------------------

mean_reputation_before = float(

    np.mean(

        [
            agent_before[
                a.id
            ][
                "reputation"
            ]

            for a in agents
        ]

    )

)


mean_reputation_after = float(

    np.mean(

        [
            a.reputation

            for a in agents
        ]

    )

)


mean_epistemic_after = float(

    np.mean(

        [
            a.epistemic_score

            for a in agents
        ]

    )

)


mean_persuasion_after = float(

    np.mean(

        [
            a.persuasion_score

            for a in agents
        ]

    )

)


mean_influence_after = float(

    np.mean(

        [
            a.influence

            for a in agents
        ]

    )

)


print(
    "\n"
    + "=" * 90
)

print(
    "INSTITUTION-WIDE LEARNING"
)

print(
    "=" * 90
)


print(
    f"Mean reputation before : "
    f"{mean_reputation_before:.3f}"
)


print(
    f"Mean reputation after  : "
    f"{mean_reputation_after:.3f}"
)


print(
    f"Mean epistemic score   : "
    f"{mean_epistemic_after:.3f}"
)


print(
    f"Mean persuasion score  : "
    f"{mean_persuasion_after:.3f}"
)


print(
    f"Mean influence         : "
    f"{mean_influence_after:.3f}"
)


# ------------------------------------------------------------
# 23. Top agents after learning
# ------------------------------------------------------------

agent_state_df = pd.DataFrame([

    {

        "agent_id":
            a.id,

        "specialty":
            a.specialty,

        "reputation":
            float(
                a.reputation
            ),

        "epistemic_score":
            float(
                a.epistemic_score
            ),

        "persuasion_score":
            float(
                a.persuasion_score
            ),

        "influence":
            float(
                a.influence
            ),

        "coalitions_evaluated":
            len(
                agent_signals[
                    a.id
                ]
            )

    }

    for a in agents

])


print(
    "\nTOP AGENTS BY REPUTATION"
)


display(

    agent_state_df.sort_values(

        "reputation",

        ascending=False

    ).head(15)

)


print(
    "\nTOP AGENTS BY EPISTEMIC SCORE"
)


display(

    agent_state_df.sort_values(

        "epistemic_score",

        ascending=False

    ).head(15)

)


print(
    "\nTOP AGENTS BY INFLUENCE"
)


display(

    agent_state_df.sort_values(

        "influence",

        ascending=False

    ).head(15)

)


# ------------------------------------------------------------
# 24. Learning summary for downstream cells
# ------------------------------------------------------------

learning_summary = {

    "episode_day":
        EVENT_DAY,

    "institutional_return":
        float(
            performance_record[
                "institutional_return"
            ]
        ),

    "institutional_max_drawdown":
        float(
            performance_record[
                "max_drawdown"
            ]
        ),

    "coalitions_evaluated":
        len(
            coalition_learning_df
        ),

    "agents_evaluated":
        int(

            (
                agent_learning_df[
                    "coalitions_evaluated"
                ]

                > 0

            ).sum()

        ),

    "mean_reputation_before":
        mean_reputation_before,

    "mean_reputation_after":
        mean_reputation_after,

    "mean_epistemic_score":
        mean_epistemic_after,

    "mean_persuasion_score":
        mean_persuasion_after,

    "mean_influence":
        mean_influence_after,

    "committee_calibration_error":
        committee_mae,

    "coalition_calibration_error":
        coalition_confidence_mae,

    "memory_records":
        len(
            history
        )

}


# ------------------------------------------------------------
# 25. Final institutional statement
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 90
)

print(
    "THE INSTITUTION HAS LEARNED"
)

print(
    "=" * 90
)


print(
    "Reality has now changed the internal social structure "
    "of the artificial financial institution."
)


print(
    "\nAgents who participated in epistemically successful "
    "coalitions gained credibility. Agents associated with "
    "poorly calibrated reasoning lost relative standing."
)


print(
    "\nPersuasive ability remains separately recorded and cannot "
    "by itself create epistemic authority."
)


print(
    "\nThe institution now carries forward a structured memory "
    "of theses, dissent, evidence, committee judgments, market "
    "regimes, realized returns, drawdowns, and falsification."
)


print(
    "\nCell 14 can therefore ask the next evolutionary question:"
)


print(
    "\nWhich coalitions should survive, dissolve, merge, mutate, "
    "or attract different agents after the institution has "
    "learned from reality?"
)

COALITION LEARNING AFTER REALITY


,coalition_id,coalition_name,asset,direction,capital,realized_return,pnl,max_drawdown,outcome_score,evidence_strength,...,calibration_score,committee_conviction,committee_calibration,eloquence_gap,eloquence_failure,epistemic_quality,learning_score,starting_regime,ending_regime,regime_changed
0,C01,Emergent Coalition 1,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
1,C02,Emergent Coalition 2,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
2,C03,Emergent Coalition 3,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
3,C04,Emergent Coalition 4,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
4,C05,Emergent Coalition 5,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
5,C06,Emergent Coalition 6,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
6,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
7,C08,Emergent Coalition 8,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
8,C09,Emergent Coalition 9,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True
9,C10,Emergent Coalition 10,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.420933,0.527625,...,0.878433,0.55275,0.868183,0.047375,0.007492,0.464801,0.63236,Calm Growth,Financial Stress,True



AGENT REPUTATION AND EPISTEMIC UPDATES


,agent_id,specialty,coalitions_evaluated,old_reputation,new_reputation,reputation_change,old_epistemic,new_epistemic,old_persuasion,new_persuasion,old_influence,new_influence,avg_outcome_score,avg_learning_score,avg_calibration,avg_eloquence_failure
0,0,Global Macro,3,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
1,1,Rates,3,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
2,2,Sovereign Credit,5,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
6,6,Public Equity Fundamental,5,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
13,13,Volatility,5,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
18,18,Distressed Debt,5,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
24,24,Corporate Restructuring,5,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
37,37,Infrastructure Finance,1,0.639012,0.621648,-0.017364,0.602891,0.568368,0.515,0.5210,0.520852,0.530909,0.420933,0.63236,0.878433,0.007492
5,5,Bank Credit,1,0.500000,0.510439,0.010439,0.500000,0.491200,0.500,0.5075,0.500000,0.500490,0.420933,0.63236,0.878433,0.007492
4,4,High Yield,1,0.500000,0.510439,0.010439,0.500000,0.491200,0.500,0.5075,0.500000,0.500490,0.420933,0.63236,0.878433,0.007492



LARGEST REPUTATION GAINS


...


TOP AGENTS BY EPISTEMIC SCORE


,agent_id,specialty,reputation,epistemic_score,persuasion_score,influence,coalitions_evaluated
0,0,Global Macro,0.621648,0.568368,0.521,0.530909,3
1,1,Rates,0.621648,0.568368,0.521,0.530909,3
2,2,Sovereign Credit,0.621648,0.568368,0.521,0.530909,5
6,6,Public Equity Fundamental,0.621648,0.568368,0.521,0.530909,5
13,13,Volatility,0.621648,0.568368,0.521,0.530909,5
18,18,Distressed Debt,0.621648,0.568368,0.521,0.530909,5
24,24,Corporate Restructuring,0.621648,0.568368,0.521,0.530909,5
37,37,Infrastructure Finance,0.621648,0.568368,0.521,0.530909,1
38,38,Power Markets,0.500000,0.500000,0.500,0.500000,0
36,36,Stablecoins,0.500000,0.500000,0.500,0.500000,0



TOP AGENTS BY INFLUENCE


,agent_id,specialty,reputation,epistemic_score,persuasion_score,influence,coalitions_evaluated
0,0,Global Macro,0.621648,0.568368,0.5210,0.530909,3
1,1,Rates,0.621648,0.568368,0.5210,0.530909,3
2,2,Sovereign Credit,0.621648,0.568368,0.5210,0.530909,5
6,6,Public Equity Fundamental,0.621648,0.568368,0.5210,0.530909,5
13,13,Volatility,0.621648,0.568368,0.5210,0.530909,5
18,18,Distressed Debt,0.621648,0.568368,0.5210,0.530909,5
24,24,Corporate Restructuring,0.621648,0.568368,0.5210,0.530909,5
37,37,Infrastructure Finance,0.621648,0.568368,0.5210,0.530909,1
5,5,Bank Credit,0.510439,0.491200,0.5075,0.500490,1
4,4,High Yield,0.510439,0.491200,0.5075,0.500490,1



THE INSTITUTION HAS LEARNED
Reality has now changed the internal social structure of the artificial financial institution.

Agents who participated in epistemically successful coalitions gained credibility. Agents associated with poorly calibrated reasoning lost relative standing.

Persuasive ability remains separately recorded and cannot by itself create epistemic authority.

The institution now carries forward a structured memory of theses, dissent, evidence, committee judgments, market regimes, realized returns, drawdowns, and falsification.

Cell 14 can therefore ask the next evolutionary question:

Which coalitions should survive, dissolve, merge, mutate, or attract different agents after the institution has learned from reality?


## CODE UNIT 14 — Migration, mutation, merger and dissolution

The defining feature of the capstone is that organizational structure itself can change. This unit reviews each coalition after outcomes arrive. Coalitions may persist, dissolve, or require mutation when the environment changes.

Dissolution returns agents to the population and capital to the center. The coalition disappears as an organization, but its memory remains. This creates organizational mortality without informational amnesia.

Mutation is especially interesting. A Financial Stress coalition may become Distressed Opportunities after the acute crisis passes. An inflation-defense group may become a duration-opportunity coalition after policy tightening succeeds. The same people can therefore reorganize around a new thesis rather than merely preserving yesterday’s department.

In a fuller multi-period run, this cell would also permit recruitment, member migration, coalition splitting, and merger when two groups discover overlapping mandates. The notebook records these transitions so that the final analysis can reconstruct the changing institutional map.

This is the point at which the swarm becomes genuinely autonomous in an organizational sense: it is not only choosing actions; it is changing the structure through which future actions will be chosen.

In [30]:
# ============================================================
# CODE UNIT 14 — ORGANIZATIONAL EVOLUTION:
#                PERSISTENCE, MUTATION, MERGER,
#                DISSOLUTION AND AGENT MIGRATION
# ============================================================
#
# PURPOSE
# -------
# Cells 12 and 13 changed what the institution knows.
#
# Cell 14 changes what the institution IS.
#
# The institution now uses realized outcomes, epistemic learning,
# calibration, agent reputation, and environmental change to
# redesign its own temporary organizational structure.
#
# Coalitions are not permanent departments.
#
# They may:
#
#       PERSIST
#       MUTATE
#       RECRUIT
#       RELEASE
#       MERGE
#       SPLIT
#       DISSOLVE
#
# Agents may migrate between coalitions.
#
# Organizational evolution is therefore endogenous:
#
#       REALITY
#          ↓
#       LEARNING
#          ↓
#       REPUTATION
#          ↓
#       ORGANIZATIONAL CHANGE
#          ↓
#       NEW INSTITUTIONAL STRUCTURE
#
# ============================================================


# ------------------------------------------------------------
# 1. Evolution parameters
# ------------------------------------------------------------

MIN_COALITION_SIZE = 3
MAX_COALITION_SIZE = 8

PERSIST_THRESHOLD = 0.66
MUTATE_THRESHOLD = 0.48
DISSOLVE_THRESHOLD = 0.34

RECRUIT_REPUTATION_THRESHOLD = 0.58
RELEASE_REPUTATION_THRESHOLD = 0.35

MERGER_SIMILARITY_THRESHOLD = 0.55
MERGER_COMPLEMENTARITY_THRESHOLD = 0.25

MAX_RECRUITS_PER_COALITION = 2

MIN_AGENT_REPUTATION = 0.05

EVOLUTION_SEED = SEED + 14

rng_evolution = np.random.default_rng(
    EVOLUTION_SEED
)


# ------------------------------------------------------------
# 2. Agent lookup
# ------------------------------------------------------------

agent_lookup = {
    a.id: a
    for a in agents
}


# ------------------------------------------------------------
# 3. Coalition learning lookup
# ------------------------------------------------------------

coalition_learning_lookup = {

    row["coalition_id"]:
        row.to_dict()

    for _, row
    in coalition_learning_df.iterrows()

}


# ------------------------------------------------------------
# 4. Helper: specialty family
#
# We reuse the intellectual families introduced earlier,
# but keep a safe fallback in case Cell 6's family()
# function is unavailable.
# ------------------------------------------------------------

def specialty_family(
    specialty
):

    try:

        return family(
            specialty
        )

    except Exception:

        s = specialty.lower()

        if any(
            k in s
            for k in [
                "credit",
                "distressed",
                "bank"
            ]
        ):

            return "credit"

        if any(
            k in s
            for k in [
                "macro",
                "rates",
                "sovereign",
                "fx",
                "commodities"
            ]
        ):

            return "macro"

        if any(
            k in s
            for k in [
                "equity",
                "value",
                "growth",
                "quality",
                "small cap"
            ]
        ):

            return "equity"

        if any(
            k in s
            for k in [
                "venture",
                "private",
                "m&a",
                "restructuring",
                "capital structure"
            ]
        ):

            return "private"

        if any(
            k in s
            for k in [
                "risk",
                "treasury",
                "liquidity"
            ]
        ):

            return "risk"

        if any(
            k in s
            for k in [
                "crypto",
                "digital",
                "stablecoin",
                "payments",
                "fintech"
            ]
        ):

            return "digital"

        if any(
            k in s
            for k in [
                "ai ",
                "semiconductor",
                "power",
                "infrastructure"
            ]
        ):

            return "tech"

        if any(
            k in s
            for k in [
                "quant",
                "econometric",
                "systematic",
                "derivative",
                "volatility",
                "option",
                "microstructure"
            ]
        ):

            return "quant"

        return "outside"


# ------------------------------------------------------------
# 5. Agent institutional quality
#
# Recruitment should not depend on reputation alone.
#
# We combine:
#
#   reputation
#   epistemic score
#   influence
#
# Persuasion is intentionally NOT given independent weight here.
# ------------------------------------------------------------

def agent_quality(
    agent
):

    return float(

        0.45
        *
        agent.reputation

        +

        0.40
        *
        agent.epistemic_score

        +

        0.15
        *
        agent.influence

    )


# ------------------------------------------------------------
# 6. Coalition diversity
# ------------------------------------------------------------

def coalition_diversity(
    member_ids
):

    if len(
        member_ids
    ) == 0:

        return 0.0


    families = [

        specialty_family(
            agent_lookup[
                aid
            ].specialty
        )

        for aid
        in member_ids

        if aid
        in agent_lookup

    ]


    if len(
        families
    ) == 0:

        return 0.0


    return float(

        len(
            set(
                families
            )
        )

        /
        len(
            families
        )

    )


# ------------------------------------------------------------
# 7. Coalition member quality
# ------------------------------------------------------------

def coalition_member_quality(
    member_ids
):

    values = [

        agent_quality(
            agent_lookup[
                aid
            ]
        )

        for aid
        in member_ids

        if aid
        in agent_lookup

    ]


    if len(
        values
    ) == 0:

        return 0.0


    return float(
        np.mean(
            values
        )
    )


# ------------------------------------------------------------
# 8. Coalition evolutionary fitness
#
# IMPORTANT:
#
# Fitness is not simply realized return.
#
# A coalition may lose money yet remain epistemically valuable
# if:
#
#   - its evidence was strong,
#   - its debate was rigorous,
#   - it was well calibrated,
#   - the regime changed unexpectedly.
#
# Likewise, a lucky profitable coalition should not automatically
# become institutionally dominant.
# ------------------------------------------------------------

def coalition_fitness(
    coalition
):

    cid = coalition[
        "id"
    ]


    learning = coalition_learning_lookup.get(
        cid,
        {}
    )


    members = coalition.get(
        "members",
        []
    )


    learning_score = float(
        learning.get(
            "learning_score",
            0.50
        )
    )


    epistemic_quality = float(
        learning.get(
            "epistemic_quality",
            0.50
        )
    )


    calibration = float(
        learning.get(
            "calibration_score",
            0.50
        )
    )


    outcome = float(
        learning.get(
            "outcome_score",
            0.50
        )
    )


    member_quality = (
        coalition_member_quality(
            members
        )
    )


    diversity = (
        coalition_diversity(
            members
        )
    )


    regime_changed_local = bool(
        learning.get(
            "regime_changed",
            False
        )
    )


    # --------------------------------------------------------
    # Environmental-change allowance
    #
    # A regime transition slightly reduces the penalty attached
    # to poor realized outcomes because environmental instability
    # makes inference more difficult.
    # --------------------------------------------------------

    regime_allowance = (

        0.04

        if regime_changed_local

        else 0.0

    )


    fitness = (

        0.28
        *
        learning_score

        +

        0.22
        *
        epistemic_quality

        +

        0.16
        *
        calibration

        +

        0.14
        *
        outcome

        +

        0.12
        *
        member_quality

        +

        0.08
        *
        diversity

        +

        regime_allowance

    )


    return float(
        np.clip(
            fitness,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 9. Determine base evolutionary state
# ------------------------------------------------------------

evolution_records = []


for c in coalitions:

    cid = c[
        "id"
    ]


    learning = coalition_learning_lookup.get(
        cid
    )


    funded = (
        cid
        in episode_lookup
    )


    fitness = coalition_fitness(
        c
    )


    if learning is None:

        # ----------------------------------------------------
        # Coalition never received enough realized evidence.
        #
        # We should not interpret absence of funding as proof
        # that the coalition is epistemically worthless.
        # ----------------------------------------------------

        base_action = (
            "REVIEW"
        )

        reason = (
            "Insufficient realized evidence for "
            "strong evolutionary judgment."
        )


    else:

        outcome = float(
            learning[
                "outcome_score"
            ]
        )


        learning_score = float(
            learning[
                "learning_score"
            ]
        )


        regime_changed_local = bool(
            learning[
                "regime_changed"
            ]
        )


        # ----------------------------------------------------
        # Strong coalition
        # ----------------------------------------------------

        if (
            fitness
            >=
            PERSIST_THRESHOLD
        ):

            base_action = (
                "PERSIST"
            )

            reason = (
                "Strong combined epistemic, calibration, "
                "performance and membership fitness."
            )


        # ----------------------------------------------------
        # Weak coalition
        # ----------------------------------------------------

        elif (
            fitness
            <
            DISSOLVE_THRESHOLD
        ):

            base_action = (
                "DISSOLVE"
            )

            reason = (
                "Coalition fitness fell below the "
                "institutional survival threshold."
            )


        # ----------------------------------------------------
        # Regime changed:
        # prefer mutation over premature extinction
        # ----------------------------------------------------

        elif regime_changed_local:

            base_action = (
                "MUTATE"
            )

            reason = (
                "The environment changed materially; "
                "the coalition should adapt before "
                "being judged obsolete."
            )


        # ----------------------------------------------------
        # Intermediate fitness
        # ----------------------------------------------------

        elif (
            fitness
            >=
            MUTATE_THRESHOLD
        ):

            base_action = (
                "MUTATE"
            )

            reason = (
                "Coalition remains potentially useful "
                "but requires organizational adaptation."
            )


        else:

            base_action = (
                "DISSOLVE"
            )

            reason = (
                "Weak learning and outcome signals do not "
                "justify preserving the current structure."
            )


    evolution_records.append({

        "coalition_id":
            cid,

        "coalition_name":
            c.get(
                "name",
                cid
            ),

        "funded":
            funded,

        "member_count_before":
            len(
                c.get(
                    "members",
                    []
                )
            ),

        "fitness":
            fitness,

        "base_action":
            base_action,

        "reason":
            reason

    })


evolution_df = pd.DataFrame(
    evolution_records
)


print(
    "=" * 92
)

print(
    "INITIAL ORGANIZATIONAL EVOLUTION ASSESSMENT"
)

print(
    "=" * 92
)


display(

    evolution_df.sort_values(

        "fitness",

        ascending=False

    )

)


# ------------------------------------------------------------
# 10. Identify weak members inside surviving coalitions
#
# Mutation may involve releasing agents whose accumulated
# epistemic credibility is substantially weaker than the
# coalition's other members.
#
# We deliberately use a conservative rule.
# ------------------------------------------------------------

def identify_release_candidates(
    coalition
):

    members = coalition.get(
        "members",
        []
    )


    if len(
        members
    ) <= MIN_COALITION_SIZE:

        return []


    qualities = {

        aid:
            agent_quality(
                agent_lookup[
                    aid
                ]
            )

        for aid
        in members

        if aid
        in agent_lookup

    }


    if len(
        qualities
    ) == 0:

        return []


    coalition_mean = float(
        np.mean(
            list(
                qualities.values()
            )
        )
    )


    candidates = []


    for aid, quality in qualities.items():

        agent = agent_lookup[
            aid
        ]


        if (

            agent.reputation
            <
            RELEASE_REPUTATION_THRESHOLD

            and

            quality
            <
            coalition_mean
            -
            0.10

        ):

            candidates.append(
                aid
            )


    # Never shrink below minimum viable size
    max_release = max(

        0,

        len(
            members
        )
        -
        MIN_COALITION_SIZE

    )


    candidates = sorted(

        candidates,

        key=lambda aid:
            qualities[
                aid
            ]

    )


    return candidates[
        :max_release
    ]


# ------------------------------------------------------------
# 11. Recruitment logic
#
# A coalition does not simply recruit the highest-reputation
# agent.
#
# It prefers agents who:
#
#   1. have strong institutional quality,
#   2. add a new specialty family,
#   3. are not already members,
#   4. remain within coalition size constraints.
# ------------------------------------------------------------

def recruitment_candidates(
    coalition,
    current_members
):

    current_families = {

        specialty_family(
            agent_lookup[
                aid
            ].specialty
        )

        for aid
        in current_members

        if aid
        in agent_lookup

    }


    candidates = []


    for a in agents:

        if a.id in current_members:

            continue


        quality = agent_quality(
            a
        )


        if (
            a.reputation
            <
            RECRUIT_REPUTATION_THRESHOLD
        ):

            continue


        candidate_family = (
            specialty_family(
                a.specialty
            )
        )


        diversity_bonus = (

            0.20

            if candidate_family
            not in current_families

            else 0.0

        )


        epistemic_bonus = (

            0.10
            *
            a.epistemic_score

        )


        recruitment_score = (

            quality

            +
            diversity_bonus

            +
            epistemic_bonus

        )


        candidates.append({

            "agent_id":
                a.id,

            "specialty":
                a.specialty,

            "family":
                candidate_family,

            "quality":
                quality,

            "recruitment_score":
                recruitment_score

        })


    return sorted(

        candidates,

        key=lambda x:
            x[
                "recruitment_score"
            ],

        reverse=True

    )


# ------------------------------------------------------------
# 12. Apply persistence, mutation, recruitment and release
# ------------------------------------------------------------

next_generation_coalitions = []

migration_records = []

dissolved_coalitions = []


for c in coalitions:

    cid = c[
        "id"
    ]


    assessment = evolution_df[

        evolution_df[
            "coalition_id"
        ]
        ==
        cid

    ].iloc[0]


    action = assessment[
        "base_action"
    ]


    fitness = float(
        assessment[
            "fitness"
        ]
    )


    # Work on a copy
    evolved = dict(
        c
    )


    evolved[
        "previous_id"
    ] = cid


    evolved[
        "evolution_fitness"
    ] = fitness


    evolved[
        "evolution_action"
    ] = action


    evolved[
        "generation"
    ] = int(
        c.get(
            "generation",
            1
        )
    ) + 1


    members = list(
        c.get(
            "members",
            []
        )
    )


    # --------------------------------------------------------
    # DISSOLUTION
    # --------------------------------------------------------

    if action == "DISSOLVE":

        evolved[
            "status"
        ] = "DISSOLVED"


        dissolved_coalitions.append(
            evolved
        )


        for aid in members:

            migration_records.append({

                "agent_id":
                    aid,

                "specialty":
                    agent_lookup[
                        aid
                    ].specialty,

                "from_coalition":
                    cid,

                "to_coalition":
                    None,

                "movement":
                    "RELEASED_BY_DISSOLUTION",

                "reason":
                    "Coalition dissolved after "
                    "institutional learning."

            })


        continue


    # --------------------------------------------------------
    # PERSISTENCE / REVIEW / MUTATION
    # --------------------------------------------------------

    evolved[
        "status"
    ] = "ACTIVE"


    released = []


    if action == "MUTATE":

        released = (
            identify_release_candidates(
                c
            )
        )


        for aid in released:

            if aid in members:

                members.remove(
                    aid
                )


                migration_records.append({

                    "agent_id":
                        aid,

                    "specialty":
                        agent_lookup[
                            aid
                        ].specialty,

                    "from_coalition":
                        cid,

                    "to_coalition":
                        None,

                    "movement":
                        "RELEASED",

                    "reason":
                        "Low relative epistemic/reputation "
                        "fitness within mutating coalition."

                })


    # --------------------------------------------------------
    # Recruitment
    #
    # MUTATE coalitions actively recruit.
    # REVIEW coalitions may recruit one agent.
    # Strong PERSIST coalitions retain their structure unless
    # they are below minimum desired breadth.
    # --------------------------------------------------------

    recruit_limit = 0


    if action == "MUTATE":

        recruit_limit = (
            MAX_RECRUITS_PER_COALITION
        )


    elif action == "REVIEW":

        recruit_limit = 1


    elif (

        action == "PERSIST"

        and

        len(
            members
        )
        <
        MIN_COALITION_SIZE + 1

    ):

        recruit_limit = 1


    recruit_limit = min(

        recruit_limit,

        MAX_COALITION_SIZE
        -
        len(
            members
        )

    )


    if recruit_limit > 0:

        candidates = recruitment_candidates(

            c,

            members

        )


        selected = candidates[
            :recruit_limit
        ]


        for candidate in selected:

            aid = candidate[
                "agent_id"
            ]


            members.append(
                aid
            )


            migration_records.append({

                "agent_id":
                    aid,

                "specialty":
                    candidate[
                        "specialty"
                    ],

                "from_coalition":
                    None,

                "to_coalition":
                    cid,

                "movement":
                    "RECRUITED",

                "reason":
                    "High institutional quality and/or "
                    "complementary expertise."

            })


    evolved[
        "members"
    ] = members


    evolved[
        "member_specialties"
    ] = [

        agent_lookup[
            aid
        ].specialty

        for aid in members

        if aid in agent_lookup

    ]


    evolved[
        "released_agents"
    ] = released


    evolved[
        "recruited_agents"
    ] = [

        r[
            "agent_id"
        ]

        for r in migration_records

        if (
            r[
                "to_coalition"
            ]
            ==
            cid

            and

            r[
                "movement"
            ]
            ==
            "RECRUITED"
        )

    ]


    next_generation_coalitions.append(
        evolved
    )


# ------------------------------------------------------------
# 13. Coalition similarity for possible mergers
#
# Coalitions may merge when they:
#
#   - pursue related financial expressions,
#   - have overlapping members or expertise,
#   - but retain enough complementary knowledge
#     to make the merger informative.
# ------------------------------------------------------------

def coalition_similarity(
    c1,
    c2
):

    members_1 = set(
        c1.get(
            "members",
            []
        )
    )

    members_2 = set(
        c2.get(
            "members",
            []
        )
    )


    union_members = (
        members_1
        |
        members_2
    )


    if len(
        union_members
    ):

        member_overlap = (

            len(
                members_1
                &
                members_2
            )

            /
            len(
                union_members
            )

        )

    else:

        member_overlap = 0.0


    families_1 = {

        specialty_family(
            agent_lookup[
                aid
            ].specialty
        )

        for aid in members_1

        if aid in agent_lookup

    }


    families_2 = {

        specialty_family(
            agent_lookup[
                aid
            ].specialty
        )

        for aid in members_2

        if aid in agent_lookup

    }


    family_union = (
        families_1
        |
        families_2
    )


    if len(
        family_union
    ):

        family_overlap = (

            len(
                families_1
                &
                families_2
            )

            /
            len(
                family_union
            )

        )

    else:

        family_overlap = 0.0


    # --------------------------------------------------------
    # Compare investment expression using the latest memo
    # --------------------------------------------------------

    m1 = memo_lookup.get(
        c1.get(
            "previous_id",
            c1[
                "id"
            ]
        ),
        {}
    )


    m2 = memo_lookup.get(
        c2.get(
            "previous_id",
            c2[
                "id"
            ]
        ),
        {}
    )


    same_asset = (

        1.0

        if m1.get(
            "asset"
        )
        ==
        m2.get(
            "asset"
        )

        else 0.0

    )


    same_direction = (

        1.0

        if m1.get(
            "direction"
        )
        ==
        m2.get(
            "direction"
        )

        else 0.0

    )


    similarity = (

        0.30
        *
        member_overlap

        +

        0.25
        *
        family_overlap

        +

        0.30
        *
        same_asset

        +

        0.15
        *
        same_direction

    )


    # Complementarity:
    # share some intellectual terrain but not all of it
    complementarity = (

        1.0

        -

        family_overlap

    )


    return (
        float(
            similarity
        ),
        float(
            complementarity
        )
    )


# ------------------------------------------------------------
# 14. Identify merger candidates
# ------------------------------------------------------------

merger_candidates = []


for i in range(
    len(
        next_generation_coalitions
    )
):

    for j in range(
        i + 1,
        len(
            next_generation_coalitions
        )
    ):

        c1 = (
            next_generation_coalitions[
                i
            ]
        )

        c2 = (
            next_generation_coalitions[
                j
            ]
        )


        similarity, complementarity = (
            coalition_similarity(
                c1,
                c2
            )
        )


        combined_members = set(
            c1[
                "members"
            ]
        ) | set(
            c2[
                "members"
            ]
        )


        size_feasible = (

            len(
                combined_members
            )
            <=
            MAX_COALITION_SIZE

        )


        if (

            similarity
            >=
            MERGER_SIMILARITY_THRESHOLD

            and

            complementarity
            >=
            MERGER_COMPLEMENTARITY_THRESHOLD

            and

            size_feasible

        ):

            merger_candidates.append({

                "coalition_1":
                    c1[
                        "id"
                    ],

                "coalition_2":
                    c2[
                        "id"
                    ],

                "similarity":
                    similarity,

                "complementarity":
                    complementarity,

                "combined_size":
                    len(
                        combined_members
                    )

            })


merger_candidates_df = pd.DataFrame(
    merger_candidates
)


print(
    "\n"
    + "=" * 92
)

print(
    "POTENTIAL COALITION MERGERS"
)

print(
    "=" * 92
)


if len(
    merger_candidates_df
):

    display(

        merger_candidates_df.sort_values(

            [
                "similarity",
                "complementarity"
            ],

            ascending=False

        )

    )

else:

    print(
        "No merger candidates passed the institutional thresholds."
    )


# ------------------------------------------------------------
# 15. Execute non-overlapping mergers
#
# We greedily select the strongest feasible merger pairs.
# A coalition can participate in at most one merger in this
# generation.
# ------------------------------------------------------------

merged_ids = set()

merged_coalitions = []


if len(
    merger_candidates
):

    ranked_mergers = sorted(

        merger_candidates,

        key=lambda x: (

            x[
                "similarity"
            ],

            x[
                "complementarity"
            ]

        ),

        reverse=True

    )


    coalition_by_id = {

        c[
            "id"
        ]:
            c

        for c in next_generation_coalitions

    }


    merger_number = 1


    for candidate in ranked_mergers:

        cid1 = candidate[
            "coalition_1"
        ]

        cid2 = candidate[
            "coalition_2"
        ]


        if (

            cid1
            in merged_ids

            or

            cid2
            in merged_ids

        ):

            continue


        c1 = coalition_by_id[
            cid1
        ]

        c2 = coalition_by_id[
            cid2
        ]


        combined_members = list(

            dict.fromkeys(

                c1[
                    "members"
                ]

                +

                c2[
                    "members"
                ]

            )

        )


        if (
            len(
                combined_members
            )
            >
            MAX_COALITION_SIZE
        ):

            continue


        new_id = (
            f"M{merger_number:02d}"
        )


        merger_number += 1


        merged_name = (

            f"{c1.get('name', cid1)} "
            f"+ {c2.get('name', cid2)}"
        )


        merged = {

            "id":
                new_id,

            "name":
                merged_name,

            "members":
                combined_members,

            "member_specialties":
                [

                    agent_lookup[
                        aid
                    ].specialty

                    for aid
                    in combined_members

                ],

            "status":
                "ACTIVE",

            "generation":
                max(

                    int(
                        c1.get(
                            "generation",
                            1
                        )
                    ),

                    int(
                        c2.get(
                            "generation",
                            1
                        )
                    )

                ),

            "evolution_action":
                "MERGED",

            "parent_coalitions":
                [
                    cid1,
                    cid2
                ],

            "evolution_fitness":
                float(

                    np.mean(

                        [
                            c1[
                                "evolution_fitness"
                            ],

                            c2[
                                "evolution_fitness"
                            ]
                        ]

                    )

                ),

            "shared_question":
                (
                    "Merged coalition formed from "
                    "related but complementary "
                    "institutional hypotheses."
                ),

            "shared_thesis":
                (
                    "To be re-negotiated by the merged "
                    "coalition in the next decision cycle."
                ),

            "formation_day":
                EVAL_END,

            "lifespan_days":
                100,

            "dissolution":
                (
                    "Dissolve if merged expertise fails "
                    "to generate incremental epistemic value."
                )

        }


        merged_coalitions.append(
            merged
        )


        merged_ids.add(
            cid1
        )

        merged_ids.add(
            cid2
        )


        for aid in combined_members:

            migration_records.append({

                "agent_id":
                    aid,

                "specialty":
                    agent_lookup[
                        aid
                    ].specialty,

                "from_coalition":
                    f"{cid1}/{cid2}",

                "to_coalition":
                    new_id,

                "movement":
                    "MERGER",

                "reason":
                    "Related financial thesis plus "
                    "complementary expertise."

            })


# ------------------------------------------------------------
# 16. Keep non-merged surviving coalitions
# ------------------------------------------------------------

surviving_unmerged = [

    c

    for c in next_generation_coalitions

    if c[
        "id"
    ]
    not in merged_ids

]


evolved_coalitions = (

    surviving_unmerged

    +

    merged_coalitions

)


# ------------------------------------------------------------
# 17. Detect possible coalition splits
#
# Very large coalitions with low internal epistemic cohesion
# may become too heterogeneous to deliberate effectively.
#
# We use member-quality dispersion as a simple proxy.
# ------------------------------------------------------------

split_records = []

post_split_coalitions = []


for c in evolved_coalitions:

    members = c.get(
        "members",
        []
    )


    qualities = [

        agent_quality(
            agent_lookup[
                aid
            ]
        )

        for aid
        in members

        if aid in agent_lookup

    ]


    quality_dispersion = (

        float(
            np.std(
                qualities
            )
        )

        if len(
            qualities
        )
        > 1

        else 0.0

    )


    should_split = (

        len(
            members
        )
        >=
        7

        and

        quality_dispersion
        >
        0.14

    )


    if not should_split:

        c[
            "quality_dispersion"
        ] = quality_dispersion

        post_split_coalitions.append(
            c
        )

        continue


    # --------------------------------------------------------
    # Split by specialty family into two broad groups.
    #
    # This is intentionally deterministic and pedagogical.
    # --------------------------------------------------------

    ranked_members = sorted(

        members,

        key=lambda aid: (

            specialty_family(
                agent_lookup[
                    aid
                ].specialty
            ),

            -agent_quality(
                agent_lookup[
                    aid
                ]
            )

        )

    )


    group_1 = ranked_members[
        ::2
    ]

    group_2 = ranked_members[
        1::2
    ]


    if (

        len(
            group_1
        )
        <
        MIN_COALITION_SIZE

        or

        len(
            group_2
        )
        <
        MIN_COALITION_SIZE

    ):

        c[
            "quality_dispersion"
        ] = quality_dispersion

        post_split_coalitions.append(
            c
        )

        continue


    parent_id = c[
        "id"
    ]


    for suffix, group in [

        ("A", group_1),

        ("B", group_2)

    ]:

        child_id = (
            f"{parent_id}_{suffix}"
        )


        child = dict(
            c
        )


        child[
            "id"
        ] = child_id


        child[
            "name"
        ] = (

            f"{c.get('name', parent_id)} "
            f"— Branch {suffix}"
        )


        child[
            "members"
        ] = group


        child[
            "member_specialties"
        ] = [

            agent_lookup[
                aid
            ].specialty

            for aid in group

        ]


        child[
            "parent_coalition"
        ] = parent_id


        child[
            "evolution_action"
        ] = "SPLIT"


        child[
            "quality_dispersion"
        ] = float(

            np.std(

                [
                    agent_quality(
                        agent_lookup[
                            aid
                        ]
                    )

                    for aid
                    in group
                ]

            )

        )


        post_split_coalitions.append(
            child
        )


    split_records.append({

        "parent_coalition":
            parent_id,

        "child_1":
            f"{parent_id}_A",

        "child_2":
            f"{parent_id}_B",

        "parent_size":
            len(
                members
            ),

        "quality_dispersion":
            quality_dispersion

    })


# ------------------------------------------------------------
# 18. Final next-generation coalition set
# ------------------------------------------------------------

evolved_coalitions = (
    post_split_coalitions
)


# ------------------------------------------------------------
# 19. Recompute final coalition metrics
# ------------------------------------------------------------

final_evolution_records = []


for c in evolved_coalitions:

    members = c.get(
        "members",
        []
    )


    final_evolution_records.append({

        "coalition_id":
            c[
                "id"
            ],

        "coalition_name":
            c.get(
                "name",
                c[
                    "id"
                ]
            ),

        "action":
            c.get(
                "evolution_action",
                "PERSIST"
            ),

        "generation":
            c.get(
                "generation",
                2
            ),

        "member_count":
            len(
                members
            ),

        "families":
            len(

                {

                    specialty_family(
                        agent_lookup[
                            aid
                        ].specialty
                    )

                    for aid
                    in members

                    if aid
                    in agent_lookup

                }

            ),

        "diversity":
            coalition_diversity(
                members
            ),

        "member_quality":
            coalition_member_quality(
                members
            ),

        "inherited_fitness":
            float(
                c.get(
                    "evolution_fitness",
                    0.50
                )
            ),

        "status":
            c.get(
                "status",
                "ACTIVE"
            )

    })


final_evolution_df = pd.DataFrame(
    final_evolution_records
)


# ------------------------------------------------------------
# 20. Agent migration dataframe
# ------------------------------------------------------------

migration_df = pd.DataFrame(
    migration_records
)


# ------------------------------------------------------------
# 21. Split dataframe
# ------------------------------------------------------------

split_df = pd.DataFrame(
    split_records
)


# ------------------------------------------------------------
# 22. Organizational census
# ------------------------------------------------------------

original_active = len(
    coalitions
)


final_active = len(
    evolved_coalitions
)


n_dissolved = len(
    dissolved_coalitions
)


n_merged = len(
    merged_coalitions
)


n_splits = len(
    split_records
)


n_recruited = sum(

    1

    for r
    in migration_records

    if r[
        "movement"
    ]
    ==
    "RECRUITED"

)


n_released = sum(

    1

    for r
    in migration_records

    if r[
        "movement"
    ]
    in [

        "RELEASED",

        "RELEASED_BY_DISSOLUTION"

    ]

)


# ------------------------------------------------------------
# 23. Display final organizational structure
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 92
)

print(
    "NEXT-GENERATION ORGANIZATIONAL STRUCTURE"
)

print(
    "=" * 92
)


if len(
    final_evolution_df
):

    display(

        final_evolution_df.sort_values(

            "inherited_fitness",

            ascending=False

        )

    )


# ------------------------------------------------------------
# 24. Display migration
# ------------------------------------------------------------

print(
    "\nAGENT MIGRATION"
)


if len(
    migration_df
):

    display(
        migration_df
    )

else:

    print(
        "No agent migration occurred in this generation."
    )


# ------------------------------------------------------------
# 25. Display dissolution
# ------------------------------------------------------------

print(
    "\nDISSOLVED COALITIONS"
)


if len(
    dissolved_coalitions
):

    dissolved_df = pd.DataFrame([

        {

            "coalition_id":
                c[
                    "id"
                ],

            "coalition_name":
                c.get(
                    "name",
                    c[
                        "id"
                    ]
                ),

            "fitness":
                c.get(
                    "evolution_fitness",
                    np.nan
                ),

            "members":
                len(
                    c.get(
                        "members",
                        []
                    )
                )

        }

        for c
        in dissolved_coalitions

    ])


    display(
        dissolved_df
    )

else:

    print(
        "No coalition dissolved in this generation."
    )


# ------------------------------------------------------------
# 26. Display merger outcomes
# ------------------------------------------------------------

print(
    "\nEXECUTED MERGERS"
)


if len(
    merged_coalitions
):

    merger_execution_df = pd.DataFrame([

        {

            "new_coalition":
                c[
                    "id"
                ],

            "name":
                c[
                    "name"
                ],

            "parents":
                ", ".join(
                    c[
                        "parent_coalitions"
                    ]
                ),

            "members":
                len(
                    c[
                        "members"
                    ]
                ),

            "fitness":
                c[
                    "evolution_fitness"
                ]

        }

        for c
        in merged_coalitions

    ])


    display(
        merger_execution_df
    )

else:

    print(
        "No mergers executed."
    )


# ------------------------------------------------------------
# 27. Display splits
# ------------------------------------------------------------

print(
    "\nEXECUTED SPLITS"
)


if len(
    split_df
):

    display(
        split_df
    )

else:

    print(
        "No coalition splits were required."
    )


# ------------------------------------------------------------
# 28. Compare old and new institutional architecture
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 92
)

print(
    "ORGANIZATIONAL EVOLUTION SUMMARY"
)

print(
    "=" * 92
)


print(
    f"Coalitions before evolution : "
    f"{original_active}"
)


print(
    f"Coalitions after evolution  : "
    f"{final_active}"
)


print(
    f"Coalitions dissolved        : "
    f"{n_dissolved}"
)


print(
    f"Mergers executed            : "
    f"{n_merged}"
)


print(
    f"Splits executed             : "
    f"{n_splits}"
)


print(
    f"Agents recruited            : "
    f"{n_recruited}"
)


print(
    f"Agents released             : "
    f"{n_released}"
)


# ------------------------------------------------------------
# 29. Agent participation in the evolved institution
# ------------------------------------------------------------

participation = {

    a.id: 0

    for a in agents

}


for c in evolved_coalitions:

    for aid in c.get(
        "members",
        []
    ):

        if aid in participation:

            participation[
                aid
            ] += 1


participation_df = pd.DataFrame([

    {

        "agent_id":
            a.id,

        "specialty":
            a.specialty,

        "reputation":
            float(
                a.reputation
            ),

        "epistemic_score":
            float(
                a.epistemic_score
            ),

        "influence":
            float(
                a.influence
            ),

        "active_coalitions":
            participation[
                a.id
            ]

    }

    for a in agents

])


print(
    "\nAGENT PARTICIPATION AFTER EVOLUTION"
)


display(

    participation_df.sort_values(

        [
            "active_coalitions",
            "reputation"
        ],

        ascending=[
            False,
            False
        ]

    )

)


# ------------------------------------------------------------
# 30. Organizational concentration diagnostic
# ------------------------------------------------------------

if len(
    evolved_coalitions
):

    coalition_sizes = np.array(

        [

            len(
                c.get(
                    "members",
                    []
                )
            )

            for c
            in evolved_coalitions

        ],

        dtype=float

    )


    avg_coalition_size = float(
        coalition_sizes.mean()
    )


    max_coalition_size_realized = int(
        coalition_sizes.max()
    )


    multi_coalition_agents = int(

        sum(

            1

            for count
            in participation.values()

            if count > 1

        )

    )


    active_agents = int(

        sum(

            1

            for count
            in participation.values()

            if count > 0

        )

    )

else:

    avg_coalition_size = 0.0

    max_coalition_size_realized = 0

    multi_coalition_agents = 0

    active_agents = 0


print(
    "\nORGANIZATIONAL COMPLEXITY"
)


print(
    f"Active agents              : "
    f"{active_agents}/{len(agents)}"
)


print(
    f"Multi-coalition agents     : "
    f"{multi_coalition_agents}"
)


print(
    f"Average coalition size     : "
    f"{avg_coalition_size:.2f}"
)


print(
    f"Largest coalition          : "
    f"{max_coalition_size_realized}"
)


# ------------------------------------------------------------
# 31. Evolution event log
#
# This becomes part of the institution's biography.
# ------------------------------------------------------------

evolution_event = {

    "event_day":
        EVAL_END,

    "type":
        "ORGANIZATIONAL_EVOLUTION",

    "coalitions_before":
        original_active,

    "coalitions_after":
        final_active,

    "dissolved":
        n_dissolved,

    "mergers":
        n_merged,

    "splits":
        n_splits,

    "agents_recruited":
        n_recruited,

    "agents_released":
        n_released,

    "active_agents":
        active_agents,

    "multi_coalition_agents":
        multi_coalition_agents,

    "average_coalition_size":
        avg_coalition_size

}


# Institutional memory should remember not only investments,
# but also changes in its own organization.

history.append(
    evolution_event
)


if len(
    history
) > MEMORY_LIMIT:

    history = history[
        -MEMORY_LIMIT:
    ]


# ------------------------------------------------------------
# 32. Expose next-generation coalitions
#
# We preserve the original `coalitions` object for historical
# analysis and expose the evolved organization separately.
#
# A later multi-cycle extension could simply execute:
#
#       coalitions = evolved_coalitions
#
# before beginning the next decision cycle.
# ------------------------------------------------------------

next_generation = (
    evolved_coalitions
)


# ------------------------------------------------------------
# 33. Final institutional statement
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 92
)

print(
    "THE ORGANIZATION HAS CHANGED"
)

print(
    "=" * 92
)


print(
    "The institution did not merely update forecasts after "
    "observing market outcomes."
)


print(
    "\nIt changed its own internal architecture."
)


print(
    "\nSome coalitions survived because their combination of "
    "evidence, calibration, realized outcomes and membership "
    "quality remained strong."
)


print(
    "\nOthers mutated by releasing weak contributors and "
    "recruiting agents with stronger epistemic records or "
    "complementary expertise."
)


print(
    "\nRelated coalitions were allowed to merge when their "
    "financial questions overlapped but their knowledge remained "
    "sufficiently complementary."
)


print(
    "\nOversized and internally heterogeneous coalitions could "
    "split into smaller successor institutions."
)


print(
    "\nWeak coalitions could disappear entirely."
)


print(
    "\nAgents therefore inhabit an endogenous institutional "
    "labor market: reputation and epistemic performance affect "
    "where they participate, but no permanent department or "
    "organizational chart is imposed from outside."
)


print(
    "\nCell 15 can now perform the institutional autopsy: "
    "reconstructing the complete biography of how fifty "
    "specialists perceived the world, organized themselves, "
    "competed for capital, experienced reality, learned whom "
    "to trust, and ultimately redesigned their own institution."
)

INITIAL ORGANIZATIONAL EVOLUTION ASSESSMENT


,coalition_id,coalition_name,funded,member_count_before,fitness,base_action,reason
0,C01,Emergent Coalition 1,True,8,0.649204,MUTATE,The environment changed materially; the coalit...
1,C02,Emergent Coalition 2,True,8,0.647922,MUTATE,The environment changed materially; the coalit...
2,C03,Emergent Coalition 3,True,8,0.638947,MUTATE,The environment changed materially; the coalit...
3,C04,Emergent Coalition 4,True,8,0.638947,MUTATE,The environment changed materially; the coalit...
4,C05,Emergent Coalition 5,True,8,0.638947,MUTATE,The environment changed materially; the coalit...
5,C06,Emergent Coalition 6,True,8,0.638947,MUTATE,The environment changed materially; the coalit...
8,C09,Emergent Coalition 9,True,8,0.638947,MUTATE,The environment changed materially; the coalit...
6,C07,Cross-Asset Signal Divergence Working Group,True,8,0.636640,MUTATE,The environment changed materially; the coalit...
7,C08,Emergent Coalition 8,True,8,0.636640,MUTATE,The environment changed materially; the coalit...
9,C10,Emergent Coalition 10,True,8,0.635358,MUTATE,The environment changed materially; the coalit...



POTENTIAL COALITION MERGERS
No merger candidates passed the institutional thresholds.

NEXT-GENERATION ORGANIZATIONAL STRUCTURE


,coalition_id,coalition_name,action,generation,member_count,families,diversity,member_quality,inherited_fitness,status
0,C01,Emergent Coalition 1,MUTATE,2,8,6,0.750,0.586725,0.649204,ACTIVE
1,C02,Emergent Coalition 2,MUTATE,2,8,6,0.750,0.576041,0.647922,ACTIVE
2,C03,Emergent Coalition 3,MUTATE,2,8,6,0.750,0.501251,0.638947,ACTIVE
3,C04,Emergent Coalition 4,MUTATE,2,8,6,0.750,0.501251,0.638947,ACTIVE
4,C05,Emergent Coalition 5,MUTATE,2,8,6,0.750,0.501251,0.638947,ACTIVE
5,C06,Emergent Coalition 6,MUTATE,2,8,6,0.750,0.501251,0.638947,ACTIVE
8,C09,Emergent Coalition 9,MUTATE,2,8,6,0.750,0.501251,0.638947,ACTIVE
6,C07,Cross-Asset Signal Divergence Working Group,MUTATE,2,8,5,0.625,0.565357,0.636640,ACTIVE
7,C08,Emergent Coalition 8,MUTATE,2,8,5,0.625,0.565357,0.636640,ACTIVE
9,C10,Emergent Coalition 10,MUTATE,2,8,5,0.625,0.554672,0.635358,ACTIVE



AGENT MIGRATION
No agent migration occurred in this generation.

DISSOLVED COALITIONS
No coalition dissolved in this generation.

EXECUTED MERGERS
No mergers executed.

EXECUTED SPLITS
No coalition splits were required.

ORGANIZATIONAL EVOLUTION SUMMARY
Coalitions before evolution : 12
Coalitions after evolution  : 12
Coalitions dissolved        : 0
Mergers executed            : 0
Splits executed             : 0
Agents recruited            : 0
Agents released             : 0

AGENT PARTICIPATION AFTER EVOLUTION


,agent_id,specialty,reputation,epistemic_score,influence,active_coalitions
2,2,Sovereign Credit,0.621648,0.568368,0.530909,5
6,6,Public Equity Fundamental,0.621648,0.568368,0.530909,5
13,13,Volatility,0.621648,0.568368,0.530909,5
18,18,Distressed Debt,0.621648,0.568368,0.530909,5
24,24,Corporate Restructuring,0.621648,0.568368,0.530909,5
3,3,Investment Grade Credit,0.510439,0.491200,0.500490,5
29,29,Credit Risk,0.510439,0.491200,0.500490,5
40,40,AI Economics,0.510439,0.491200,0.500490,5
42,42,Commodities,0.510439,0.491200,0.500490,5
17,17,Relative Value,0.510439,0.491200,0.500490,4



ORGANIZATIONAL COMPLEXITY
Active agents              : 37/50
Multi-coalition agents     : 18
Average coalition size     : 8.00
Largest coalition          : 8

THE ORGANIZATION HAS CHANGED
The institution did not merely update forecasts after observing market outcomes.

It changed its own internal architecture.

Some coalitions survived because their combination of evidence, calibration, realized outcomes and membership quality remained strong.

Others mutated by releasing weak contributors and recruiting agents with stronger epistemic records or complementary expertise.

Related coalitions were allowed to merge when their financial questions overlapped but their knowledge remained sufficiently complementary.

Oversized and internally heterogeneous coalitions could split into smaller successor institutions.

Weak coalitions could disappear entirely.

Agents therefore inhabit an endogenous institutional labor market: reputation and epistemic performance affect where they participate, bu

## CODE UNIT 15 — Institutional autopsy and biography of the swarm

The final unit treats the experiment as the biography of an institution rather than merely a portfolio backtest. It assembles financial performance, coalition history, committee decisions, reputation trajectories, organizational evolution, and the communication network into a postmortem.

Conventional metrics remain important—ending wealth, drawdown, funded capital—but the deeper variables are institutional. How many coalitions formed? Which survived? Which agents became central? Did persuasive quality predict capital allocation more strongly than realized performance? Did dissent prove useful? Did the organization centralize under stress?

A network visualization provides a snapshot of the emergent institution. In an expanded multi-event run, snapshots can be animated so students literally watch the organizational chart change through time.

The cell also creates machine-readable board and provenance reports. These make the notebook suitable as the computational foundation for a paper, lecture, or subsequent infographic series.

The experiment began with fifty strangers and USD 10 billion. The final question is no longer simply whether they made money. It is what kind of institution they became, which relationships survived reality, and whether the organization learned when its own structure had become obsolete.

In [34]:
# ============================================================
# CODE UNIT 15 — INSTITUTIONAL AUTOPSY AND BIOGRAPHY
#                OF THE AUTONOMOUS FINANCIAL INSTITUTION
# ============================================================
#
# PURPOSE
# -------
# This is the capstone cell.
#
# The experiment began with fifty heterogeneous financial
# specialists and a common financial world whose hidden regime
# was not revealed to them.
#
# Across the notebook they:
#
#       perceived the world differently,
#       formed beliefs,
#       communicated,
#       created temporary coalitions,
#       debated,
#       generated investment memoranda,
#       competed for scarce capital,
#       faced institutional governance,
#       received portfolio allocations,
#       experienced market reality,
#       updated reputations,
#       learned whom to trust,
#       and redesigned their own organization.
#
# Cell 15 reconstructs that complete institutional biography.
#
# It performs five distinct autopsies:
#
#       1. ECONOMIC
#       2. EPISTEMIC
#       3. GOVERNANCE
#       4. ORGANIZATIONAL
#       5. PROVENANCE / REPRODUCIBILITY
#
# It deliberately does NOT collapse all institutional quality
# into a single objective function.
#
# The final object of analysis is not the prompt, model, agent,
# coalition, or portfolio.
#
#                  IT IS THE INSTITUTION.
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import json
import math
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    "/content/capstone_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "=" * 100
)

print(
    "CODE UNIT 15 — INSTITUTIONAL AUTOPSY AND BIOGRAPHY"
)

print(
    "=" * 100
)

print(
    f"\nOutput directory: {OUTPUT_DIR}"
)


# ============================================================
# 2. DEFENSIVE COMPATIBILITY HELPERS
# ============================================================

def safe_float(
    value,
    default=0.0
):

    try:

        value = float(
            value
        )

        if not np.isfinite(
            value
        ):

            return float(
                default
            )

        return value

    except Exception:

        return float(
            default
        )


def safe_int(
    value,
    default=0
):

    try:

        return int(
            value
        )

    except Exception:

        return int(
            default
        )


def safe_bool(
    value,
    default=False
):

    try:

        return bool(
            value
        )

    except Exception:

        return bool(
            default
        )


def safe_list(
    value
):

    if value is None:

        return []

    if isinstance(
        value,
        list
    ):

        return value

    if isinstance(
        value,
        tuple
    ):

        return list(
            value
        )

    if isinstance(
        value,
        set
    ):

        return list(
            value
        )

    return [
        value
    ]


def safe_dataframe(
    name
):

    obj = globals().get(
        name
    )

    if isinstance(
        obj,
        pd.DataFrame
    ):

        return obj.copy()

    return pd.DataFrame()


def make_serializable(
    obj
):

    if isinstance(
        obj,
        dict
    ):

        return {

            str(k):
                make_serializable(
                    v
                )

            for k, v
            in obj.items()

        }


    if isinstance(
        obj,
        (
            list,
            tuple,
            set
        )
    ):

        return [

            make_serializable(
                x
            )

            for x in obj

        ]


    if isinstance(
        obj,
        np.integer
    ):

        return int(
            obj
        )


    if isinstance(
        obj,
        np.floating
    ):

        value = float(
            obj
        )

        if not np.isfinite(
            value
        ):

            return None

        return value


    if isinstance(
        obj,
        np.bool_
    ):

        return bool(
            obj
        )


    if isinstance(
        obj,
        pd.Timestamp
    ):

        return obj.isoformat()


    if isinstance(
        obj,
        Path
    ):

        return str(
            obj
        )


    if isinstance(
        obj,
        float
    ):

        if not math.isfinite(
            obj
        ):

            return None


    return obj


# ============================================================
# 3. RECOVER CORE NOTEBOOK OBJECTS SAFELY
# ============================================================

agents_obj = globals().get(
    "agents",
    []
)

coalitions_obj = globals().get(
    "coalitions",
    []
)

next_generation_obj = globals().get(
    "next_generation",
    globals().get(
        "evolved_coalitions",
        coalitions_obj
    )
)

memos_obj = globals().get(
    "memos",
    []
)

decisions_obj = globals().get(
    "decisions",
    []
)

history_obj = globals().get(
    "history",
    []
)

assets_obj = globals().get(
    "assets",
    []
)

regimes_obj = globals().get(
    "regimes",
    []
)

performance_record_obj = globals().get(
    "performance_record",
    {}
)

learning_summary_obj = globals().get(
    "learning_summary",
    {}
)

CHARTER_OBJ = globals().get(
    "CHARTER",
    {}
)


# DataFrames from previous cells

allocation_df_safe = safe_dataframe(
    "allocation_df"
)

episode_df_safe = safe_dataframe(
    "episode_df"
)

coalition_learning_df_safe = safe_dataframe(
    "coalition_learning_df"
)

agent_learning_df_safe = safe_dataframe(
    "agent_learning_df"
)

agent_state_df_safe = safe_dataframe(
    "agent_state_df"
)

memo_df_safe = safe_dataframe(
    "memo_df"
)

final_evolution_df_safe = safe_dataframe(
    "final_evolution_df"
)

migration_df_safe = safe_dataframe(
    "migration_df"
)

split_df_safe = safe_dataframe(
    "split_df"
)


# ============================================================
# 4. RECOVER EXPERIMENT PARAMETERS
# ============================================================

SEED_SAFE = safe_int(
    globals().get(
        "SEED",
        766
    ),
    766
)

MODEL_SAFE = str(
    globals().get(
        "MODEL",
        "UNKNOWN"
    )
)

LIVE_LLM_SAFE = safe_bool(
    globals().get(
        "LIVE_LLM",
        False
    )
)

LLM_CALLS_SAFE = safe_int(
    globals().get(
        "LLM_CALLS",
        0
    )
)

MAX_LLM_CALLS_SAFE = safe_int(
    globals().get(
        "MAX_LLM_CALLS",
        0
    )
)

INITIAL_CAPITAL_SAFE = safe_float(
    globals().get(
        "INITIAL_CAPITAL",
        10_000_000_000
    ),
    10_000_000_000
)

EVENT_DAY_SAFE = safe_int(
    globals().get(
        "EVENT_DAY",
        0
    )
)

EVAL_DAYS_SAFE = safe_int(
    globals().get(
        "EVAL_DAYS",
        60
    ),
    60
)

EVAL_END_SAFE = safe_int(
    globals().get(
        "EVAL_END",
        EVENT_DAY_SAFE + EVAL_DAYS_SAFE
    ),
    EVENT_DAY_SAFE + EVAL_DAYS_SAFE
)

actual_eval_days_safe = safe_int(
    globals().get(
        "actual_eval_days",
        performance_record_obj.get(
            "evaluation_days",
            EVAL_DAYS_SAFE
        )
    ),
    EVAL_DAYS_SAFE
)


# ============================================================
# 5. RECOVER PROVENANCE ROBUSTLY
# ============================================================

def recover_provenance_log():

    candidate_names = [

        "provenance",
        "provenance_log",
        "llm_provenance",
        "audit_log",
        "llm_log",
        "call_log"

    ]


    for name in candidate_names:

        if name in globals():

            obj = globals()[
                name
            ]


            if isinstance(
                obj,
                list
            ):

                print(
                    f"\nProvenance recovered from: {name}"
                )

                return obj


    print(
        "\nWARNING:"
    )

    print(
        "No explicit provenance log was found."
    )

    print(
        "Cell 15 will construct a minimal reproducibility "
        "record from the notebook state."
    )


    fallback_record = {

        "type":
            "CAPSTONE_PROVENANCE_FALLBACK",

        "model":
            MODEL_SAFE,

        "live_llm":
            LIVE_LLM_SAFE,

        "llm_calls_used":
            LLM_CALLS_SAFE,

        "llm_call_limit":
            MAX_LLM_CALLS_SAFE,

        "seed":
            SEED_SAFE,

        "decision_day":
            EVENT_DAY_SAFE,

        "evaluation_end":
            EVAL_END_SAFE,

        "note":
            (
                "Explicit per-call provenance was unavailable. "
                "This record was constructed from the surviving "
                "notebook state. No artificial LLM call history "
                "has been fabricated."
            )

    }


    return [
        fallback_record
    ]


provenance_records = (
    recover_provenance_log()
)


# ============================================================
# 6. RECOVER SPECIALTY FAMILY FUNCTION
# ============================================================

def final_specialty_family(
    specialty
):

    # Reuse Cell 14 if available

    if (
        "specialty_family"
        in globals()

        and

        globals()[
            "specialty_family"
        ]
        is not final_specialty_family
    ):

        try:

            return globals()[
                "specialty_family"
            ](
                specialty
            )

        except Exception:

            pass


    # Reuse Cell 6 if available

    if "family" in globals():

        try:

            return globals()[
                "family"
            ](
                specialty
            )

        except Exception:

            pass


    # Defensive fallback

    s = str(
        specialty
    ).lower()


    if any(
        k in s
        for k in [
            "credit",
            "distressed",
            "bank"
        ]
    ):

        return "credit"


    if any(
        k in s
        for k in [
            "macro",
            "rates",
            "sovereign",
            "fx",
            "commodities"
        ]
    ):

        return "macro"


    if any(
        k in s
        for k in [
            "equity",
            "value",
            "growth",
            "quality",
            "small cap"
        ]
    ):

        return "equity"


    if any(
        k in s
        for k in [
            "venture",
            "private",
            "m&a",
            "restructuring",
            "capital structure"
        ]
    ):

        return "private"


    if any(
        k in s
        for k in [
            "risk",
            "treasury",
            "liquidity"
        ]
    ):

        return "risk"


    if any(
        k in s
        for k in [
            "crypto",
            "digital",
            "stablecoin",
            "payments",
            "fintech"
        ]
    ):

        return "digital"


    if any(
        k in s
        for k in [
            "ai ",
            "semiconductor",
            "power",
            "infrastructure"
        ]
    ):

        return "tech"


    if any(
        k in s
        for k in [
            "quant",
            "econometric",
            "systematic",
            "derivative",
            "volatility",
            "option",
            "microstructure"
        ]
    ):

        return "quant"


    return "outside"


# ============================================================
# 7. RECONSTRUCT THE COMPLETE INSTITUTIONAL CAUSAL CHAIN
# ============================================================

institutional_chain = [

    {
        "stage": 1,
        "name": "WORLD",
        "description":
            (
                "Synthetic financial environment with "
                "hidden market regimes."
            )
    },

    {
        "stage": 2,
        "name": "PERCEPTION",
        "description":
            (
                "Agents observe different subsets of "
                "the same financial world."
            )
    },

    {
        "stage": 3,
        "name": "BELIEFS",
        "description":
            (
                "Specialists transform partial evidence "
                "into explicit financial theses."
            )
    },

    {
        "stage": 4,
        "name": "COMMUNICATION",
        "description":
            (
                "Endogenous affinity determines which "
                "specialists exchange ideas."
            )
    },

    {
        "stage": 5,
        "name": "COALITIONS",
        "description":
            (
                "Temporary and potentially overlapping "
                "institutions emerge."
            )
    },

    {
        "stage": 6,
        "name": "DEBATE",
        "description":
            (
                "Coalitions negotiate agreement, dissent, "
                "falsification and minority views."
            )
    },

    {
        "stage": 7,
        "name": "MEMORANDA",
        "description":
            (
                "Coalitions translate hypotheses into "
                "formal requests for capital."
            )
    },

    {
        "stage": 8,
        "name": "GOVERNANCE",
        "description":
            (
                "The Investment Committee evaluates "
                "evidence, risk and institutional fit."
            )
    },

    {
        "stage": 9,
        "name": "CAPITAL ALLOCATION",
        "description":
            (
                "Authorized proposals compete for a "
                "scarce institutional balance sheet."
            )
    },

    {
        "stage": 10,
        "name": "REALITY",
        "description":
            (
                "Out-of-sample market outcomes generate "
                "P&L, volatility and drawdowns."
            )
    },

    {
        "stage": 11,
        "name": "LEARNING",
        "description":
            (
                "Reputation, epistemic quality, persuasion "
                "and influence are updated separately."
            )
    },

    {
        "stage": 12,
        "name": "EVOLUTION",
        "description":
            (
                "Coalitions persist, recruit, mutate, "
                "merge, split or dissolve."
            )
    }

]


chain_df = pd.DataFrame(
    institutional_chain
)


print(
    "\n"
    + "=" * 100
)

print(
    "BIOGRAPHY OF THE AUTONOMOUS FINANCIAL INSTITUTION"
)

print(
    "=" * 100
)


display(
    chain_df
)


# ============================================================
# 8. ECONOMIC AUTOPSY
# ============================================================

ending_wealth = safe_float(

    performance_record_obj.get(
        "ending_wealth",
        INITIAL_CAPITAL_SAFE
    ),

    INITIAL_CAPITAL_SAFE

)


institutional_pnl = safe_float(

    performance_record_obj.get(
        "institutional_pnl",
        ending_wealth
        -
        INITIAL_CAPITAL_SAFE
    )

)


institutional_return = safe_float(

    performance_record_obj.get(
        "institutional_return",
        (
            ending_wealth
            /
            INITIAL_CAPITAL_SAFE
            -
            1.0
        )
        if INITIAL_CAPITAL_SAFE > 0
        else 0.0
    )

)


institutional_max_dd = safe_float(

    performance_record_obj.get(
        "max_drawdown",
        0.0
    )

)


institutional_vol = safe_float(

    performance_record_obj.get(
        "realized_volatility",
        0.0
    )

)


institutional_sharpe = safe_float(

    performance_record_obj.get(
        "realized_sharpe",
        0.0
    )

)


coalition_hit_rate = safe_float(

    performance_record_obj.get(
        "coalition_hit_rate",
        0.0
    )

)


starting_regime = str(

    performance_record_obj.get(
        "starting_regime",
        performance_record_obj.get(
            "starting_hidden_regime",
            ""
        )
    )

)


ending_regime = str(

    performance_record_obj.get(
        "ending_regime",
        performance_record_obj.get(
            "ending_hidden_regime",
            starting_regime
        )
    )

)


regime_changed = safe_bool(

    performance_record_obj.get(
        "regime_changed",
        (
            starting_regime
            !=
            ending_regime
        )
    )

)


economic_autopsy = {

    "initial_capital":
        INITIAL_CAPITAL_SAFE,

    "ending_wealth":
        ending_wealth,

    "institutional_pnl":
        institutional_pnl,

    "institutional_return":
        institutional_return,

    "max_drawdown":
        institutional_max_dd,

    "realized_volatility":
        institutional_vol,

    "realized_sharpe":
        institutional_sharpe,

    "coalition_hit_rate":
        coalition_hit_rate,

    "evaluation_days":
        actual_eval_days_safe,

    "starting_regime":
        starting_regime,

    "ending_regime":
        ending_regime,

    "regime_changed":
        regime_changed

}


print(
    "\n"
    + "=" * 100
)

print(
    "I. ECONOMIC AUTOPSY"
)

print(
    "=" * 100
)


print(
    f"Initial capital          : "
    f"${INITIAL_CAPITAL_SAFE:,.0f}"
)

print(
    f"Ending wealth            : "
    f"${ending_wealth:,.0f}"
)

print(
    f"Institutional P&L        : "
    f"${institutional_pnl:,.0f}"
)

print(
    f"Institutional return     : "
    f"{institutional_return:.2%}"
)

print(
    f"Maximum drawdown         : "
    f"{institutional_max_dd:.2%}"
)

print(
    f"Realized volatility      : "
    f"{institutional_vol:.2%}"
)

print(
    f"Sharpe-like ratio        : "
    f"{institutional_sharpe:.2f}"
)

print(
    f"Coalition hit rate       : "
    f"{coalition_hit_rate:.1%}"
)

print(
    f"Hidden regime            : "
    f"{starting_regime} → {ending_regime}"
)


# ============================================================
# 9. CAPITAL ALLOCATION AUTOPSY
# ============================================================

def first_existing_column(
    df,
    candidates
):

    for c in candidates:

        if c in df.columns:

            return c

    return None


committee_col = first_existing_column(

    allocation_df_safe,

    [
        "committee_approved",
        "approved_capital",
        "capital"
    ]

)


final_col = first_existing_column(

    allocation_df_safe,

    [
        "final_allocation",
        "capital",
        "allocated_capital"
    ]

)


if (
    len(
        allocation_df_safe
    )
    >
    0
):

    if committee_col:

        total_committee_approved = safe_float(

            allocation_df_safe[
                committee_col
            ].sum()

        )

    else:

        total_committee_approved = 0.0


    if final_col:

        total_final_allocation = safe_float(

            allocation_df_safe[
                final_col
            ].sum()

        )

        funded_positions = safe_int(

            (
                allocation_df_safe[
                    final_col
                ]
                >
                0
            ).sum()

        )

    else:

        total_final_allocation = 0.0
        funded_positions = 0

else:

    total_committee_approved = 0.0
    total_final_allocation = 0.0
    funded_positions = 0


capital_reduction = max(

    0.0,

    total_committee_approved
    -
    total_final_allocation

)


capital_reduction_pct = (

    capital_reduction
    /
    total_committee_approved

    if total_committee_approved > 0

    else 0.0

)


cash_after_allocation = (

    INITIAL_CAPITAL_SAFE
    -
    total_final_allocation

)


capital_autopsy = {

    "committee_approved":
        total_committee_approved,

    "final_allocated":
        total_final_allocation,

    "capital_reduction":
        capital_reduction,

    "capital_reduction_pct":
        capital_reduction_pct,

    "funded_positions":
        funded_positions,

    "cash_after_allocation":
        cash_after_allocation

}


print(
    "\nCAPITAL ALLOCATION"
)

print(
    f"Committee authorized     : "
    f"${total_committee_approved:,.0f}"
)

print(
    f"Final deployed capital   : "
    f"${total_final_allocation:,.0f}"
)

print(
    f"Portfolio-level reduction: "
    f"${capital_reduction:,.0f} "
    f"({capital_reduction_pct:.1%})"
)

print(
    f"Funded coalition positions: "
    f"{funded_positions}"
)

print(
    f"Residual capital / cash  : "
    f"${cash_after_allocation:,.0f}"
)


# ============================================================
# 10. COALITION ECONOMIC CONTRIBUTION
# ============================================================

desired_episode_columns = [

    "coalition_id",
    "coalition_name",
    "asset",
    "direction",
    "capital",
    "realized_return",
    "pnl",
    "max_drawdown",
    "reality_score"

]


available_episode_columns = [

    c

    for c in desired_episode_columns

    if c in episode_df_safe.columns

]


if (
    len(
        episode_df_safe
    )
    >
    0
):

    coalition_pnl_ranking = (

        episode_df_safe[
            available_episode_columns
        ]

        .copy()

    )


    if "pnl" in coalition_pnl_ranking.columns:

        coalition_pnl_ranking = (

            coalition_pnl_ranking

            .sort_values(
                "pnl",
                ascending=False
            )

            .reset_index(
                drop=True
            )

        )


    coalition_pnl_ranking.insert(

        0,

        "rank",

        np.arange(
            1,
            len(
                coalition_pnl_ranking
            )
            +
            1
        )

    )


    print(
        "\nCOALITION ECONOMIC CONTRIBUTION"
    )


    display(
        coalition_pnl_ranking
    )


else:

    coalition_pnl_ranking = pd.DataFrame()

    print(
        "\nNo coalition-level performance table "
        "was available for ranking."
    )


# ============================================================
# 11. EPISTEMIC AUTOPSY
# ============================================================

evidence_corr = safe_float(

    performance_record_obj.get(
        "evidence_return_correlation",
        0.0
    )

)


persuasion_corr = safe_float(

    performance_record_obj.get(
        "persuasion_return_correlation",
        0.0
    )

)


committee_corr = safe_float(

    performance_record_obj.get(
        "committee_return_correlation",
        0.0
    )

)


debate_corr = safe_float(

    performance_record_obj.get(
        "debate_return_correlation",
        0.0
    )

)


mean_reputation_before = safe_float(

    learning_summary_obj.get(
        "mean_reputation_before",
        0.0
    )

)


mean_reputation_after = safe_float(

    learning_summary_obj.get(
        "mean_reputation_after",
        0.0
    )

)


mean_epistemic_score = safe_float(

    learning_summary_obj.get(
        "mean_epistemic_score",
        0.0
    )

)


mean_persuasion_score = safe_float(

    learning_summary_obj.get(
        "mean_persuasion_score",
        0.0
    )

)


mean_influence = safe_float(

    learning_summary_obj.get(
        "mean_influence",
        0.0
    )

)


committee_calibration_error = safe_float(

    learning_summary_obj.get(
        "committee_calibration_error",
        0.0
    )

)


coalition_calibration_error = safe_float(

    learning_summary_obj.get(
        "coalition_calibration_error",
        0.0
    )

)


epistemic_autopsy = {

    "evidence_return_correlation":
        evidence_corr,

    "persuasion_return_correlation":
        persuasion_corr,

    "committee_return_correlation":
        committee_corr,

    "debate_return_correlation":
        debate_corr,

    "mean_reputation_before":
        mean_reputation_before,

    "mean_reputation_after":
        mean_reputation_after,

    "mean_epistemic_score":
        mean_epistemic_score,

    "mean_persuasion_score":
        mean_persuasion_score,

    "mean_influence":
        mean_influence,

    "committee_calibration_error":
        committee_calibration_error,

    "coalition_calibration_error":
        coalition_calibration_error

}


print(
    "\n"
    + "=" * 100
)

print(
    "II. EPISTEMIC AUTOPSY"
)

print(
    "=" * 100
)


print(
    f"Evidence → return correlation   : "
    f"{evidence_corr:.3f}"
)

print(
    f"Persuasion → return correlation : "
    f"{persuasion_corr:.3f}"
)

print(
    f"Committee → return correlation  : "
    f"{committee_corr:.3f}"
)

print(
    f"Debate → return correlation     : "
    f"{debate_corr:.3f}"
)

print(
    f"Mean reputation                 : "
    f"{mean_reputation_before:.3f} "
    f"→ {mean_reputation_after:.3f}"
)

print(
    f"Mean epistemic score            : "
    f"{mean_epistemic_score:.3f}"
)

print(
    f"Mean persuasion score           : "
    f"{mean_persuasion_score:.3f}"
)

print(
    f"Mean influence                  : "
    f"{mean_influence:.3f}"
)

print(
    f"Committee calibration error     : "
    f"{committee_calibration_error:.3f}"
)

print(
    f"Coalition calibration error     : "
    f"{coalition_calibration_error:.3f}"
)


# ============================================================
# 12. TOP EPISTEMIC AGENTS
# ============================================================

if (
    len(
        agent_state_df_safe
    )
    >
    0
):

    sort_columns = [

        c

        for c in [
            "epistemic_score",
            "reputation"
        ]

        if c in agent_state_df_safe.columns

    ]


    if sort_columns:

        top_epistemic_agents = (

            agent_state_df_safe

            .sort_values(

                sort_columns,

                ascending=False

            )

            .head(10)

            .copy()

        )

    else:

        top_epistemic_agents = (

            agent_state_df_safe

            .head(10)

            .copy()

        )


else:

    # Reconstruct from agent objects if necessary

    reconstructed_agents = []


    for a in agents_obj:

        reconstructed_agents.append({

            "agent_id":
                getattr(
                    a,
                    "id",
                    ""
                ),

            "specialty":
                getattr(
                    a,
                    "specialty",
                    ""
                ),

            "reputation":
                safe_float(
                    getattr(
                        a,
                        "reputation",
                        0.5
                    )
                ),

            "epistemic_score":
                safe_float(
                    getattr(
                        a,
                        "epistemic_score",
                        0.5
                    )
                ),

            "persuasion_score":
                safe_float(
                    getattr(
                        a,
                        "persuasion_score",
                        0.5
                    )
                ),

            "influence":
                safe_float(
                    getattr(
                        a,
                        "influence",
                        0.5
                    )
                )

        })


    reconstructed_agent_df = pd.DataFrame(
        reconstructed_agents
    )


    if len(
        reconstructed_agent_df
    ):

        top_epistemic_agents = (

            reconstructed_agent_df

            .sort_values(

                [
                    "epistemic_score",
                    "reputation"
                ],

                ascending=False

            )

            .head(10)

        )

    else:

        top_epistemic_agents = pd.DataFrame()


print(
    "\nTOP EPISTEMIC AGENTS"
)


if len(
    top_epistemic_agents
):

    display(
        top_epistemic_agents
    )

else:

    print(
        "No agent state information available."
    )


# ============================================================
# 13. REPUTATION GAINS AND LOSSES
# ============================================================

if (
    len(
        agent_learning_df_safe
    )
    >
    0

    and

    "reputation_change"
    in agent_learning_df_safe.columns
):

    evaluated_agent_changes = (

        agent_learning_df_safe.copy()

    )


    if (
        "coalitions_evaluated"
        in evaluated_agent_changes.columns
    ):

        evaluated_agent_changes = (

            evaluated_agent_changes[

                evaluated_agent_changes[
                    "coalitions_evaluated"
                ]
                >
                0

            ]

        )


    biggest_gains = (

        evaluated_agent_changes

        .sort_values(
            "reputation_change",
            ascending=False
        )

        .head(5)

    )


    biggest_losses = (

        evaluated_agent_changes

        .sort_values(
            "reputation_change",
            ascending=True
        )

        .head(5)

    )


    print(
        "\nLARGEST REPUTATION GAINS"
    )

    display(
        biggest_gains
    )


    print(
        "\nLARGEST REPUTATION LOSSES"
    )

    display(
        biggest_losses
    )


else:

    evaluated_agent_changes = pd.DataFrame()
    biggest_gains = pd.DataFrame()
    biggest_losses = pd.DataFrame()


# ============================================================
# 14. GOVERNANCE AUTOPSY
# ============================================================

if len(
    decisions_obj
):

    decision_frame = pd.DataFrame(
        decisions_obj
    )

else:

    decision_frame = pd.DataFrame()


if (
    len(
        decision_frame
    )
    >
    0

    and

    "decision"
    in decision_frame.columns
):

    decision_distribution = (

        decision_frame[
            "decision"
        ]

        .value_counts()

        .to_dict()

    )

else:

    decision_distribution = {}


if (
    len(
        memo_df_safe
    )
    >
    0
):

    mean_evidence = (

        safe_float(
            memo_df_safe[
                "evidence_strength"
            ].mean()
        )

        if "evidence_strength"
        in memo_df_safe.columns

        else 0.0

    )


    mean_persuasion = (

        safe_float(
            memo_df_safe[
                "persuasive_quality"
            ].mean()
        )

        if "persuasive_quality"
        in memo_df_safe.columns

        else 0.0

    )


    mean_eloquence_gap = (

        safe_float(
            memo_df_safe[
                "eloquence_gap"
            ].mean()
        )

        if "eloquence_gap"
        in memo_df_safe.columns

        else (
            mean_persuasion
            -
            mean_evidence
        )

    )


else:

    mean_evidence = 0.0
    mean_persuasion = 0.0
    mean_eloquence_gap = 0.0


governance_autopsy = {

    "proposals":
        len(
            memos_obj
        ),

    "committee_decisions":
        len(
            decisions_obj
        ),

    "decision_distribution":
        decision_distribution,

    "mean_evidence_strength":
        mean_evidence,

    "mean_persuasive_quality":
        mean_persuasion,

    "mean_eloquence_gap":
        mean_eloquence_gap,

    "committee_calibration_error":
        committee_calibration_error

}


print(
    "\n"
    + "=" * 100
)

print(
    "III. GOVERNANCE AUTOPSY"
)

print(
    "=" * 100
)


print(
    f"Investment proposals        : "
    f"{len(memos_obj)}"
)

print(
    f"Committee decisions         : "
    f"{len(decisions_obj)}"
)

print(
    f"Decision distribution       : "
    f"{decision_distribution}"
)

print(
    f"Mean evidence strength      : "
    f"{mean_evidence:.3f}"
)

print(
    f"Mean persuasive quality     : "
    f"{mean_persuasion:.3f}"
)

print(
    f"Mean eloquence gap          : "
    f"{mean_eloquence_gap:.3f}"
)

print(
    f"Committee calibration error : "
    f"{committee_calibration_error:.3f}"
)


# ============================================================
# 15. ORGANIZATIONAL AUTOPSY
# ============================================================

original_active_safe = safe_int(

    globals().get(
        "original_active",
        len(
            coalitions_obj
        )
    )

)


final_active_safe = safe_int(

    globals().get(
        "final_active",
        len(
            next_generation_obj
        )
    )

)


n_dissolved_safe = safe_int(

    globals().get(
        "n_dissolved",
        0
    )

)


n_merged_safe = safe_int(

    globals().get(
        "n_merged",
        0
    )

)


n_splits_safe = safe_int(

    globals().get(
        "n_splits",
        0
    )

)


n_recruited_safe = safe_int(

    globals().get(
        "n_recruited",
        0
    )

)


n_released_safe = safe_int(

    globals().get(
        "n_released",
        0
    )

)


# Reconstruct participation if Cell 14 did not expose it

participation_final = {

    getattr(
        a,
        "id",
        f"A{i:02d}"
    ):
        0

    for i, a
    in enumerate(
        agents_obj
    )

}


for c in next_generation_obj:

    for aid in c.get(
        "members",
        []
    ):

        if aid not in participation_final:

            participation_final[
                aid
            ] = 0

        participation_final[
            aid
        ] += 1


active_agents_safe = safe_int(

    globals().get(
        "active_agents",
        sum(
            1
            for v in participation_final.values()
            if v > 0
        )
    )

)


multi_coalition_agents_safe = safe_int(

    globals().get(
        "multi_coalition_agents",
        sum(
            1
            for v in participation_final.values()
            if v > 1
        )
    )

)


coalition_sizes_final = [

    len(
        c.get(
            "members",
            []
        )
    )

    for c in next_generation_obj

]


avg_coalition_size_safe = safe_float(

    globals().get(
        "avg_coalition_size",
        np.mean(
            coalition_sizes_final
        )
        if coalition_sizes_final
        else 0.0
    )

)


largest_coalition_safe = safe_int(

    globals().get(
        "max_coalition_size_realized",
        max(
            coalition_sizes_final
        )
        if coalition_sizes_final
        else 0
    )

)


organizational_autopsy = {

    "agents":
        len(
            agents_obj
        ),

    "original_coalitions":
        original_active_safe,

    "next_generation_coalitions":
        final_active_safe,

    "coalitions_dissolved":
        n_dissolved_safe,

    "mergers":
        n_merged_safe,

    "splits":
        n_splits_safe,

    "agents_recruited":
        n_recruited_safe,

    "agents_released":
        n_released_safe,

    "active_agents":
        active_agents_safe,

    "multi_coalition_agents":
        multi_coalition_agents_safe,

    "average_coalition_size":
        avg_coalition_size_safe,

    "largest_coalition":
        largest_coalition_safe

}


print(
    "\n"
    + "=" * 100
)

print(
    "IV. ORGANIZATIONAL AUTOPSY"
)

print(
    "=" * 100
)


print(
    f"Agents                         : "
    f"{len(agents_obj)}"
)

print(
    f"Coalitions before evolution    : "
    f"{original_active_safe}"
)

print(
    f"Coalitions after evolution     : "
    f"{final_active_safe}"
)

print(
    f"Dissolutions                   : "
    f"{n_dissolved_safe}"
)

print(
    f"Mergers                        : "
    f"{n_merged_safe}"
)

print(
    f"Splits                         : "
    f"{n_splits_safe}"
)

print(
    f"Agents recruited               : "
    f"{n_recruited_safe}"
)

print(
    f"Agents released                : "
    f"{n_released_safe}"
)

print(
    f"Active agents                  : "
    f"{active_agents_safe}"
)

print(
    f"Multi-coalition agents         : "
    f"{multi_coalition_agents_safe}"
)

print(
    f"Average coalition size         : "
    f"{avg_coalition_size_safe:.2f}"
)

print(
    f"Largest coalition              : "
    f"{largest_coalition_safe}"
)


# ============================================================
# 16. CONSTRUCT FINAL EVOLVED NETWORK
# ============================================================

G_final = nx.Graph()


for i, a in enumerate(
    agents_obj
):

    aid = getattr(
        a,
        "id",
        f"A{i:02d}"
    )

    specialty = getattr(
        a,
        "specialty",
        "Unknown"
    )


    G_final.add_node(

        aid,

        specialty=specialty,

        reputation=safe_float(
            getattr(
                a,
                "reputation",
                0.5
            )
        ),

        epistemic_score=safe_float(
            getattr(
                a,
                "epistemic_score",
                0.5
            )
        ),

        influence=safe_float(
            getattr(
                a,
                "influence",
                0.5
            )
        ),

        family=final_specialty_family(
            specialty
        )

    )


for c in next_generation_obj:

    members = [

        aid

        for aid
        in c.get(
            "members",
            []
        )

        if aid in G_final.nodes

    ]


    for i in range(
        len(
            members
        )
    ):

        for j in range(
            i + 1,
            len(
                members
            )
        ):

            u = members[
                i
            ]

            v = members[
                j
            ]


            if G_final.has_edge(
                u,
                v
            ):

                G_final[
                    u
                ][
                    v
                ][
                    "weight"
                ] += 1

            else:

                G_final.add_edge(

                    u,
                    v,

                    weight=1

                )


# ============================================================
# 17. NETWORK AUTOPSY
# ============================================================

network_nodes = (
    G_final.number_of_nodes()
)

network_edges = (
    G_final.number_of_edges()
)


network_density = (

    float(
        nx.density(
            G_final
        )
    )

    if network_nodes > 1

    else 0.0

)


active_subgraph_nodes = [

    node

    for node, degree
    in G_final.degree()

    if degree > 0

]


if active_subgraph_nodes:

    G_active = G_final.subgraph(
        active_subgraph_nodes
    )


    components = list(
        nx.connected_components(
            G_active
        )
    )


    connected_components = len(
        components
    )


    largest_component = max(

        [
            len(
                c
            )
            for c in components
        ],

        default=0

    )


else:

    connected_components = 0
    largest_component = 0


network_autopsy = {

    "nodes":
        network_nodes,

    "edges":
        network_edges,

    "density":
        network_density,

    "active_nodes":
        len(
            active_subgraph_nodes
        ),

    "connected_components":
        connected_components,

    "largest_component":
        largest_component

}


print(
    "\nFINAL ORGANIZATIONAL NETWORK"
)

print(
    f"Nodes                  : "
    f"{network_nodes}"
)

print(
    f"Edges                  : "
    f"{network_edges}"
)

print(
    f"Density                : "
    f"{network_density:.3f}"
)

print(
    f"Connected active agents: "
    f"{len(active_subgraph_nodes)}"
)

print(
    f"Connected components   : "
    f"{connected_components}"
)

print(
    f"Largest component      : "
    f"{largest_component}"
)


# ============================================================
# 18. VISUALIZE THE EVOLVED INSTITUTION
# ============================================================

network_figure_path = (

    OUTPUT_DIR

    /
    "final_institutional_network.png"

)


if network_nodes > 0:

    plt.figure(
        figsize=(
            15,
            10
        )
    )


    pos = nx.spring_layout(

        G_final,

        seed=SEED_SAFE,

        weight="weight",

        k=0.75

    )


    node_sizes = [

        500

        +

        1800
        *
        safe_float(
            G_final.nodes[
                node
            ].get(
                "influence",
                0.5
            )
        )

        for node
        in G_final.nodes()

    ]


    node_values = [

        safe_float(
            G_final.nodes[
                node
            ].get(
                "epistemic_score",
                0.5
            )
        )

        for node
        in G_final.nodes()

    ]


    edge_widths = [

        0.8

        +

        1.2
        *
        safe_float(
            G_final[
                u
            ][
                v
            ].get(
                "weight",
                1
            )
        )

        for u, v
        in G_final.edges()

    ]


    nx.draw_networkx_edges(

        G_final,

        pos,

        width=edge_widths,

        alpha=0.25

    )


    nodes_artist = nx.draw_networkx_nodes(

        G_final,

        pos,

        node_size=node_sizes,

        node_color=node_values,

        cmap="viridis",

        vmin=0.0,

        vmax=1.0,

        alpha=0.90

    )


    nx.draw_networkx_labels(

        G_final,

        pos,

        font_size=7

    )


    plt.colorbar(

        nodes_artist,

        label="Epistemic Score"

    )


    plt.title(

        "Next-Generation Autonomous Financial Institution\n"
        "Node Size = Influence | "
        "Node Color = Epistemic Score | "
        "Edges = Shared Coalition Membership"

    )


    plt.axis(
        "off"
    )


    plt.tight_layout()


    plt.savefig(

        network_figure_path,

        dpi=180,

        bbox_inches="tight"

    )


    plt.show()


else:

    print(
        "\nNo network visualization generated because "
        "the final graph contains no nodes."
    )


# ============================================================
# 19. MULTIDIMENSIONAL INSTITUTIONAL SCORECARD
#
# IMPORTANT:
#
# These are descriptive diagnostics.
#
# They are NOT combined into a single institutional score.
# ============================================================

economic_score = float(

    np.clip(

        0.50

        +

        2.0
        *
        institutional_return

        +

        institutional_max_dd,

        0.0,
        1.0

    )

)


epistemic_score_dimension = float(

    np.clip(

        0.40
        *
        mean_epistemic_score

        +

        0.25
        *
        mean_reputation_after

        +

        0.20
        *
        (
            1.0
            -
            committee_calibration_error
        )

        +

        0.15
        *
        (
            1.0
            -
            coalition_calibration_error
        ),

        0.0,
        1.0

    )

)


governance_score = float(

    np.clip(

        0.35
        *
        mean_evidence

        +

        0.25
        *
        (
            1.0
            -
            committee_calibration_error
        )

        +

        0.20
        *
        (
            1.0
            -
            max(
                0.0,
                mean_eloquence_gap
            )
        )

        +

        0.20
        *
        (
            1.0
            -
            capital_reduction_pct
        ),

        0.0,
        1.0

    )

)


organizational_score = float(

    np.clip(

        0.30
        *
        (
            active_agents_safe
            /
            max(
                len(
                    agents_obj
                ),
                1
            )
        )

        +

        0.20
        *
        (
            multi_coalition_agents_safe
            /
            max(
                len(
                    agents_obj
                ),
                1
            )
        )

        +

        0.25
        *
        (
            1.0
            -
            min(
                1.0,

                n_dissolved_safe
                /
                max(
                    original_active_safe,
                    1
                )
            )
        )

        +

        0.25
        *
        min(
            1.0,
            network_density
            *
            3.0
        ),

        0.0,
        1.0

    )

)


institutional_scorecard = pd.DataFrame({

    "dimension": [

        "Economic",

        "Epistemic",

        "Governance",

        "Organizational"

    ],

    "diagnostic_score": [

        economic_score,

        epistemic_score_dimension,

        governance_score,

        organizational_score

    ]

})


print(
    "\n"
    + "=" * 100
)

print(
    "MULTIDIMENSIONAL INSTITUTIONAL SCORECARD"
)

print(
    "=" * 100
)


display(
    institutional_scorecard
)


print(
    "\nThese dimensions are intentionally NOT aggregated "
    "into one institutional performance number."
)


# ============================================================
# 20. BEFORE / AFTER ORGANIZATIONAL ARCHITECTURE
# ============================================================

original_memberships = sum(

    len(
        c.get(
            "members",
            []
        )
    )

    for c in coalitions_obj

)


evolved_memberships = sum(

    len(
        c.get(
            "members",
            []
        )
    )

    for c in next_generation_obj

)


original_active_agents = len(

    set(

        aid

        for c in coalitions_obj

        for aid in c.get(
            "members",
            []
        )

    )

)


architecture_comparison = pd.DataFrame([

    {

        "metric":
            "Coalitions",

        "before":
            original_active_safe,

        "after":
            final_active_safe

    },

    {

        "metric":
            "Total coalition memberships",

        "before":
            original_memberships,

        "after":
            evolved_memberships

    },

    {

        "metric":
            "Average coalition size",

        "before":
            (
                original_memberships
                /
                max(
                    original_active_safe,
                    1
                )
            ),

        "after":
            avg_coalition_size_safe

    },

    {

        "metric":
            "Active agents",

        "before":
            original_active_agents,

        "after":
            active_agents_safe

    },

    {

        "metric":
            "Multi-coalition agents",

        "before":
            np.nan,

        "after":
            multi_coalition_agents_safe

    }

])


print(
    "\nORGANIZATIONAL ARCHITECTURE: BEFORE VS. AFTER"
)


display(
    architecture_comparison
)


# ============================================================
# 21. BUILD THE INSTITUTION'S CHRONOLOGICAL BIOGRAPHY
# ============================================================

institutional_biography = [

    {
        "chapter": 1,
        "title": "Birth",
        "event":
            (
                f"{len(agents_obj)} heterogeneous specialists "
                "entered a common synthetic financial world."
            )
    },

    {
        "chapter": 2,
        "title": "Perception",
        "event":
            (
                "Each specialist observed only a partial subset "
                "of the underlying financial environment."
            )
    },

    {
        "chapter": 3,
        "title": "Belief Formation",
        "event":
            (
                "Partial evidence became explicit, testable "
                "and falsifiable financial beliefs."
            )
    },

    {
        "chapter": 4,
        "title": "Social Organization",
        "event":
            (
                f"{len(coalitions_obj)} temporary coalitions "
                "emerged from endogenous communication."
            )
    },

    {
        "chapter": 5,
        "title": "Deliberation",
        "event":
            (
                "Coalitions negotiated theses while preserving "
                "dissent, uncertainty and minority reports."
            )
    },

    {
        "chapter": 6,
        "title": "Competition for Capital",
        "event":
            (
                f"{len(memos_obj)} investment memoranda competed "
                "for scarce institutional capital."
            )
    },

    {
        "chapter": 7,
        "title": "Governance",
        "event":
            (
                f"{len(decisions_obj)} proposals faced "
                "multi-perspective Investment Committee review."
            )
    },

    {
        "chapter": 8,
        "title": "Portfolio Formation",
        "event":
            (
                f"${total_final_allocation:,.0f} was ultimately "
                "deployed after portfolio-level reconciliation."
            )
    },

    {
        "chapter": 9,
        "title": "Reality",
        "event":
            (
                f"After {actual_eval_days_safe} out-of-sample "
                "days, institutional wealth became "
                f"${ending_wealth:,.0f}."
            )
    },

    {
        "chapter": 10,
        "title": "Learning",
        "event":
            (
                "Realized outcomes changed agent reputation, "
                "epistemic scores and institutional influence."
            )
    },

    {
        "chapter": 11,
        "title": "Evolution",
        "event":
            (
                f"The organization moved from "
                f"{original_active_safe} to "
                f"{final_active_safe} coalitions, with "
                f"{n_dissolved_safe} dissolutions, "
                f"{n_merged_safe} mergers and "
                f"{n_splits_safe} splits."
            )
    },

    {
        "chapter": 12,
        "title": "Institutional Memory",
        "event":
            (
                f"{len(history_obj)} structured memory records "
                "carry experience into future decision cycles."
            )
    }

]


biography_df = pd.DataFrame(
    institutional_biography
)


print(
    "\n"
    + "=" * 100
)

print(
    "THE INSTITUTION'S BIOGRAPHY"
)

print(
    "=" * 100
)


display(
    biography_df
)


# ============================================================
# 22. FINAL COALITION RECORDS
# ============================================================

final_coalition_records = []


for c in next_generation_obj:

    final_coalition_records.append({

        "id":
            c.get(
                "id",
                ""
            ),

        "name":
            c.get(
                "name",
                c.get(
                    "id",
                    ""
                )
            ),

        "members":
            c.get(
                "members",
                []
            ),

        "member_specialties":
            c.get(
                "member_specialties",
                []
            ),

        "status":
            c.get(
                "status",
                ""
            ),

        "evolution_action":
            c.get(
                "evolution_action",
                ""
            ),

        "generation":
            safe_int(
                c.get(
                    "generation",
                    0
                )
            ),

        "evolution_fitness":
            safe_float(
                c.get(
                    "evolution_fitness",
                    0.0
                )
            )

    })


# ============================================================
# 23. BUILD FINAL BOARD REPORT
# ============================================================

top_agents_records = (

    top_epistemic_agents

    .head(10)

    .to_dict(
        orient="records"
    )

    if len(
        top_epistemic_agents
    )

    else []

)


board_report = {

    "status":
        (
            "SYNTHETIC EDUCATIONAL AUTONOMOUS "
            "FINANCIAL INSTITUTION EXPERIMENT"
        ),

    "title":
        (
            "50-Agent Autonomous Financial Institution — "
            "Capstone Institutional Autopsy"
        ),

    "generated_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "model":
        MODEL_SAFE,

    "live_llm":
        LIVE_LLM_SAFE,

    "llm_calls":
        LLM_CALLS_SAFE,

    "seed":
        SEED_SAFE,

    "institution":
        {

            "agents":
                len(
                    agents_obj
                ),

            "initial_capital":
                INITIAL_CAPITAL_SAFE,

            "mandate":
                CHARTER_OBJ

        },

    "economic_autopsy":
        economic_autopsy,

    "capital_autopsy":
        capital_autopsy,

    "epistemic_autopsy":
        epistemic_autopsy,

    "governance_autopsy":
        governance_autopsy,

    "organizational_autopsy":
        organizational_autopsy,

    "network_autopsy":
        network_autopsy,

    "institutional_scorecard":
        institutional_scorecard.to_dict(
            orient="records"
        ),

    "top_epistemic_agents":
        top_agents_records,

    "next_generation_coalitions":
        final_coalition_records,

    "biography":
        institutional_biography,

    "institutional_memory_records":
        len(
            history_obj
        )

}


# ============================================================
# 24. BUILD ROBUST PROVENANCE EXPORT
# ============================================================

provenance_export = {

    "experiment":
        (
            "50-Agent Autonomous Financial Institution"
        ),

    "seed":
        SEED_SAFE,

    "model":
        MODEL_SAFE,

    "live_llm":
        LIVE_LLM_SAFE,

    "llm_call_limit":
        MAX_LLM_CALLS_SAFE,

    "llm_calls_used":
        LLM_CALLS_SAFE,

    "decision_day":
        EVENT_DAY_SAFE,

    "evaluation_end":
        EVAL_END_SAFE,

    "evaluation_days":
        actual_eval_days_safe,

    "world_regimes":
        list(
            regimes_obj
        ),

    "assets":
        list(
            assets_obj
        ),

    "institutional_chain":
        institutional_chain,

    "provenance_log":
        provenance_records,

    "memory":
        history_obj

}


# ============================================================
# 25. EXPORT JSON REPORTS
# ============================================================

board_report_path = (

    OUTPUT_DIR

    /
    "board_report.json"

)


provenance_path = (

    OUTPUT_DIR

    /
    "provenance_export.json"

)


with open(
    board_report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        make_serializable(
            board_report
        ),

        f,

        indent=2,

        ensure_ascii=False

    )


with open(
    provenance_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        make_serializable(
            provenance_export
        ),

        f,

        indent=2,

        ensure_ascii=False

    )


# ============================================================
# 26. EXPORT ANALYTICAL TABLES
# ============================================================

table_exports = {

    "institutional_chain.csv":
        chain_df,

    "coalition_performance.csv":
        episode_df_safe,

    "coalition_learning.csv":
        coalition_learning_df_safe,

    "agent_learning.csv":
        agent_learning_df_safe,

    "agent_final_state.csv":
        agent_state_df_safe,

    "capital_allocation.csv":
        allocation_df_safe,

    "organizational_evolution.csv":
        final_evolution_df_safe,

    "agent_migration.csv":
        migration_df_safe,

    "institutional_biography.csv":
        biography_df,

    "institutional_scorecard.csv":
        institutional_scorecard,

    "architecture_comparison.csv":
        architecture_comparison

}


for filename, dataframe in table_exports.items():

    if isinstance(
        dataframe,
        pd.DataFrame
    ):

        dataframe.to_csv(

            OUTPUT_DIR
            /
            filename,

            index=False

        )


# ============================================================
# 27. HUMAN-READABLE EXECUTIVE SUMMARY
# ============================================================

executive_summary = f"""
50-AGENT AUTONOMOUS FINANCIAL INSTITUTION
CAPSTONE EXECUTIVE SUMMARY
======================================================================

INITIAL CONDITIONS
------------------
Agents: {len(agents_obj)}
Initial Capital: ${INITIAL_CAPITAL_SAFE:,.0f}
Decision Day: {EVENT_DAY_SAFE}
Evaluation Horizon: {actual_eval_days_safe} days
Starting Hidden Regime: {starting_regime}
Ending Hidden Regime: {ending_regime}

ECONOMIC OUTCOME
----------------
Ending Wealth: ${ending_wealth:,.0f}
Institutional P&L: ${institutional_pnl:,.0f}
Institutional Return: {institutional_return:.2%}
Maximum Drawdown: {institutional_max_dd:.2%}
Realized Volatility: {institutional_vol:.2%}
Sharpe-like Ratio: {institutional_sharpe:.2f}
Coalition Hit Rate: {coalition_hit_rate:.1%}

CAPITAL ALLOCATION
------------------
Committee Authorized: ${total_committee_approved:,.0f}
Final Capital Deployed: ${total_final_allocation:,.0f}
Residual Capital: ${cash_after_allocation:,.0f}
Funded Positions: {funded_positions}

EPISTEMIC OUTCOME
-----------------
Evidence / Return Correlation: {evidence_corr:.3f}
Persuasion / Return Correlation: {persuasion_corr:.3f}
Committee / Return Correlation: {committee_corr:.3f}
Debate / Return Correlation: {debate_corr:.3f}

Mean Reputation:
{mean_reputation_before:.3f}
→ {mean_reputation_after:.3f}

Mean Epistemic Score:
{mean_epistemic_score:.3f}

Mean Persuasion Score:
{mean_persuasion_score:.3f}

ORGANIZATIONAL EVOLUTION
------------------------
Coalitions Before: {original_active_safe}
Coalitions After: {final_active_safe}
Dissolutions: {n_dissolved_safe}
Mergers: {n_merged_safe}
Splits: {n_splits_safe}
Agents Recruited: {n_recruited_safe}
Agents Released: {n_released_safe}
Active Agents: {active_agents_safe}
Multi-Coalition Agents: {multi_coalition_agents_safe}

PROVENANCE
----------
Model: {MODEL_SAFE}
Live LLM: {LIVE_LLM_SAFE}
LLM Calls: {LLM_CALLS_SAFE}/{MAX_LLM_CALLS_SAFE}
Provenance Records: {len(provenance_records)}
Institutional Memory Records: {len(history_obj)}

CORE RESULT
-----------
The experiment did not merely produce a portfolio.

It produced an institution that perceived, reasoned, communicated,
organized, argued, allocated capital, experienced consequences,
learned from those consequences, and modified its own organizational
structure.

The final object of analysis is therefore not an individual model,
prompt, agent, or coalition.

It is an adaptive artificial financial institution.
"""


executive_summary_path = (

    OUTPUT_DIR

    /
    "executive_summary.txt"

)


with open(
    executive_summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        executive_summary
    )


print(
    "\n"
    + executive_summary
)


# ============================================================
# 28. PROVENANCE AND REPRODUCIBILITY AUDIT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "V. PROVENANCE AND REPRODUCIBILITY AUDIT"
)

print(
    "=" * 100
)


print(
    f"Random seed                  : "
    f"{SEED_SAFE}"
)

print(
    f"LLM model                    : "
    f"{MODEL_SAFE}"
)

print(
    f"Live LLM enabled             : "
    f"{LIVE_LLM_SAFE}"
)

print(
    f"LLM calls used               : "
    f"{LLM_CALLS_SAFE}/{MAX_LLM_CALLS_SAFE}"
)

print(
    f"Provenance records           : "
    f"{len(provenance_records)}"
)

print(
    f"Institutional memory records : "
    f"{len(history_obj)}"
)

print(
    f"Output directory             : "
    f"{OUTPUT_DIR}"
)


# ============================================================
# 29. EXPORT MANIFEST
# ============================================================

export_manifest = []


for path in sorted(
    OUTPUT_DIR.iterdir()
):

    if path.is_file():

        export_manifest.append({

            "file":
                path.name,

            "bytes":
                path.stat().st_size

        })


export_manifest_df = pd.DataFrame(
    export_manifest
)


print(
    "\nEXPORTED CAPSTONE ARTIFACTS"
)


display(
    export_manifest_df
)


# ============================================================
# 30. FINAL CAPSTONE SYNTHESIS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CAPSTONE SYNTHESIS"
)

print(
    "=" * 100
)


print(
    """
The experiment began with fifty specialists and no predetermined
institutional answer.

They inhabited the same financial world, but they did not see the
same world.

Each specialist observed reality through a different disciplinary
lens. Partial observations became beliefs. Beliefs became arguments.
Arguments created relationships. Relationships created coalitions.

Those coalitions were not permanent departments.

They were temporary epistemic institutions.

They did not merely vote. They debated.

They identified agreement, disagreement, causal mechanisms,
falsification conditions, unresolved uncertainty and minority views.

The coalitions then entered an internal market for scarce capital.

Investment memoranda transformed beliefs into economic commitments.
The Investment Committee subjected those commitments to governance
from multiple perspectives. Evidence was deliberately separated from
eloquence. Confidence was distinguished from accuracy. Persuasion was
not allowed to become a synonym for truth.

Portfolio-level constraints then forced individually attractive
proposals to coexist within one institutional balance sheet.

And then the institution encountered something that no agent,
coalition, committee or language model could negotiate with:

                              REALITY.

Out-of-sample market outcomes generated profits, losses, volatility
and drawdowns.

Reality therefore became the institution's ultimate adjudicator.

But the experiment did not stop with P&L.

The consequences of decisions altered the institution itself.

Agents accumulated or lost epistemic reputation. Persuasive ability
remained distinct from demonstrated accuracy. Quiet experts could
gain institutional influence. Persuasive but unreliable specialists
could lose it. Committee confidence itself became an object of
calibration.

The institution developed memory.

It remembered not only what happened, but what had been believed,
what evidence had supported the belief, who had dissented, what the
committee had authorized, what regime had prevailed, and what reality
eventually delivered.

Finally, learning changed organizational structure.

Coalitions could persist.

Coalitions could recruit.

Coalitions could release members.

Coalitions could mutate.

Related coalitions could merge.

Internally unstable coalitions could split.

Weak structures could disappear entirely.

The artificial financial institution therefore became endogenous
at three progressively deeper levels:

                    ENDOGENOUS BELIEFS

                    ENDOGENOUS CAPITAL ALLOCATION

                    ENDOGENOUS ORGANIZATIONAL STRUCTURE

This is the central result of the capstone.

The experiment is no longer principally about whether an artificial
agent can answer a financial question.

It asks whether a society of heterogeneous artificial specialists
can create something resembling an institution: a persistent system
of differentiated expertise, argument, governance, scarce-resource
allocation, consequences, memory, trust and organizational change.

The unit of analysis is therefore no longer the prompt.

It is no longer the model.

It is no longer the individual agent.

It is not even the coalition.

                         THE UNIT OF ANALYSIS
                              IS THE
                           INSTITUTION.
"""
)


# ============================================================
# 31. FINAL RESEARCH QUESTION
# ============================================================

CAPSTONE_QUESTION = """

Can a heterogeneous society of artificial financial specialists
discover opportunities, organize itself into temporary institutions,
argue for scarce capital, learn whom to trust, dissolve obsolete
structures, and redesign its own organization as the financial
world changes?

"""


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL RESEARCH QUESTION"
)

print(
    "=" * 100
)


print(
    CAPSTONE_QUESTION
)


# ============================================================
# 32. FINAL COMPLETION STATUS
# ============================================================

completion_status = {

    "cell":
        15,

    "status":
        "COMPLETE",

    "agents":
        len(
            agents_obj
        ),

    "initial_capital":
        INITIAL_CAPITAL_SAFE,

    "ending_wealth":
        ending_wealth,

    "institutional_return":
        institutional_return,

    "coalitions_initial":
        original_active_safe,

    "coalitions_final":
        final_active_safe,

    "institutional_memory_records":
        len(
            history_obj
        ),

    "provenance_records":
        len(
            provenance_records
        ),

    "output_directory":
        str(
            OUTPUT_DIR
        )

}


print(
    "\nFINAL COMPLETION STATUS"
)


display(
    pd.DataFrame(
        [
            completion_status
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "END OF THE 50-AGENT AUTONOMOUS FINANCIAL "
    "INSTITUTION CAPSTONE"
)

print(
    "=" * 100
)

CODE UNIT 15 — INSTITUTIONAL AUTOPSY AND BIOGRAPHY

Output directory: /content/capstone_outputs

No explicit provenance log was found.
Cell 15 will construct a minimal reproducibility record from the notebook state.

BIOGRAPHY OF THE AUTONOMOUS FINANCIAL INSTITUTION


,stage,name,description
0,1,WORLD,Synthetic financial environment with hidden ma...
1,2,PERCEPTION,Agents observe different subsets of the same f...
2,3,BELIEFS,Specialists transform partial evidence into ex...
3,4,COMMUNICATION,Endogenous affinity determines which specialis...
4,5,COALITIONS,Temporary and potentially overlapping institut...
5,6,DEBATE,"Coalitions negotiate agreement, dissent, falsi..."
6,7,MEMORANDA,Coalitions translate hypotheses into formal re...
7,8,GOVERNANCE,"The Investment Committee evaluates evidence, r..."
8,9,CAPITAL ALLOCATION,Authorized proposals compete for a scarce inst...
9,10,REALITY,"Out-of-sample market outcomes generate P&L, vo..."



I. ECONOMIC AUTOPSY
Initial capital          : $10,000,000,000
Ending wealth            : $9,898,489,377
Institutional P&L        : $-101,510,623
Institutional return     : -1.02%
Maximum drawdown         : -3.32%
Realized volatility      : 5.59%
Sharpe-like ratio        : -0.76
Coalition hit rate       : 0.0%
Hidden regime            : Calm Growth → Financial Stress

CAPITAL ALLOCATION
Committee authorized     : $7,500,000,000
Final deployed capital   : $3,600,000,000
Portfolio-level reduction: $3,900,000,000 (52.0%)
Funded coalition positions: 12
Residual capital / cash  : $6,400,000,000

COALITION ECONOMIC CONTRIBUTION


,rank,coalition_id,coalition_name,asset,direction,capital,realized_return,pnl,max_drawdown,reality_score
0,1,C01,Emergent Coalition 1,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
1,2,C02,Emergent Coalition 2,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
2,3,C03,Emergent Coalition 3,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
3,4,C04,Emergent Coalition 4,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
4,5,C05,Emergent Coalition 5,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
5,6,C06,Emergent Coalition 6,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
6,7,C07,Cross-Asset Signal Divergence Working Group,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
7,8,C08,Emergent Coalition 8,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
8,9,C09,Emergent Coalition 9,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025
9,10,C10,Emergent Coalition 10,EQUITY,long,300000000.0,-0.030557,-9.167026e+06,-0.090056,0.436025



II. EPISTEMIC AUTOPSY
Evidence → return correlation   : 0.000
Persuasion → return correlation : 0.000
Committee → return correlation  : 0.000
Debate → return correlation     : 0.000
Mean reputation                 : 0.522 → 0.526
Mean epistemic score            : 0.506
Mean persuasion score           : 0.508
Mean influence                  : 0.505
Committee calibration error     : 0.132
Coalition calibration error     : 0.122

TOP EPISTEMIC AGENTS


...



50-AGENT AUTONOMOUS FINANCIAL INSTITUTION
CAPSTONE EXECUTIVE SUMMARY

INITIAL CONDITIONS
------------------
Agents: 50
Initial Capital: $10,000,000,000
Decision Day: 100
Evaluation Horizon: 60 days
Starting Hidden Regime: Calm Growth
Ending Hidden Regime: Financial Stress

ECONOMIC OUTCOME
----------------
Ending Wealth: $9,898,489,377
Institutional P&L: $-101,510,623
Institutional Return: -1.02%
Maximum Drawdown: -3.32%
Realized Volatility: 5.59%
Sharpe-like Ratio: -0.76
Coalition Hit Rate: 0.0%

CAPITAL ALLOCATION
------------------
Committee Authorized: $7,500,000,000
Final Capital Deployed: $3,600,000,000
Residual Capital: $6,400,000,000
Funded Positions: 12

EPISTEMIC OUTCOME
-----------------
Evidence / Return Correlation: 0.000
Persuasion / Return Correlation: 0.000
Committee / Return Correlation: 0.000
Debate / Return Correlation: 0.000

Mean Reputation:
0.522
→ 0.526

Mean Epistemic Score:
0.506

Mean Persuasion Score:
0.508

ORGANIZATIONAL EVOLUTION
------------------------

,file,bytes
0,agent_final_state.csv,4017
1,agent_learning.csv,9160
2,agent_migration.csv,1
3,architecture_comparison.csv,163
4,board_report.json,14869
5,capital_allocation.csv,1734
6,coalition_learning.csv,4447
7,coalition_performance.csv,4618
8,executive_summary.txt,1810
9,final_institutional_network.png,408288



CAPSTONE SYNTHESIS

The experiment began with fifty specialists and no predetermined
institutional answer.

They inhabited the same financial world, but they did not see the
same world.

Each specialist observed reality through a different disciplinary
lens. Partial observations became beliefs. Beliefs became arguments.
Arguments created relationships. Relationships created coalitions.

Those coalitions were not permanent departments.

They were temporary epistemic institutions.

They did not merely vote. They debated.

They identified agreement, disagreement, causal mechanisms,
falsification conditions, unresolved uncertainty and minority views.

The coalitions then entered an internal market for scarce capital.

Investment memoranda transformed beliefs into economic commitments.
The Investment Committee subjected those commitments to governance
from multiple perspectives. Evidence was deliberately separated from
eloquence. Confidence was distinguished from accuracy. Persuasion was
n

,cell,status,agents,initial_capital,ending_wealth,institutional_return,coalitions_initial,coalitions_final,institutional_memory_records,provenance_records,output_directory
0,15,COMPLETE,50,1.000000e+10,9.898489e+09,-0.010151,12,12,13,1,/content/capstone_outputs



END OF THE 50-AGENT AUTONOMOUS FINANCIAL INSTITUTION CAPSTONE


##THE NARRATIVE




###The Autonomous Financial Institution: From Fifty Specialists to an Adaptive Society of Capital

The purpose of this notebook is not simply to demonstrate that fifty artificial-intelligence agents can analyze financial markets. Nor is it another experiment in which a collection of models votes on whether stocks will rise, interest rates will fall, or credit spreads will widen. The ambition is considerably larger. The notebook asks what happens when we stop treating artificial intelligence as a collection of tools for solving isolated financial problems and instead ask whether those tools can begin to constitute something resembling a **financial institution**.

That distinction is fundamental. A conventional financial model receives data and produces an answer. An agent can go further: it can interpret information, use tools, formulate a judgment and act. A swarm of agents adds another layer because multiple forms of expertise can coexist. But an institution requires something more difficult. It needs specialization, communication, disagreement, governance, competition for scarce resources, accountability for decisions, memory of past successes and failures, and mechanisms through which its own organization can change.

The notebook therefore begins with a hypothetical financial institution endowed with approximately $10 billion of capital and fifty artificial specialists. These agents are deliberately heterogeneous. Some resemble professionals we would recognize immediately inside a large investment organization: macroeconomists, rates specialists, credit analysts, equity investors, derivatives experts, portfolio managers, treasury specialists and risk officers. Others come from venture capital, private equity, infrastructure, digital assets, payments, semiconductors and artificial-intelligence economics. Still others introduce intellectual traditions that financial institutions do not normally organize as investment departments: game theory, information theory, complex systems and control engineering.

The diversity is intentional. If all fifty agents possessed the same information, used the same analytical framework and optimized the same objective, adding agents would provide little more than computational redundancy. The interesting possibility emerges when intelligent specialists **see the same world differently**.

## A World That No Agent Can See Completely

The experiment therefore creates a synthetic financial world governed by hidden regimes. The economy can move through conditions such as calm growth, inflationary pressure, financial stress, recovery and technological expansion. Each regime affects asset returns, volatility and observable macro-financial variables differently. Equities, duration, credit, volatility hedges, digital assets and cash consequently behave differently as the underlying state changes.

The critical feature is that the true regime is not handed to the agents.

This creates the first important institutional problem: **partial knowledge**.

A rates specialist pays particular attention to inflation, sovereign yields and macroeconomic conditions. A credit specialist observes spreads, liquidity and deterioration in financing conditions. An AI-economics specialist may focus more heavily on technology demand, financing conditions and investment in computational infrastructure. A market-risk agent notices volatility and liquidity. A digital-assets specialist observes another subset of the environment.

They inhabit the same economy but possess different windows into it.

This resembles a real financial institution much more closely than a centralized model with a perfectly organized dataset. Inside an actual bank, asset manager or investment fund, information is distributed. The credit desk knows things the equity desk does not. Treasury sees liquidity pressures that a fundamental analyst may barely notice. A sector specialist may detect an industrial transformation before it appears clearly in macroeconomic statistics.

The notebook therefore converts informational incompleteness from a nuisance into a structural characteristic of the artificial institution.

## From Perception to Belief

Partial observations must next become judgments. Each agent receives its own evidence packet and constructs a financial belief: a thesis, a degree of confidence, an investment direction, a horizon, supporting evidence, desirable collaborators and, importantly, conditions under which the thesis should be considered false.

This last element matters enormously.

The agents are not merely being asked, “What do you think?” They are being encouraged to construct **falsifiable beliefs**. A thesis without a condition under which it could be rejected is closer to rhetoric than disciplined investment analysis.

The language model is particularly useful here because financial judgment cannot always be compressed into a single numerical forecast. An experienced investor might say that liquidity is deteriorating, credit markets have not yet fully incorporated that deterioration, equity volatility remains deceptively subdued, and the combination creates an asymmetric opportunity to acquire convex protection. That reasoning contains causal structure, uncertainty and temporal sequencing. Natural language allows the agents to express those dimensions.

The result is a society of specialists possessing heterogeneous interpretations of a common but incompletely observed world.

## Intelligence Becomes Social

At this point the notebook makes an important transition. The agents do not report independently to a central optimizer. They begin to communicate.

An endogenous network is constructed according to intellectual affinity, complementary expertise, confidence and investment horizon. Agents discover other agents with whom collaboration might be useful. Crucially, collaboration does not require complete agreement.

That is an important departure from simplistic swarm architectures. If coalitions were composed only of agents that already agreed, they would risk becoming artificial echo chambers. Productive financial institutions often benefit from the opposite phenomenon: people who share enough of a problem to work together but approach it from different intellectual traditions.

A macro specialist concerned about financial conditions may therefore find a natural collaborator in a bank-credit analyst, a liquidity specialist and a derivatives expert. Their models are different, but their partial observations may point toward a common economic mechanism. Similarly, an AI-economics agent, semiconductor specialist, infrastructure financier and power-market analyst might recognize that apparently separate signals are components of a larger investment thesis concerning AI infrastructure.

This produces **temporary coalitions**.

The word temporary is essential. These are not permanent departments imposed by the notebook designer. They are endogenous constellations created because a particular configuration of knowledge appears useful at a particular moment. Agents may participate in more than one coalition. Organizational boundaries are therefore permeable.

The institution has begun to organize itself.

## Coalitions Must Argue, Not Merely Aggregate

Once formed, the coalitions do not simply average their members' forecasts. They debate.

The notebook asks each coalition to identify what its members genuinely agree about, where they disagree, what causal mechanism could connect their observations, what the strongest objection to their thesis might be, what evidence is missing and what developments would falsify their position.

An internal skeptic is explicitly preserved.

This is one of the most important governance principles in the notebook. Consensus is not treated as automatically desirable. A minority report can contain valuable information. Indeed, one of the dangers of collective intelligence is that the social process that produces agreement can destroy precisely the informational diversity that made the collective useful in the first place.

The coalition therefore emerges from debate with a **negotiated thesis**, not necessarily unanimous conviction.

It must also translate that thesis into a financial expression. Should the institution own equities? Reduce credit exposure? Buy duration? Acquire volatility protection? Take exposure to digital assets? Remain in cash? Should the expression be long or short?

The progression is now becoming distinctly institutional:

$
\text{Observation}
\rightarrow
\text{Belief}
\rightarrow
\text{Communication}
\rightarrow
\text{Coalition}
\rightarrow
\text{Debate}
\rightarrow
\text{Investment Thesis}.
$

But a thesis still costs nothing. The next step introduces scarcity.

## The Internal Market for Capital

Every coalition must prepare an investment memorandum and request capital.

This changes the nature of the experiment. Coalitions are no longer merely intellectual communities; they become competitors for the institution's finite balance sheet.

The amount requested is not arbitrary. It reflects confidence, debate quality, volatility, downside risk, uncertainty, investment horizon and the quality of the coalition's argument. A high-confidence thesis concerning a relatively stable opportunity might justify substantial capital. A highly uncertain proposition in a volatile asset should generally receive less.

Here the notebook introduces a distinction that is central not merely to AI but to human organizations: **evidence and persuasion are not the same thing**.

A coalition can produce an eloquent memorandum without possessing strong evidence. Another may possess an excellent insight but communicate it poorly. If an artificial institution rewards linguistic fluency as though it were epistemic accuracy, it creates a dangerous internal selection mechanism. The most rhetorically capable agents could accumulate capital and influence even if they are systematically wrong.

The notebook therefore tracks evidence strength and persuasive quality separately. The difference between them becomes an “eloquence gap,” which governance can examine rather than unconsciously reward.

## Governance Enters the Architecture

The investment memoranda proceed to an artificial Investment Committee composed of several institutional perspectives: Chief Investment Officer, Chief Risk Officer, Treasury, Quantitative Review, Skeptic and Institutional Memory.

This is not simply six agents voting.

Each represents a different institutional responsibility. The CIO asks whether the opportunity deserves capital. The CRO asks what can go wrong. Treasury considers liquidity and balance-sheet consequences. Quantitative Review examines analytical coherence. The Skeptic challenges narrative seduction and overconfidence. Institutional Memory asks whether the organization has encountered similar arguments before and what happened.

The committee can approve, resize, condition or reject proposals.

But even committee approval does not mean immediate deployment. This distinction mirrors real institutional finance. A proposal can be attractive individually while being inappropriate when considered alongside every other position.

The next layer therefore reconciles all authorized proposals against the institution's aggregate constraints: minimum liquidity, maximum exposure to a single coalition, asset concentration, directional concentration and total gross exposure.

Capital allocation becomes a **balance-sheet problem**.

If approved requests exceed available deployable capital, the notebook does not simply shrink every position proportionally. Scarce capital is allocated according to institutional priority, taking into account committee conviction, evidence, debate quality, coalition confidence and excessive reliance on persuasion.

The institution has now converted distributed intelligence into actual financial commitments.

## Reality Gets the Final Vote

Up to this point everything has occurred inside the cognitive and organizational machinery of the institution. Agents have observed, reasoned, persuaded and governed.

Then the notebook introduces the one participant that cannot be persuaded: **reality**.

The portfolio is carried forward through an out-of-sample period. Actual synthetic asset returns determine coalition-level profits and losses, realized volatility and drawdowns. Because coalition identities are preserved, the notebook can attribute economic consequences back to the specific institutions that proposed each position.

Only after the investment episode is completed is the hidden regime revealed.

This prevents hindsight from contaminating the decision process. The agents had to operate under uncertainty; the researcher can subsequently determine what environment they were actually navigating.

The notebook can now ask much richer questions than “Did the portfolio make money?” Did strong evidence predict better outcomes? Did persuasive arguments outperform weaker ones? Was committee conviction calibrated to reality? Did high-quality debate matter? Which coalitions generated value? Which consumed risk without compensation?

Economic performance becomes an instrument for **epistemic evaluation**.

## Learning Whom to Trust

The next transformation is particularly important. Reality changes the social structure of the institution.

Agents have several attributes that are deliberately kept separate: reputation, epistemic score, persuasive ability and influence.

An agent that participated in successful, well-reasoned and well-calibrated coalitions can improve its epistemic standing. An agent that argued persuasively for propositions that failed does not receive the same reward merely because its argument sounded compelling. Conversely, a quiet specialist whose evidence repeatedly proves useful can gradually acquire greater institutional influence.

This creates the beginnings of an endogenous market for trust.

The institution also develops memory. It records what a coalition believed, what evidence supported the thesis, who participated, what dissent existed, what the committee decided, how much capital was deployed, which regime eventually prevailed and what economic result followed.

Memory therefore becomes more than a transcript. It is **institutional experience**.

## The Institution Changes Itself

A conventional adaptive model changes parameters. This notebook goes further: the organization itself can change.

Coalitions receive evolutionary fitness measures that incorporate outcomes, epistemic quality, calibration, member quality, diversity and the circumstances of the regime. Successful coalitions may persist. Others may recruit new expertise. Weak members may be released. Coalitions may mutate after regime changes. Related groups may merge. Large or internally heterogeneous groups may split. Structures that no longer provide sufficient value may disappear.

This introduces organizational mortality.

It also prevents one successful configuration from becoming permanently entrenched. A coalition that worked during financial stress may be poorly suited to technological expansion. The institution must therefore distinguish between expertise that is generally reliable and organizational structures whose usefulness depends on context.

The result is a recursive loop:

$
\text{World}
\rightarrow
\text{Beliefs}
\rightarrow
\text{Coalitions}
\rightarrow
\text{Capital}
\rightarrow
\text{Reality}
\rightarrow
\text{Learning}
\rightarrow
\text{Organizational Evolution}.
$

The output of one cycle is no longer merely a portfolio return. It is a **different institution**.

## The Final Institutional Autopsy

The last cell reconstructs the entire biography.

It examines economic performance: wealth, return, drawdown, volatility, coalition contributions and capital deployment. It examines epistemic performance: whether evidence, persuasion, debate and committee conviction were related to realized outcomes. It examines governance: what was approved, resized or rejected and whether the committee was well calibrated. It examines organization: which coalitions survived, merged, split or disappeared, which agents migrated and how the network changed.

Importantly, these dimensions are not collapsed into one master score.

That restraint is deliberate. A financial institution can earn money for bad reasons. It can possess excellent analytical processes and suffer losses because an unlikely event occurred. It can have strong governance but weak opportunity discovery. It can be profitable while becoming dangerously concentrated. Economic, epistemic, governance and organizational quality are related, but they are not identical.

The notebook therefore ends not with a declaration that the artificial institution “won,” but with a richer object: its **biography**.

And this reveals the larger intellectual progression behind the experiment.

At the beginning, artificial intelligence consisted of individual specialists solving problems. Then specialists became a swarm. The swarm developed communication. Communication produced coalitions. Coalitions developed internal argument. Argument competed for scarce capital. Capital became subject to governance. Decisions encountered reality. Reality altered reputation. Reputation changed influence. Memory accumulated. Learning eventually changed organizational structure itself.

There are therefore three increasingly profound forms of endogeneity in the system:

$
\boxed{\text{Endogenous Beliefs}}
$

$
\boxed{\text{Endogenous Capital Allocation}}
$

$
\boxed{\text{Endogenous Organizational Structure}}
$

That final transition is what makes the notebook substantially different from a conventional multi-agent financial demonstration. We are no longer asking merely whether AI can help a financial professional make a decision. We are asking whether heterogeneous artificial specialists can create a persistent social architecture for making decisions under uncertainty—an architecture containing expertise, disagreement, competition, governance, consequences, memory and adaptation.

The deepest question is consequently not whether one of the fifty agents is intelligent enough.

It is whether **intelligence can become institutional**.

Can a heterogeneous society of artificial financial specialists discover opportunities that no single specialist could recognize alone? Can they organize around those opportunities without being told in advance what the correct organizational structure should be? Can they disagree productively? Can they distinguish evidence from eloquence? Can they compete for scarce resources without destroying collective coherence? Can they learn which members deserve trust? Can they preserve useful dissent? Can they dismantle organizations that have outlived their purpose and create new ones when the world changes?

Those questions move us beyond the familiar progression from models to agents and from agents to swarms. They point toward a more consequential possibility: **autonomous artificial institutions whose intelligence resides not only in their individual components, but also in the evolving relationships among them**.

That is why the final lesson of the notebook can be expressed in one sentence.

**The unit of analysis is no longer the model. It is no longer the agent. It is not even the swarm. The unit of analysis is the institution.**

## Conclusions — From Swarm Intelligence to Endogenous Financial Institutions

This capstone changes the unit of analysis. Earlier swarm notebooks treated the population of agents as an instrument for solving a particular problem. Here the population becomes the institution. The central research object is no longer only a forecast, trade, valuation, or optimization. It is the evolving architecture through which beliefs are formed, challenged, funded, remembered, and eventually abandoned.

The notebook begins with a deliberately sparse constitution: fifty persistent specialists, USD 10 billion of synthetic capital, liquidity and concentration constraints, and a changing financial world. Everything else is allowed to become endogenous. Agents observe different slices of the same reality. They articulate beliefs in natural language. A dynamic communication network identifies potentially useful relationships, but numerical affinity does not automatically create a team. Candidate members must discover a coherent reason to organize. Their coalition then becomes a temporary institution with a thesis, dissenting view, lifespan, falsification condition, and dissolution rule.

The use of Claude Sonnet 5 is central rather than decorative. Python handles the fast clock—market simulation, signals, network construction, accounting identities, portfolio returns, drawdowns, and hard constraints. The LLM handles institutional judgment—interpretation, negotiation, memorandum writing, cross-examination, committee reasoning, and selected organizational changes. This division of labor makes the experiment computationally tractable and conceptually faithful to the idea that autonomous systems can combine numerical machinery with language-based reasoning.

The Investment Committee introduces scarcity and governance. Coalitions do not merely announce ideas; they compete for capital. CIO, CRO, Treasury, quantitative review, skepticism, and institutional memory force proposals to confront opportunity cost, tail risk, liquidity, evidentiary quality, alternative explanations, and historical precedent. A proposal can be approved, rejected, resized, or made conditional. Hard constraints remain deterministic: language can influence allocation, but it cannot create capital that does not exist or silently eliminate the liquidity reserve.

One of the most important design choices is the separation of evidence, persuasion, and realized truth. Financial organizations are social systems. Strong narratives can attract resources. The notebook therefore permits coalitions to be persuasive, but records persuasive quality separately from evidentiary strength. When future returns arrive, reality becomes the adjudicator. A coalition that repeatedly wins arguments but loses money should eventually suffer an epistemic penalty. Conversely, an agent whose dissent repeatedly proves correct should gain influence.

The simulation also creates organizational mortality. Coalitions can dissolve. Their members return to the population. A coalition can mutate when the environment changes: Financial Stress can become Distressed Opportunities; inflation defense can become duration opportunity; technology enthusiasm can fragment into infrastructure, power, and valuation debates. Institutional memory survives these deaths. The organization can forget a structure without forgetting what happened.

Several extensions follow naturally. The notebook can be run across many random seeds to test whether similar organizational forms repeatedly emerge. Committee composition can be changed to study governance. Persuasion can be experimentally amplified or suppressed to measure narrative capture. Network constraints can test whether excessive communication creates herding. Richer agent memory can record individual collaboration histories. Real historical data can eventually replace the synthetic world while preserving the architecture.

The most ambitious extension is not simply more agents. It is multiple evolving institutions. A bank-like swarm, hedge-fund-like swarm, insurer-like swarm, venture swarm, and payment-network swarm could interact without being permanently labeled as such. Organizational identity itself could emerge from liabilities, opportunities, regulation, and accumulated history. At that point the experiment moves from autonomous financial institutions toward artificial financial ecosystems.

The capstone should therefore be evaluated on two levels. The first is financial: returns, drawdowns, liquidity, risk-adjusted contribution, and capital efficiency. The second is institutional: coalition diversity, lifespan, mutation, centralization, dissent quality, calibration, reputation dynamics, and the relationship between persuasive power and realized performance. A financially successful simulation with pathological governance is not necessarily a successful institution. Likewise, an institution that occasionally sacrifices return to preserve solvency may be behaving exactly as its constitution requires.

The final question can now be stated precisely:

> **Can a heterogeneous society of artificial financial specialists discover opportunities, organize itself into temporary institutions, argue for scarce capital, learn whom to trust, dissolve obsolete structures, and redesign its own organization as the financial world changes?**

If the answer is even partially yes in a controlled synthetic laboratory, then the significance of swarm architectures in finance extends well beyond task automation. The frontier becomes the design of governed artificial institutions whose internal behavioral dynamics are themselves adaptive. That is the conceptual step from a swarm that solves financial problems to a financial organization that continuously decides what problems matter, who should solve them, how much capital their answers deserve, and when the organization itself must change.

## Selected References

1. Anthropic. Claude Platform documentation and Messages API documentation.
2. Hamilton, J. D. (1989). “A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle.” *Econometrica*.
3. Markowitz, H. (1952). “Portfolio Selection.” *Journal of Finance*.
4. Simon, H. A. *Administrative Behavior*.
5. March, J. G. (1991). “Exploration and Exploitation in Organizational Learning.” *Organization Science*.
6. Arthur, W. B. *Increasing Returns and Path Dependence in the Economy*.
7. Holland, J. H. *Adaptation in Natural and Artificial Systems*.
8. Newman, M. E. J. *Networks: An Introduction*.
9. Shapley, L. S. (1953). “A Value for n-Person Games.”
10. Schelling, T. C. *Micromotives and Macrobehavior*.

**Implementation note:** the notebook is configured for `claude-sonnet-5` and includes a live compatibility self-test before substantive swarm inference. Because model APIs evolve, re-check Anthropic's current documentation if the notebook is run substantially later.